In [1]:
##12.1 先把 LB_shp 里所有 shp 读进来并合并
import geopandas as gpd
from pathlib import Path
import pandas as pd

LB_DIR = Path("QM data/12/LB_shp")  # ←改成你自己的路径
shps = sorted(LB_DIR.glob("*.shp"))
print("n_shp:", len(shps), "example:", shps[:3])

gdfs = [gpd.read_file(p) for p in shps]
lsoa_all = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)

print(lsoa_all.shape)
print(lsoa_all.columns)

n_shp: 33 example: [PosixPath('QM data/12/LB_shp/Barking and Dagenham.shp'), PosixPath('QM data/12/LB_shp/Barnet.shp'), PosixPath('QM data/12/LB_shp/Bexley.shp')]
(4994, 7)
Index(['lsoa21cd', 'lsoa21nm', 'msoa21cd', 'msoa21nm', 'lad22cd', 'lad22nm',
       'geometry'],
      dtype='object')


In [2]:
##12.2找出“LSOA 代码列”到底叫什么
cols = [c for c in lsoa_all.columns if "lsoa" in c.lower() or c.lower().endswith("cd")]
print(cols)

['lsoa21cd', 'lsoa21nm', 'msoa21cd', 'lad22cd']


In [3]:
##12.3在代码里统一列名为 lsoa_code（并规范格式）

lsoa_all = lsoa_all.rename(columns={"lsoa21cd": "lsoa_code"})
lsoa_all["lsoa_code"] = lsoa_all["lsoa_code"].astype(str).str.strip().str.upper()

# QA：必须唯一、不能为空
assert lsoa_all["lsoa_code"].isna().sum() == 0
assert lsoa_all["lsoa_code"].duplicated().sum() == 0

lsoa_all[["lsoa_code"]].head()

,lsoa_code
0,E01000011
1,E01000046
2,E01000051
3,E01000077
4,E01000083


In [4]:
##12.4 保存成一个“干净的边界文件”
out_path = LB_DIR.parent / "lsoa_boundary_clean.gpkg"
lsoa_all.to_file(out_path, driver="GPKG", layer="lsoa")
print("saved to:", out_path)

saved to: QM data/12/lsoa_boundary_clean.gpkg


In [5]:
## 1.1清洗 DPI
import pandas as pd
import numpy as np

# ====== 0) paths ======
DPI_PATH = Path("QM data/01/digitalpropensityindexlsoas.xlsx")   # 你的文件
SHEET = "Online_share_LSOA"

OUT_DIR = Path("QM data/01/clean")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ====== 1) auto-detect header row (find the row where col0 == 'LSOA code') ======
peek = pd.read_excel(DPI_PATH, sheet_name=SHEET, header=None, nrows=30)
header_row = peek.index[peek.iloc[:, 0].astype(str).str.strip().eq("LSOA code")]

if len(header_row) == 0:
    raise ValueError("Cannot find header row with 'LSOA code'. Please inspect the file.")
header_row = int(header_row[0])  # in your file, this is 5

# ====== 2) read data using detected header row ======
dpi = pd.read_excel(DPI_PATH, sheet_name=SHEET, header=header_row)

# ====== 3) standardise column names ======
dpi = dpi.rename(columns={
    "LSOA code": "lsoa_code",
    "Region": "region",
    "Local Authority name": "lad_name",
    "Digital Propensity Score": "dpi_score",
    "95% confidence interval: upper limit": "dpi_ci_upper",
    "95% confidence interval: lower limit": "dpi_ci_lower",
})

# ====== 4) clean key + types ======
dpi["lsoa_code"] = dpi["lsoa_code"].astype(str).str.strip().str.upper()
dpi["region"] = dpi["region"].astype(str).str.strip()

for c in ["dpi_score", "dpi_ci_upper", "dpi_ci_lower"]:
    if c in dpi.columns:
        dpi[c] = pd.to_numeric(dpi[c], errors="coerce")  # 'Not applicable' -> NaN

# ====== 5) (optional) keep London only ======
KEEP_LONDON_ONLY = True
if KEEP_LONDON_ONLY:
    dpi = dpi.loc[dpi["region"].eq("London")].copy()

# ====== 6) QA checks ======
assert dpi["lsoa_code"].isna().sum() == 0, "LSOA code has missing values."
assert dpi["lsoa_code"].duplicated().sum() == 0, "Duplicate LSOA codes found."

# sanity check for score range (should be within [0,1] typically)
if dpi["dpi_score"].notna().any():
    mn, mx = float(dpi["dpi_score"].min()), float(dpi["dpi_score"].max())
    print("dpi_score range:", mn, mx)

print("clean DPI shape:", dpi.shape)
print(dpi.head())

# ====== 7) save ======
out_csv = OUT_DIR / ("clean_dpi_london_lsoa.csv" if KEEP_LONDON_ONLY else "clean_dpi_all_lsoa.csv")
out_parq = OUT_DIR / (out_csv.stem + ".parquet")

dpi.to_csv(out_csv, index=False)
dpi.to_parquet(out_parq, index=False)

print("Saved:", out_csv)
print("Saved:", out_parq)

dpi_score range: 0.748 1.0
clean DPI shape: (4835, 6)
   lsoa_code  region              lad_name  dpi_score  dpi_ci_upper  \
0  E01000001  London        City of London      0.970           NaN   
1  E01000002  London        City of London      0.978           NaN   
2  E01000003  London        City of London      0.946           NaN   
3  E01000005  London        City of London      0.938           NaN   
4  E01000006  London  Barking and Dagenham      0.975           NaN   

   dpi_ci_lower  
0           NaN  
1           NaN  
2           NaN  
3           NaN  
4           NaN  
Saved: QM data/01/clean/clean_dpi_london_lsoa.csv
Saved: QM data/01/clean/clean_dpi_london_lsoa.parquet


In [6]:
##1.2 join验证
import pandas as pd
import geopandas as gpd
from pathlib import Path

# ===== paths =====
DPI_PATH = Path("QM data/01/digitalpropensityindexlsoas.xlsx")
DPI_SHEET = "Online_share_LSOA"

BOUNDARY_GPKG = Path("QM data/12/lsoa_boundary_clean.gpkg")
BOUNDARY_LAYER = "lsoa"

OUT_DIR = Path("QM data/clean")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== Step 2: clean DPI (auto-detect header row) =====
peek = pd.read_excel(DPI_PATH, sheet_name=DPI_SHEET, header=None, nrows=30)
header_row = peek.index[peek.iloc[:, 0].astype(str).str.strip().eq("LSOA code")]
if len(header_row) == 0:
    raise ValueError("Cannot find header row with 'LSOA code' in DPI file.")
header_row = int(header_row[0])  # your file is typically 5

dpi = pd.read_excel(DPI_PATH, sheet_name=DPI_SHEET, header=header_row)

dpi = dpi.rename(columns={
    "LSOA code": "lsoa_code",
    "Region": "region",
    "Local Authority name": "lad_name",
    "Digital Propensity Score": "dpi_score",
    "95% confidence interval: upper limit": "dpi_ci_upper",
    "95% confidence interval: lower limit": "dpi_ci_lower",
})

dpi["lsoa_code"] = dpi["lsoa_code"].astype(str).str.strip().str.upper()
dpi["region"] = dpi["region"].astype(str).str.strip()

for c in ["dpi_score", "dpi_ci_upper", "dpi_ci_lower"]:
    if c in dpi.columns:
        dpi[c] = pd.to_numeric(dpi[c], errors="coerce")  # "Not applicable" -> NaN

dpi_london = dpi.loc[dpi["region"].eq("London")].copy()

# QA
assert dpi_london["lsoa_code"].isna().sum() == 0
assert dpi_london["lsoa_code"].duplicated().sum() == 0

print("DPI (London) shape:", dpi_london.shape)
print("dpi_score range:", float(dpi_london["dpi_score"].min()), float(dpi_london["dpi_score"].max()))

# SAVE (CSV only)
dpi_csv = OUT_DIR / "clean_dpi_london_lsoa.csv"
dpi_london.to_csv(dpi_csv, index=False)
print("Saved DPI CSV:", dpi_csv)

# ===== Step 2.5: join back to boundary + QA =====
boundary = gpd.read_file(BOUNDARY_GPKG, layer=BOUNDARY_LAYER)
boundary["lsoa_code"] = boundary["lsoa_code"].astype(str).str.strip().str.upper()

print("Boundary shape:", boundary.shape)
assert boundary["lsoa_code"].isna().sum() == 0
assert boundary["lsoa_code"].duplicated().sum() == 0

g = boundary.merge(
    dpi_london[["lsoa_code", "dpi_score", "dpi_ci_lower", "dpi_ci_upper", "lad_name"]],
    on="lsoa_code",
    how="left",
)

missing = int(g["dpi_score"].isna().sum())
print("Joined shape:", g.shape)
print("Missing dpi_score after join:", missing, f"({missing/len(g):.2%})")

# quick mismatch diagnostics
if missing > 0:
    bset = set(boundary["lsoa_code"])
    dset = set(dpi_london["lsoa_code"])
    print("Example only in boundary:", sorted(bset - dset)[:10])
    print("Example only in DPI:", sorted(dset - bset)[:10])

# SAVE joined geo (GPKG)
out_gpkg = OUT_DIR / "master_lsoa_boundary_plus_dpi.gpkg"
g.to_file(out_gpkg, driver="GPKG", layer="lsoa_dpi")
print("Saved joined GPKG:", out_gpkg)


DPI (London) shape: (4835, 6)
dpi_score range: 0.748 1.0
Saved DPI CSV: QM data/clean/clean_dpi_london_lsoa.csv
Boundary shape: (4994, 7)
Joined shape: (4994, 11)
Missing dpi_score after join: 335 (6.71%)
Example only in boundary: ['E01033784', 'E01033785', 'E01033786', 'E01033787', 'E01033788', 'E01033789', 'E01033790', 'E01033791', 'E01033792', 'E01033862']
Example only in DPI: ['E01000010', 'E01000023', 'E01000048', 'E01000092', 'E01000109', 'E01000125', 'E01000148', 'E01000155', 'E01000262', 'E01000378']
Saved joined GPKG: QM data/clean/master_lsoa_boundary_plus_dpi.gpkg


In [7]:
##1.3 缺失 335 个都集中在哪些 borough/LAD？
# g 是你 merge 完的 GeoDataFrame（boundary + dpi）
missing_rows = g[g["dpi_score"].isna()].copy()

# 看缺失主要来自哪些 LAD（你的边界里有 lad22nm）
print(missing_rows["lad22nm"].value_counts().head(20))

lad22nm
Tower Hamlets           50
Newham                  33
Greenwich               25
Southwark               19
Hillingdon              17
Hounslow                15
Barnet                  15
Brent                   14
Croydon                 14
Hackney                 13
Harrow                  12
Wandsworth              12
Barking and Dagenham    11
Lewisham                 9
Westminster              8
Camden                   6
Islington                6
Ealing                   6
Lambeth                  5
Havering                 5
Name: count, dtype: int64


In [8]:
##1.4 用官方 LSOA 2011↔2021 Lookup 把体系统一
import pandas as pd

LOOKUP_CSV = "QM data/LSOA11_LSOA21_LAD22_EW_LU_v2_-7939559626189777291.csv"  # <- 改成你下载的文件路径

lookup = pd.read_csv(LOOKUP_CSV, dtype=str)
# 关键字段通常包含：LSOA11CD, LSOA21CD（见数据说明）:contentReference[oaicite:3]{index=3}

# dpi_london 你已经有了，主键叫 lsoa_code
dpi_map = dpi_london.merge(
    lookup[["LSOA11CD", "LSOA21CD"]],
    left_on="lsoa_code",
    right_on="LSOA11CD",
    how="left"
)

# 把映射后的 2021 code 作为新的 join key
dpi_map["lsoa21_code"] = dpi_map["LSOA21CD"]

print("Unmapped DPI rows:", dpi_map["lsoa21_code"].isna().sum())

# 如果出现多个 2011 -> 同一个 2021 的情况，聚合一下（简单平均；写进 appendix 说明是 best-fit）
dpi_21 = (
    dpi_map.dropna(subset=["lsoa21_code"])
           .groupby("lsoa21_code", as_index=False)
           .agg(
               dpi_score=("dpi_score", "mean"),
               dpi_ci_lower=("dpi_ci_lower", "mean"),
               dpi_ci_upper=("dpi_ci_upper", "mean"),
           )
           .rename(columns={"lsoa21_code": "lsoa_code"})  # 统一回你的主键命名
)

# 然后用 dpi_21 去 join 你的 boundary（boundary 的 lsoa_code 就是 2021 的）
g2 = boundary.merge(dpi_21, on="lsoa_code", how="left")
print("Missing after mapping:", g2["dpi_score"].isna().sum())


Unmapped DPI rows: 0
Missing after mapping: 181


In [9]:
##缺的是“没匹配到行”还是“匹配到了但 score 为 NA”
# 1) 先看看原始 DPI London 有多少 NA
print("NA dpi_score in dpi_london:", dpi_london["dpi_score"].isna().sum())

# 2) 再做一个带 indicator 的 merge，区分没匹配 vs 匹配了但 NA
g2 = boundary.merge(dpi_21, on="lsoa_code", how="left", indicator=True)

print(g2["_merge"].value_counts())  # both / left_only
print("NA dpi_score in joined g2:", g2["dpi_score"].isna().sum())

# 3) 看缺失里，有多少是 left_only（根本没匹配到），多少是 both 但 score NA
miss = g2[g2["dpi_score"].isna()].copy()
print(miss["_merge"].value_counts())

# 4) 如果你想看“根本没匹配到”的 LSOA code 样例
print("Example left_only codes:", miss.loc[miss["_merge"].eq("left_only"), "lsoa_code"].head(20).tolist())


NA dpi_score in dpi_london: 0
_merge
both          4813
left_only      181
right_only       0
Name: count, dtype: int64
NA dpi_score in joined g2: 181
_merge
left_only     181
right_only      0
both            0
Name: count, dtype: int64
Example left_only codes: ['E01034472', 'E01034473', 'E01034477', 'E01034469', 'E01034476', 'E01034478', 'E01033917', 'E01033919', 'E01033921', 'E01033922', 'E01033913', 'E01033915', 'E01033920', 'E01033924', 'E01033911', 'E01033789', 'E01033792', 'E01033936', 'E01033939', 'E01033940']


In [10]:
##：把研究范围定为 “DPI 覆盖的 LSOA21”
# g2 是 boundary + dpi_21 merge 后的结果（含 _merge）
analysis = g2[g2["_merge"].eq("both")].copy()
print("analysis n:", len(analysis))  # 4813
print("dropped:", (g2["_merge"].eq("left_only")).sum())  # 181

analysis n: 4813
dropped: 181


In [11]:
##保存两个文件（一个完整，一个分析样本）：
from pathlib import Path
OUT_DIR = Path("QM data/clean"); OUT_DIR.mkdir(parents=True, exist_ok=True)

g2.to_file(OUT_DIR/"master_lsoa21_plus_dpi_all.gpkg", driver="GPKG", layer="lsoa_dpi_all")
analysis.to_file(OUT_DIR/"master_lsoa21_plus_dpi_analysis.gpkg", driver="GPKG", layer="lsoa_dpi_analysis")

In [12]:
l21_set = set(lookup["LSOA21CD"].astype(str).str.strip().str.upper())
left_only = g2.loc[g2["_merge"].eq("left_only"), "lsoa_code"]

print("left_only not in lookup LSOA21CD:",
      (~left_only.isin(l21_set)).sum(), "out of", len(left_only))

left_only not in lookup LSOA21CD: 181 out of 181


In [13]:
##3
import pandas as pd
import geopandas as gpd
from pathlib import Path

# =========================================================
# Step 3 标准清洗脚本
# IMD2019（LSOA11）→ 用 lookup 映射到 LSOA21 → 人口加权聚合 → join master
# =========================================================

# ========= 0) paths: 只改这里 =========
IMD_PATH = Path("QM data/03/File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv")  # <- 你的 IMD2019 文件路径（csv）
LOOKUP_CSV = Path("QM data/LSOA11_LSOA21_LAD22_EW_LU_v2_-7939559626189777291.csv")  # <- 你现在用的 lookup
MASTER_GPKG = Path("QM data/clean/master_lsoa21_plus_dpi_analysis.gpkg")  # <- DPI+边界的分析样本（4813 行）
MASTER_LAYER = "lsoa_dpi_analysis"

OUT_DIR = Path("QM data/clean")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ========= 1) 读 master（LSOA21 + DPI） =========
master = gpd.read_file(MASTER_GPKG, layer=MASTER_LAYER)
master["lsoa_code"] = master["lsoa_code"].astype(str).str.strip().str.upper()
assert master["lsoa_code"].isna().sum() == 0
assert master["lsoa_code"].duplicated().sum() == 0
print("master shape:", master.shape)

# ========= 2) 读 IMD（LSOA11） =========
imd = pd.read_csv(IMD_PATH, dtype=str)
print("imd raw shape:", imd.shape)

COL_LSOA11 = "LSOA code (2011)"
COL_POP   = "Total population: mid 2015 (excluding prisoners)"

# 统一主键
imd["lsoa11cd"] = imd[COL_LSOA11].astype(str).str.strip().str.upper()

# 需要用到的分数列（你可按需要增减）
score_cols_long = [
    "Index of Multiple Deprivation (IMD) Score",
    "Income Score (rate)",
    "Employment Score (rate)",
    "Education, Skills and Training Score",
    "Health Deprivation and Disability Score",
    "Crime Score",
    "Barriers to Housing and Services Score",
    "Living Environment Score",
]

# 转成数值
to_numeric_cols = score_cols_long + [COL_POP]
for c in to_numeric_cols:
    if c in imd.columns:
        imd[c] = pd.to_numeric(imd[c], errors="coerce")

# QA：IMD 每个 LSOA11 一行
print("duplicate lsoa11cd:", imd["lsoa11cd"].duplicated().sum())

# ========= 3) 读 lookup：LSOA11 -> LSOA21 =========
lookup = pd.read_csv(LOOKUP_CSV, dtype=str)
lookup["LSOA11CD"] = lookup["LSOA11CD"].astype(str).str.strip().str.upper()
lookup["LSOA21CD"] = lookup["LSOA21CD"].astype(str).str.strip().str.upper()

imd_map = imd.merge(
    lookup[["LSOA11CD", "LSOA21CD"]],
    left_on="lsoa11cd",
    right_on="LSOA11CD",
    how="left"
)

print("unmapped IMD rows (no LSOA21CD):", imd_map["LSOA21CD"].isna().sum())

# ========= 4) 聚合到 LSOA21（人口加权） =========
# 权重：mid-2015 population
imd_map["w"] = imd_map[COL_POP].fillna(0)

# 只保留能映射到 LSOA21 的
imd_map2 = imd_map.dropna(subset=["LSOA21CD"]).copy()

# --- 4.1 分母：每个 LSOA21 的权重总和 ---
den = imd_map2.groupby("LSOA21CD")["w"].sum().rename("w_sum")

# --- 4.2 分子：每列先乘 w，再按 LSOA21 求和 ---
weighted_cols = {f"{c}__w": imd_map2[c] * imd_map2["w"] for c in score_cols_long}
num = (
    imd_map2.assign(**weighted_cols)
            .groupby("LSOA21CD")[[f"{c}__w" for c in score_cols_long]]
            .sum()
)

# --- 4.3 加权均值：num / den ---
imd_21_scores = num.div(den, axis=0)

# --- 4.4 同时聚合人口总和（可保留，便于解释/加权） ---
pop_sum = imd_map2.groupby("LSOA21CD")[COL_POP].sum().rename("imd_pop2015")

# --- 4.5 组装最终表 ---
imd_21 = (
    pd.concat([imd_21_scores, pop_sum], axis=1)
      .reset_index()
      .rename(columns={"LSOA21CD": "lsoa_code"})
)

imd_21["lsoa_code"] = imd_21["lsoa_code"].astype(str).str.strip().str.upper()
assert imd_21["lsoa_code"].duplicated().sum() == 0

# --- 4.6 改短列名（后续 PCA/回归更方便） ---
imd_21 = imd_21.rename(columns={
    "Index of Multiple Deprivation (IMD) Score__w": "imd_score",
    "Income Score (rate)__w": "imd_income_score",
    "Employment Score (rate)__w": "imd_employ_score",
    "Education, Skills and Training Score__w": "imd_edu_score",
    "Health Deprivation and Disability Score__w": "imd_health_score",
    "Crime Score__w": "imd_crime_score",
    "Barriers to Housing and Services Score__w": "imd_barriers_score",
    "Living Environment Score__w": "imd_livenv_score",
})

print("imd_21 shape:", imd_21.shape)

# ========= 5) join 回 master（LSOA21 + DPI） =========
master2 = master.merge(imd_21, on="lsoa_code", how="left")

miss_imd = int(master2["imd_score"].isna().sum())
print("Missing imd_score after join:", miss_imd, f"({miss_imd/len(master2):.2%})")

# ========= 6) 保存产物 =========
# 6.1 干净的 IMD(LSOA21) 表
imd_out_csv = OUT_DIR / "clean_imd2019_lsoa21_weighted.csv"
imd_21.to_csv(imd_out_csv, index=False)
print("saved:", imd_out_csv)

# 6.2 更新后的 master（dpi + imd）
master_out_gpkg = OUT_DIR / "master_lsoa21_dpi_imd.gpkg"
master2.to_file(master_out_gpkg, driver="GPKG", layer="lsoa_dpi_imd")
print("saved:", master_out_gpkg)



master shape: (4813, 11)
imd raw shape: (32844, 57)
duplicate lsoa11cd: 0
unmapped IMD rows (no LSOA21CD): 0
imd_21 shape: (32747, 10)
Missing imd_score after join: 0 (0.00%)
saved: QM data/clean/clean_imd2019_lsoa21_weighted.csv
saved: QM data/clean/master_lsoa21_dpi_imd.gpkg


In [14]:
import geopandas as gpd
from pathlib import Path

CLEAN_DIR = Path("QM data/clean")

MASTER_GPKG = CLEAN_DIR / "master_lsoa21_dpi_imd.gpkg"
LAYER = "lsoa_dpi_imd"   # 你保存时用的 layer 名；如果不确定我下面也给你查法

# 1) 读 master
gdf = gpd.read_file(MASTER_GPKG, layer=LAYER)

# 2) 导出 analysis_table_raw（不含 geometry）
analysis_table_raw = gdf.drop(columns="geometry").copy()
out_csv = CLEAN_DIR / "analysis_table_raw.csv"
analysis_table_raw.to_csv(out_csv, index=False)
print("Saved:", out_csv, "shape:", analysis_table_raw.shape)

# 3) 导出 geo_lsoa（只留 geometry）
geo_lsoa = gdf[["lsoa_code", "geometry"]].copy()
out_geo = CLEAN_DIR / "geo_lsoa.gpkg"
geo_lsoa.to_file(out_geo, driver="GPKG", layer="geo_lsoa")
print("Saved:", out_geo, "shape:", geo_lsoa.shape)

Saved: QM data/clean/analysis_table_raw.csv shape: (4813, 19)
Saved: QM data/clean/geo_lsoa.gpkg shape: (4813, 2)


In [15]:
## 5
import pandas as pd
import numpy as np
import re
from pathlib import Path

# =========================
# 0) Paths (只改这里)
# =========================
INFRA_PATH = Path("QM data/05/bba_all22.csv")  # <- 你的质量表（含 oa11cd + bba***_dow/uf/sfu）
OA_TO_LSOA11_LOOKUP = Path("QM data/Output_Area_to_Lower_layer_Super_Output_Area_to_Middle_layer_Super_Output_Area_to_Local_Authority_District_(December_2011)_Lookup_in_England_and_Wales.csv")  # <- 你需要的 OA11->LSOA11 对照表（建议带 AREAWT/POPWT）
LSOA11_TO_LSOA21_LOOKUP = Path("QM data/LSOA11_LSOA21_LAD22_EW_LU_v2_-7939559626189777291.csv")  # <- 你已经在用的 lookup
MASTER_TABLE = Path("QM data/clean/analysis_table_raw.csv")  # <- 你的主分析表（无 geometry 的那份；如果还没导出，就先从 master2 drop geometry 导出）

OUT_DIR = Path("QM data/clean")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 1) Read infra (OA11)
# =========================
infra = pd.read_csv(INFRA_PATH, dtype=str)
infra["oa11cd"] = infra["oa11cd"].astype(str).str.strip().str.upper()
assert infra["oa11cd"].isna().sum() == 0
assert infra["oa11cd"].duplicated().sum() == 0  # 一般每个 OA 一行；如果不是，先告诉我

# 把所有 bba***_* 列转成 numeric
bba_cols = [c for c in infra.columns if c.startswith("bba") and (c.endswith("_dow") or c.endswith("_uf") or c.endswith("_sfu"))]
for c in bba_cols:
    infra[c] = pd.to_numeric(infra[c], errors="coerce")

print("infra shape:", infra.shape)
print("bba cols:", len(bba_cols))

# =========================
# 2) 自动挑“最新一组”作为核心 proxy（可选）
#    例如 bba225_* 可能比 bba166_* 更新
# =========================
def latest_bba_cols(cols, suffix):
    # 找 bba(\d+)_suffix 的最大数字
    patt = re.compile(rf"^bba(\d+){suffix}$")
    hits = []
    for c in cols:
        m = patt.match(c)
        if m:
            hits.append((int(m.group(1)), c))
    if not hits:
        return None
    hits.sort()
    return hits[-1][1]  # 最大那组

latest_dow = latest_bba_cols(infra.columns, "_dow")
latest_uf  = latest_bba_cols(infra.columns, "_uf")
latest_sfu = latest_bba_cols(infra.columns, "_sfu")

print("latest dow:", latest_dow, "latest uf:", latest_uf, "latest sfu:", latest_sfu)

# 你后面做 PCA/聚类时，最常用就是 1 个 download + 1 个 upload
# 这里先给“核心列”一个统一命名（不影响你保留全量 bba_cols）
core_map = {}
if latest_dow: core_map[latest_dow] = "bb_q_dl_core"
if latest_uf:  core_map[latest_uf]  = "bb_q_ul_core"
if latest_sfu: core_map[latest_sfu] = "bb_q_sfu_core"

infra = infra.rename(columns=core_map)

# =========================
# 3) OA11 -> LSOA11 lookup
# =========================
oa_lsoa = pd.read_csv(OA_TO_LSOA11_LOOKUP, dtype=str)

# 尝试兼容常见列名：OA11CD / LSOA11CD 或 oa11cd / lsoa11cd
# 你如果列名不同，改这里两个 rename 即可
oa_lsoa = oa_lsoa.rename(columns={
    "OA11CD": "oa11cd",
    "LSOA11CD": "lsoa11cd",
    "oa11cd": "oa11cd",
    "lsoa11cd": "lsoa11cd",
})

oa_lsoa["oa11cd"] = oa_lsoa["oa11cd"].astype(str).str.strip().str.upper()
oa_lsoa["lsoa11cd"] = oa_lsoa["lsoa11cd"].astype(str).str.strip().str.upper()

# 权重列：优先用 AREAWT / POPWT / weight；没有就全当 1
weight_col = None
for cand in ["AREAWT", "POPWT", "weight", "WEIGHT", "areawt", "popwt"]:
    if cand in oa_lsoa.columns:
        weight_col = cand
        break

if weight_col is None:
    oa_lsoa["w"] = 1.0
else:
    oa_lsoa["w"] = pd.to_numeric(oa_lsoa[weight_col], errors="coerce").fillna(0)

print("OA->LSOA11 lookup shape:", oa_lsoa.shape, "weight_col:", weight_col)

# =========================
# 4) merge infra to lookup + 聚合到 LSOA11（加权均值）
# =========================
m = oa_lsoa.merge(infra, on="oa11cd", how="left", validate="m:1")

# 检查 OA 是否有匹配到质量数据
miss_oa = m[latest_dow].isna().sum() if latest_dow in m.columns else 0
print("missing core dl on merged rows:", miss_oa)

# 你可以选择：聚合全量 bba_cols；但模型通常只用 core 两列
agg_cols = ["bb_q_dl_core", "bb_q_ul_core", "bb_q_sfu_core"]
agg_cols = [c for c in agg_cols if c in m.columns]  # 存在才算

# 分母：每个 LSOA11 权重和
den11 = m.groupby("lsoa11cd")["w"].sum().rename("w_sum")

# 分子：每列乘权重后求和
num11 = (
    m.assign(**{f"{c}__w": m[c] * m["w"] for c in agg_cols})
     .groupby("lsoa11cd")[[f"{c}__w" for c in agg_cols]]
     .sum()
)

lsoa11_q = num11.div(den11, axis=0).reset_index()

# 改短列名
rename_back = {f"{c}__w": c for c in agg_cols}
lsoa11_q = lsoa11_q.rename(columns=rename_back)

print("lsoa11_q shape:", lsoa11_q.shape)

# =========================
# 5) LSOA11 -> LSOA21（用你已有 lookup），再聚合到 LSOA21（仍用权重）
# =========================
l11_l21 = pd.read_csv(LSOA11_TO_LSOA21_LOOKUP, dtype=str)
l11_l21["LSOA11CD"] = l11_l21["LSOA11CD"].astype(str).str.strip().str.upper()
l11_l21["LSOA21CD"] = l11_l21["LSOA21CD"].astype(str).str.strip().str.upper()

# 把 LSOA11 质量表映射到 LSOA21
m2 = l11_l21[["LSOA11CD", "LSOA21CD"]].merge(
    lsoa11_q, left_on="LSOA11CD", right_on="lsoa11cd", how="left"
)

# 这里没有额外权重的话，用 1；更严格可用 IMD 那个 pop2015 做权重（后面可以升级）
m2["w2"] = 1.0

den21 = m2.groupby("LSOA21CD")["w2"].sum().rename("w_sum")
num21 = (
    m2.assign(**{f"{c}__w": m2[c] * m2["w2"] for c in agg_cols})
      .groupby("LSOA21CD")[[f"{c}__w" for c in agg_cols]]
      .sum()
)
lsoa21_q = num21.div(den21, axis=0).reset_index().rename(columns={"LSOA21CD": "lsoa_code"})
lsoa21_q["lsoa_code"] = lsoa21_q["lsoa_code"].astype(str).str.strip().str.upper()
lsoa21_q = lsoa21_q.rename(columns={f"{c}__w": c for c in agg_cols})

print("lsoa21_q shape:", lsoa21_q.shape)

# =========================
# 6) join 回 master（analysis_table_raw）
# =========================
master = pd.read_csv(MASTER_TABLE, dtype=str)
master["lsoa_code"] = master["lsoa_code"].astype(str).str.strip().str.upper()

# 把质量列转 numeric（join 前后都能做）
for c in agg_cols:
    if c in lsoa21_q.columns:
        lsoa21_q[c] = pd.to_numeric(lsoa21_q[c], errors="coerce")

master2 = master.merge(lsoa21_q, on="lsoa_code", how="left")

for c in agg_cols:
    if c in master2.columns:
        miss = master2[c].isna().sum()
        print(f"Missing {c} after join:", miss, f"({miss/len(master2):.2%})")

# =========================
# 7) 保存产物
# =========================
# 7.1 LSOA21 质量表（干净）
out_q = OUT_DIR / "clean_infra_quality_lsoa21.csv"
lsoa21_q.to_csv(out_q, index=False)
print("saved:", out_q)

# 7.2 更新后的 analysis_table_raw
out_master = OUT_DIR / "analysis_table_raw_plus_quality.csv"
master2.to_csv(out_master, index=False)
print("saved:", out_master)


infra shape: (231975, 22)
bba cols: 21
latest dow: bba225_dow latest uf: bba225_uf latest sfu: bba225_sfu
OA->LSOA11 lookup shape: (181408, 10) weight_col: None
missing core dl on merged rows: 0
lsoa11_q shape: (34753, 4)
lsoa21_q shape: (34633, 4)
Missing bb_q_dl_core after join: 0 (0.00%)
Missing bb_q_ul_core after join: 0 (0.00%)
Missing bb_q_sfu_core after join: 0 (0.00%)
saved: QM data/clean/clean_infra_quality_lsoa21.csv
saved: QM data/clean/analysis_table_raw_plus_quality.csv


In [16]:
## 6
import pandas as pd
import numpy as np
import re
from pathlib import Path

# =========================
# 0) Paths（只改这里）
# =========================
CLEAN_DIR = Path("QM data/clean")

MASTER_TABLE = CLEAN_DIR / "analysis_table_raw_plus_quality.csv"   # 你刚刚质量那步输出的主分析表
CRIME_PATH   = Path("QM data/06/MPS LSOA Level Crime (most recent 24 months).csv")                 # <- 改成你的治安数据文件路径（csv/xlsx都行）
# 如果后面发现 crime 是 LSOA11 而 master 是 LSOA21，可用这个 lookup 做映射（可留着）
LSOA11_TO_LSOA21_LOOKUP = Path("QM data/LSOA11_LSOA21_LAD22_EW_LU_v2_-7939559626189777291.csv")

OUT_CLEAN_CRIME = CLEAN_DIR / "clean_crime_lsoa21.csv"
OUT_MASTER_PLUS = CLEAN_DIR / "analysis_table_raw_plus_quality_plus_crime.csv"

# 你要的时间窗口：默认最近 12 个月（从数据里自动取最后12个YYYYMM列）
WINDOW_N_MONTHS = 12

# =========================
# 1) Read master
# =========================
master = pd.read_csv(MASTER_TABLE, dtype=str)
master["lsoa_code"] = master["lsoa_code"].astype(str).str.strip().str.upper()

# 人口分母：优先用你 IMD 聚合来的 imd_pop2015（如果没有，就只能先输出“总量”，不做率）
pop_col = "imd_pop2015"
if pop_col in master.columns:
    master[pop_col] = pd.to_numeric(master[pop_col], errors="coerce")
else:
    print(f"[WARN] master 里没有 {pop_col}，将只输出 crime_count，不输出 per-1000 rate。")

print("master shape:", master.shape)

# =========================
# 2) Read crime
# =========================
if CRIME_PATH.suffix.lower() in [".xlsx", ".xls"]:
    crime = pd.read_excel(CRIME_PATH, dtype=str)
else:
    crime = pd.read_csv(CRIME_PATH, dtype=str)

# 统一列名（按你给的表头）
crime = crime.rename(columns={
    "LSOA Code": "lsoa_code",
    "LSOA Name": "lsoa_name",
    "Borough": "borough",
    "Major Category": "major_cat",
    "Minor Category": "minor_cat",
})
crime["lsoa_code"] = crime["lsoa_code"].astype(str).str.strip().str.upper()

print("crime raw shape:", crime.shape)

# =========================
# 3) 找出所有月份列 YYYYMM，并转为数值
# =========================
month_cols = [c for c in crime.columns if re.fullmatch(r"\d{6}", str(c))]
month_cols = sorted(month_cols)  # 按时间排序

if len(month_cols) == 0:
    raise ValueError("没有识别到月份列（形如 202312）。请检查列名是否真的是 YYYYMM。")

# 转数值
for c in month_cols:
    crime[c] = pd.to_numeric(crime[c], errors="coerce").fillna(0)

print("month cols:", month_cols[0], "...", month_cols[-1], "n=", len(month_cols))

# =========================
# 4) 选择时间窗口（默认最近12个月）
# =========================
use_months = month_cols[-WINDOW_N_MONTHS:] if len(month_cols) >= WINDOW_N_MONTHS else month_cols
print("using months:", use_months)

# =========================
# 5) 聚合到 LSOA：跨 Major/Minor 求和
#    产出两套：总量 &（可选）按 major_cat 的分解（便于后面做“风险环境结构”）
# =========================
# 5.1 总犯罪：每个 LSOA 最近 N 个月总量
crime_lsoa = (
    crime.groupby("lsoa_code", as_index=False)[use_months]
         .sum()
)

crime_lsoa["crime_count_window"] = crime_lsoa[use_months].sum(axis=1)
crime_lsoa = crime_lsoa[["lsoa_code", "crime_count_window"]].copy()

# 5.2 可选：按 major_cat 输出一个长表（做 appendix / EDA 很好用）
crime_major = (
    crime.groupby(["lsoa_code", "major_cat"], as_index=False)[use_months]
         .sum()
)
crime_major["crime_count_window"] = crime_major[use_months].sum(axis=1)
crime_major = crime_major[["lsoa_code", "major_cat", "crime_count_window"]].copy()

print("crime_lsoa shape:", crime_lsoa.shape)
print("crime_major shape:", crime_major.shape)

# 6) join 到 master（直接按 lsoa_code）
if "_merge" in master.columns:
    master = master.drop(columns=["_merge"])

master2 = master.merge(
    crime_lsoa,
    on="lsoa_code",
    how="left",
    indicator="_merge_crime"
)

miss = master2["crime_count_window"].isna().sum()
print("Missing crime_count_window after join:", miss, f"({miss/len(master2):.2%})")
print(master2["_merge_crime"].value_counts())

# ✅ 关键：把缺失当作 0（治安计数缺失通常可视为无记录=0）
master2["crime_count_window"] = master2["crime_count_window"].fillna(0)

# 8) 构造 crime rate（每千人）+ log1p
if pop_col in master2.columns:
    denom = pd.to_numeric(master2[pop_col], errors="coerce").replace({0: np.nan})
    master2["crime_rate_per_1000_window"] = master2["crime_count_window"] / denom * 1000
    master2["crime_rate_per_1000_window_log1p"] = np.log1p(master2["crime_rate_per_1000_window"])

master2 = master2.drop(columns=["_merge_crime"])

# =========================
# 9) 保存产物
# =========================
# clean crime 表（LSOA21 key + crime_count [+ rate 如果你想也可以另存]）
out_crime = master2[["lsoa_code", "crime_count_window"]].copy()
if "crime_rate_per_1000_window" in master2.columns:
    out_crime["crime_rate_per_1000_window"] = master2["crime_rate_per_1000_window"]

out_crime.to_csv(OUT_CLEAN_CRIME, index=False)
print("saved:", OUT_CLEAN_CRIME)

# 更新后的主分析表
master2.to_csv(OUT_MASTER_PLUS, index=False)
print("saved:", OUT_MASTER_PLUS)

# （可选）按 major_cat 的长表：用于 EDA/附录（如果你需要）
crime_major_out = CLEAN_DIR / "clean_crime_major_window_lsoa.csv"
crime_major.to_csv(crime_major_out, index=False)
print("saved:", crime_major_out)


master shape: (4813, 22)
crime raw shape: (105690, 29)
month cols: 202312 ... 202511 n= 24
using months: ['202412', '202501', '202502', '202503', '202504', '202505', '202506', '202507', '202508', '202509', '202510', '202511']
crime_lsoa shape: (4988, 2)
crime_major shape: (47616, 3)
Missing crime_count_window after join: 6 (0.12%)
_merge_crime
both          4807
left_only        6
right_only       0
Name: count, dtype: int64
saved: QM data/clean/clean_crime_lsoa21.csv
saved: QM data/clean/analysis_table_raw_plus_quality_plus_crime.csv
saved: QM data/clean/clean_crime_major_window_lsoa.csv


In [17]:
import pandas as pd
from pathlib import Path

MASTER_PLUS = Path("QM data/clean/analysis_table_raw_plus_quality_plus_crime.csv")
df = pd.read_csv(MASTER_PLUS, dtype=str)
df["lsoa_code"] = df["lsoa_code"].astype(str).str.strip().str.upper()

# 缺失行
miss = df[df["crime_count_window"].isna()].copy()
print("missing n:", len(miss))
print(miss[["lsoa_code"]].head(20))

# 如果你 master 里有 borough/lad 名称列，也一起看（列名按你文件实际改）
for cand in ["lad22nm", "borough", "lsoa_name"]:
    if cand in miss.columns:
        print("\nTop missing by", cand)
        print(miss[cand].value_counts().head(20))


missing n: 0
Empty DataFrame
Columns: [lsoa_code]
Index: []

Top missing by lad22nm
Series([], Name: count, dtype: int64)


In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 0) Paths（只改这里）
# =========================
CLEAN_DIR = Path("QM data/clean")

MASTER_TABLE   = CLEAN_DIR / "analysis_table_raw_plus_quality_plus_crime.csv"
EDUCATION_PATH = Path("QM data/07/Qualifications.xlsx")   # <- 改成你的教育数据文件路径（csv/xlsx都行）

OUT_CLEAN_EDU  = CLEAN_DIR / "clean_edu_lsoa21.csv"
OUT_MASTER_PLUS = CLEAN_DIR / "analysis_table_raw_plus_quality_plus_crime_plus_edu.csv"

# =========================
# 1) Read master
# =========================
master = pd.read_csv(MASTER_TABLE, dtype=str)
master["lsoa_code"] = master["lsoa_code"].astype(str).str.strip().str.upper()
print("master shape:", master.shape)

# =========================
# 2) Read education
# =========================
if EDUCATION_PATH.suffix.lower() in [".xlsx", ".xls"]:
    edu = pd.read_excel(EDUCATION_PATH, dtype=str)
else:
    edu = pd.read_csv(EDUCATION_PATH, dtype=str)

# 统一列名（按你给的表头）
edu = edu.rename(columns={
    "LSOA code": "lsoa_code",
    "local authority code": "lad_code",
    "local authority name": "lad_name",
    "Usual residents aged 16+": "res_16plus",
    "none": "q_none",
    "Level 1": "q_lvl1",
    "Level 2": "q_lvl2",
    "Apprentice-ship": "q_apprent",
    "Level 3": "q_lvl3",
    "Level 4+": "q_lvl4plus",
    "Other": "q_other",
})

# key 规范化
edu["lsoa_code"] = edu["lsoa_code"].astype(str).str.strip().str.upper()

# 把计数字段转成数值（缺失当 0；总人口 res_16plus 不应为 0/NA）
count_cols = ["res_16plus", "q_none", "q_lvl1", "q_lvl2", "q_apprent", "q_lvl3", "q_lvl4plus", "q_other"]
for c in count_cols:
    edu[c] = pd.to_numeric(edu[c], errors="coerce")

print("edu raw shape:", edu.shape)

# =========================
# 3) QA：是否一行一个 LSOA？如有重复就聚合
# =========================
dup = edu["lsoa_code"].duplicated().sum()
print("duplicate lsoa_code:", dup)

if dup > 0:
    # 对人数列求和；对名称列取第一个（它们应该一致）
    edu = (
        edu.groupby("lsoa_code", as_index=False)
           .agg(
               lad_code=("lad_code", "first"),
               lad_name=("lad_name", "first"),
               res_16plus=("res_16plus", "sum"),
               q_none=("q_none", "sum"),
               q_lvl1=("q_lvl1", "sum"),
               q_lvl2=("q_lvl2", "sum"),
               q_apprent=("q_apprent", "sum"),
               q_lvl3=("q_lvl3", "sum"),
               q_lvl4plus=("q_lvl4plus", "sum"),
               q_other=("q_other", "sum"),
           )
    )

# res_16plus 不能缺失
bad_total = edu["res_16plus"].isna().sum()
print("NA res_16plus:", bad_total)

# =========================
# 4) 构造比例变量（% of 16+）
# =========================
den = edu["res_16plus"].replace({0: np.nan})

edu["pct_no_qual"]      = edu["q_none"]     / den * 100
edu["pct_level4plus"]   = edu["q_lvl4plus"] / den * 100
edu["pct_level3"]       = edu["q_lvl3"]     / den * 100
edu["pct_level2"]       = edu["q_lvl2"]     / den * 100
edu["pct_level1"]       = edu["q_lvl1"]     / den * 100
edu["pct_apprent"]      = edu["q_apprent"]  / den * 100
edu["pct_other_qual"]   = edu["q_other"]    / den * 100

# 可选：一个“低技能风险”合成（无学历+L1 ）
edu["pct_low_skill"] = (edu["q_none"] + edu["q_lvl1"]) / den * 100

# （可选）如果你后面 PCA 想“方向统一：越高越风险”
# 无学历/低技能是风险（正向）；Level4+是保护（可做反向）
edu["risk_edu_low"] = edu["pct_low_skill"]
edu["risk_edu_highskill_inverse"] = 100 - edu["pct_level4plus"]  # 越高=高技能越少=风险越高

# 只保留要 join 的列（你也可以保留 lad_name 方便 QA）
edu_clean = edu[[
    "lsoa_code",
    "pct_no_qual",
    "pct_level4plus",
    "pct_low_skill",
    "risk_edu_low",
    "risk_edu_highskill_inverse",
]].copy()

# =========================
# 5) join 回 master（注意 indicator 改名防冲突）
# =========================
if "_merge" in master.columns:
    master = master.drop(columns=["_merge"])

master2 = master.merge(edu_clean, on="lsoa_code", how="left", indicator="_merge_edu")

miss = master2["pct_no_qual"].isna().sum()
print("Missing pct_no_qual after join:", miss, f"({miss/len(master2):.2%})")
print(master2["_merge_edu"].value_counts())

# 缺失处理：教育比例缺失通常很少；这里用“全伦敦中位数”填补（比填 0 更合理）
for c in ["pct_no_qual", "pct_level4plus", "pct_low_skill", "risk_edu_low", "risk_edu_highskill_inverse"]:
    med = pd.to_numeric(master2[c], errors="coerce").median()
    master2[c] = pd.to_numeric(master2[c], errors="coerce").fillna(med)

master2 = master2.drop(columns=["_merge_edu"])

# =========================
# 6) 保存
# =========================
edu_clean.to_csv(OUT_CLEAN_EDU, index=False)
print("saved:", OUT_CLEAN_EDU, "shape:", edu_clean.shape)

master2.to_csv(OUT_MASTER_PLUS, index=False)
print("saved:", OUT_MASTER_PLUS, "shape:", master2.shape)

master shape: (4813, 24)
edu raw shape: (4994, 11)
duplicate lsoa_code: 0
NA res_16plus: 0
Missing pct_no_qual after join: 0 (0.00%)
_merge_edu
both          4813
left_only        0
right_only       0
Name: count, dtype: int64
saved: QM data/clean/clean_edu_lsoa21.csv shape: (4994, 6)
saved: QM data/clean/analysis_table_raw_plus_quality_plus_crime_plus_edu.csv shape: (4813, 29)


In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 0) Paths（只改这里）
# =========================
CLEAN_DIR = Path("QM data/clean")

MASTER_TABLE = CLEAN_DIR / "analysis_table_raw_plus_quality_plus_crime_plus_edu.csv"
POP10_PATH   = Path("QM data/10/census2021-ts038-lsoa.csv")  # <- 改成你的文件路径（csv/xlsx都行）

OUT_CLEAN_POP10 = CLEAN_DIR / "clean_pop10_disability_lsoa21.csv"
OUT_MASTER_PLUS = CLEAN_DIR / "analysis_table_raw_plus_quality_plus_crime_plus_edu_plus_pop10.csv"

# （可选）如遇到 LSOA11/LSOA21 不一致，才用 lookup
LSOA11_TO_LSOA21_LOOKUP = Path("QM data/LSOA11_LSOA21_LAD22_EW_LU_v2_-7939559626189777291.csv")

# =========================
# 1) Read master
# =========================
master = pd.read_csv(MASTER_TABLE, dtype=str)
master["lsoa_code"] = master["lsoa_code"].astype(str).str.strip().str.upper()
print("master shape:", master.shape)

# =========================
# 2) Read pop10
# =========================
if POP10_PATH.suffix.lower() in [".xlsx", ".xls"]:
    pop = pd.read_excel(POP10_PATH, dtype=str)
else:
    pop = pd.read_csv(POP10_PATH, dtype=str)

# 统一列名
pop = pop.rename(columns={
    "geography code": "lsoa_code",
    "geography": "geography_name",
    "date": "date",
})

pop["lsoa_code"] = pop["lsoa_code"].astype(str).str.strip().str.upper()
pop["date"] = pd.to_datetime(pop["date"], errors="coerce")

print("pop raw shape:", pop.shape)
print("date range:", pop["date"].min(), "->", pop["date"].max())

# =========================
# 3) 选最新 date（同一 LSOA 可能有多个 date）
# =========================
latest_date = pop["date"].max()
pop_latest = pop[pop["date"] == latest_date].copy()
print("using latest date:", latest_date.date(), "shape:", pop_latest.shape)

# =========================
# 4) 数值列转为 numeric
# =========================
col_total = "Disability: Total: All usual residents"
col_disabled = "Disability: Disabled under the Equality Act"
col_lot = "Disability: Disabled under the Equality Act: Day-to-day activities limited a lot"
col_little = "Disability: Disabled under the Equality Act: Day-to-day activities limited a little"
col_not_disabled = "Disability: Not disabled under the Equality Act"
col_not_limited = ("Disability: Not disabled under the Equality Act: Has long term physical or mental health condition "
                   "but day-to-day activities are not limited")
col_no_condition = "Disability: Not disabled under the Equality Act: No long term physical or mental health conditions"

need_cols = [col_total, col_disabled, col_lot, col_little, col_not_disabled, col_not_limited, col_no_condition]
missing_cols = [c for c in need_cols if c not in pop_latest.columns]
if missing_cols:
    raise ValueError(f"这些列在你的文件里找不到（可能拼写不同）：{missing_cols}")

for c in need_cols:
    pop_latest[c] = pd.to_numeric(pop_latest[c], errors="coerce")

# =========================
# 5) QA：一行一个 LSOA？如有重复则聚合（求和）
# =========================
dup = pop_latest["lsoa_code"].duplicated().sum()
print("duplicate lsoa_code at latest date:", dup)

if dup > 0:
    pop_latest = (
        pop_latest.groupby("lsoa_code", as_index=False)[need_cols]
                  .sum()
    )
else:
    pop_latest = pop_latest[["lsoa_code"] + need_cols].copy()

# total 不能为 0
den = pop_latest[col_total].replace({0: np.nan})

# =========================
# 6) 构造核心比例变量（%）
# =========================
pop_latest["pct_disabled"] = pop_latest[col_disabled] / den * 100
pop_latest["pct_limited_a_lot"] = pop_latest[col_lot] / den * 100
pop_latest["pct_limited_a_little"] = pop_latest[col_little] / den * 100
pop_latest["pct_long_term_not_limited"] = pop_latest[col_not_limited] / den * 100
pop_latest["pct_no_long_term_condition"] = pop_latest[col_no_condition] / den * 100

# 可选：风险方向统一（越高=风险越高）
pop_latest["risk_disability_limited"] = (pop_latest[col_lot] + pop_latest[col_little]) / den * 100

pop_clean = pop_latest[[
    "lsoa_code",
    "pct_disabled",
    "pct_limited_a_lot",
    "pct_limited_a_little",
    "pct_long_term_not_limited",
    "pct_no_long_term_condition",
    "risk_disability_limited",
]].copy()

print("pop_clean shape:", pop_clean.shape)

# =========================
# 7) join 回 master（先直接按 lsoa_code）
# =========================
if "_merge" in master.columns:
    master = master.drop(columns=["_merge"])

master2 = master.merge(pop_clean, on="lsoa_code", how="left", indicator="_merge_pop10")
miss = master2["pct_disabled"].isna().sum()
print("Missing pct_disabled after join:", miss, f"({miss/len(master2):.2%})")
print(master2["_merge_pop10"].value_counts())

# =========================
# 8) 若缺失很多：尝试 LSOA11->LSOA21 映射（只有 miss 明显 >0 才值得）
# =========================
if miss > 0:
    print("[INFO] join 有缺失，尝试使用 LSOA11->LSOA21 lookup 映射...")
    lu = pd.read_csv(LSOA11_TO_LSOA21_LOOKUP, dtype=str)
    lu["LSOA11CD"] = lu["LSOA11CD"].astype(str).str.strip().str.upper()
    lu["LSOA21CD"] = lu["LSOA21CD"].astype(str).str.strip().str.upper()

    tmp = pop_clean.merge(lu[["LSOA11CD", "LSOA21CD"]], left_on="lsoa_code", right_on="LSOA11CD", how="left")
    unmapped = tmp["LSOA21CD"].isna().sum()
    print("unmapped after lookup:", unmapped)

    pop21 = (
        tmp.dropna(subset=["LSOA21CD"])
           .groupby("LSOA21CD", as_index=False)
           .agg({
               "pct_disabled": "mean",
               "pct_limited_a_lot": "mean",
               "pct_limited_a_little": "mean",
               "pct_long_term_not_limited": "mean",
               "pct_no_long_term_condition": "mean",
               "risk_disability_limited": "mean",
           })
           .rename(columns={"LSOA21CD": "lsoa_code"})
    )

    master2 = master.merge(pop21, on="lsoa_code", how="left", indicator="_merge_pop10_2")
    miss2 = master2["pct_disabled"].isna().sum()
    print("Missing pct_disabled after mapping:", miss2, f"({miss2/len(master2):.2%})")
    print(master2["_merge_pop10_2"].value_counts())
    master2 = master2.drop(columns=["_merge_pop10_2"])
else:
    master2 = master2.drop(columns=["_merge_pop10"])

# =========================
# 9) 缺失处理（如果仍有少量缺失：用中位数填补）
# =========================
for c in [
    "pct_disabled", "pct_limited_a_lot", "pct_limited_a_little",
    "pct_long_term_not_limited", "pct_no_long_term_condition", "risk_disability_limited"
]:
    master2[c] = pd.to_numeric(master2[c], errors="coerce")
    med = master2[c].median()
    master2[c] = master2[c].fillna(med)

# =========================
# 10) 保存
# =========================
pop_clean.to_csv(OUT_CLEAN_POP10, index=False)
print("saved:", OUT_CLEAN_POP10)

master2.to_csv(OUT_MASTER_PLUS, index=False)
print("saved:", OUT_MASTER_PLUS, "shape:", master2.shape)


master shape: (4813, 29)
pop raw shape: (35672, 10)
date range: 2021-01-01 00:00:00 -> 2021-01-01 00:00:00
using latest date: 2021-01-01 shape: (35672, 10)
duplicate lsoa_code at latest date: 0
pop_clean shape: (35672, 7)
Missing pct_disabled after join: 0 (0.00%)
_merge_pop10
both          4813
left_only        0
right_only       0
Name: count, dtype: int64
saved: QM data/clean/clean_pop10_disability_lsoa21.csv
saved: QM data/clean/analysis_table_raw_plus_quality_plus_crime_plus_edu_plus_pop10.csv shape: (4813, 35)


In [20]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

CLEAN_DIR = Path("QM data/clean")

MASTER_TABLE = CLEAN_DIR / "analysis_table_raw_plus_quality_plus_crime_plus_edu_plus_pop10.csv"
FUEL_PATH    = Path("QM data/11/Sub-regional_fuel_poverty_statistics_2023.xlsx")  # <- 改成你的实际路径

OUT_CLEAN_FUEL  = CLEAN_DIR / "clean_afford_fuelpoverty_lsoa.csv"
OUT_MASTER_PLUS = CLEAN_DIR / "analysis_table_raw_plus_quality_plus_crime_plus_edu_plus_pop10_plus_afford.csv"

# =========================
# 1) master
# =========================
master = pd.read_csv(MASTER_TABLE, dtype=str)
master["lsoa_code"] = master["lsoa_code"].astype(str).str.strip().str.upper()
print("master shape:", master.shape)

# =========================
# 2) fuel read
# =========================
if FUEL_PATH.suffix.lower() in [".xlsx", ".xls"]:
    fuel = pd.read_excel(FUEL_PATH, dtype=str)
else:
    fuel = pd.read_csv(FUEL_PATH, dtype=str)

print("fuel raw shape:", fuel.shape)
print("fuel columns:")
print(list(fuel.columns))

# ---- helper: 标准化列名（去空格/小写）
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip().lower()

cols_norm = {c: norm(c) for c in fuel.columns}

# =========================
# 3) 自动识别关键列
# =========================
# (A) 找 LSOA code 列：包含 "lsoa" + "code" 或者等价写法
lsoa_candidates = [c for c,n in cols_norm.items() if ("lsoa" in n and "code" in n)]
if not lsoa_candidates:
    # 有些文件可能叫 geography code
    lsoa_candidates = [c for c,n in cols_norm.items() if n in ["geography code", "geography_code"]]

if not lsoa_candidates:
    raise ValueError("找不到 LSOA code 列。请把 fuel.columns 贴出来我帮你精确匹配。")
LSOA_COL = lsoa_candidates[0]

# (B) households 总数列
hh_total_candidates = [c for c,n in cols_norm.items() if "household" in n and "number" in n and "fuel" not in n]
HH_TOTAL_COL = hh_total_candidates[0] if hh_total_candidates else None

# (C) fuel poor households 数
hh_fp_candidates = [c for c,n in cols_norm.items() if "household" in n and "fuel poverty" in n and ("number" in n or "count" in n)]
HH_FP_COL = hh_fp_candidates[0] if hh_fp_candidates else None

# (D) fuel poverty % 列
pct_candidates = [c for c,n in cols_norm.items() if ("proportion" in n or "%" in n or "percent" in n) and "fuel" in n]
PCT_COL = pct_candidates[0] if pct_candidates else None

print("Detected columns:",
      "\n  LSOA_COL =", LSOA_COL,
      "\n  HH_TOTAL_COL =", HH_TOTAL_COL,
      "\n  HH_FP_COL =", HH_FP_COL,
      "\n  PCT_COL =", PCT_COL)

# =========================
# 4) rename 成统一字段名
# =========================
rename_map = {LSOA_COL: "lsoa_code"}
if HH_TOTAL_COL: rename_map[HH_TOTAL_COL] = "hh_total"
if HH_FP_COL: rename_map[HH_FP_COL] = "hh_fuel_poor"
if PCT_COL: rename_map[PCT_COL] = "fuel_poverty_pct"

fuel = fuel.rename(columns=rename_map)

# key 规范化
fuel["lsoa_code"] = fuel["lsoa_code"].astype(str).str.strip().str.upper()

# =========================
# 5) 数值列转 numeric（允许某些文件没有 hh_total/hh_fuel_poor，只要 pct 有）
# =========================
for c in ["hh_total", "hh_fuel_poor", "fuel_poverty_pct"]:
    if c in fuel.columns:
        fuel[c] = pd.to_numeric(fuel[c], errors="coerce")

# =========================
# 6) clean 输出字段（最少要 pct）
# =========================
if "fuel_poverty_pct" not in fuel.columns:
    raise ValueError("你的 fuel poverty 文件里没有识别到百分比列（fuel_poverty_pct）。请把列名截图/复制给我。")

fuel_clean = fuel[["lsoa_code", "fuel_poverty_pct"]].copy()
fuel_clean["risk_fuel_poverty"] = fuel_clean["fuel_poverty_pct"]

# 若同时有 households，可做一致性 QA
if ("hh_total" in fuel.columns) and ("hh_fuel_poor" in fuel.columns):
    den = fuel["hh_total"].replace({0: np.nan})
    fuel_clean["fuel_poverty_pct_calc"] = fuel["hh_fuel_poor"] / den * 100
    fuel_clean["fuel_pct_abs_diff"] = (fuel_clean["fuel_poverty_pct"] - fuel_clean["fuel_poverty_pct_calc"]).abs()
    print("fuel_pct_abs_diff summary:")
    print(fuel_clean["fuel_pct_abs_diff"].describe())

# 去重：如果同一 lsoa 多行，取平均（通常不会发生）
dup = fuel_clean["lsoa_code"].duplicated().sum()
print("duplicate lsoa_code:", dup)
if dup > 0:
    fuel_clean = fuel_clean.groupby("lsoa_code", as_index=False).mean(numeric_only=True)

print("fuel_clean shape:", fuel_clean.shape)

# =========================
# 7) join 回 master
# =========================
if "_merge" in master.columns:
    master = master.drop(columns=["_merge"])

master2 = master.merge(fuel_clean[["lsoa_code", "fuel_poverty_pct", "risk_fuel_poverty"]],
                       on="lsoa_code", how="left", indicator="_merge_afford")

miss = master2["fuel_poverty_pct"].isna().sum()
print("Missing fuel_poverty_pct after join:", miss, f"({miss/len(master2):.2%})")
print(master2["_merge_afford"].value_counts())

# 缺失填补（如果有极少缺失）
master2["fuel_poverty_pct"] = pd.to_numeric(master2["fuel_poverty_pct"], errors="coerce")
med = master2["fuel_poverty_pct"].median()
master2["fuel_poverty_pct"] = master2["fuel_poverty_pct"].fillna(med)
master2["risk_fuel_poverty"] = pd.to_numeric(master2["risk_fuel_poverty"], errors="coerce").fillna(med)

master2 = master2.drop(columns=["_merge_afford"])

# =========================
# 8) save
# =========================
fuel_clean.to_csv(OUT_CLEAN_FUEL, index=False)
print("saved:", OUT_CLEAN_FUEL)

master2.to_csv(OUT_MASTER_PLUS, index=False)
print("saved:", OUT_MASTER_PLUS, "shape:", master2.shape)

master shape: (4813, 35)
fuel raw shape: (33758, 8)
fuel columns:
['LSOA Code', 'LSOA Name', 'Local Authority Code', 'Local Authority Name', 'Region', 'Number of households', 'Number of households in fuel poverty', 'Proportion of households fuel poor (%)']
Detected columns: 
  LSOA_COL = LSOA Code 
  HH_TOTAL_COL = Number of households 
  HH_FP_COL = Number of households in fuel poverty 
  PCT_COL = Proportion of households fuel poor (%)
fuel_pct_abs_diff summary:
count    33755.000000
mean         0.002490
std          0.001446
min          0.000000
25%          0.001244
50%          0.002497
75%          0.003740
max          0.005000
Name: fuel_pct_abs_diff, dtype: float64
duplicate lsoa_code: 1
fuel_clean shape: (33757, 5)
Missing fuel_poverty_pct after join: 0 (0.00%)
_merge_afford
both          4813
left_only        0
right_only       0
Name: count, dtype: int64
saved: QM data/clean/clean_afford_fuelpoverty_lsoa.csv
saved: QM data/clean/analysis_table_raw_plus_quality_plus_crime_

In [21]:
## 整合 rDERI 分数到主分析表
import pandas as pd
from pathlib import Path

CLEAN_DIR = Path("QM data/clean")

# 读取主分析表
MASTER_TABLE = CLEAN_DIR / "analysis_table_raw_plus_quality_plus_crime_plus_edu_plus_pop10_plus_afford.csv"
RDERI_CSV = Path("QM data/rDERI_LSOA_London_v1.6.csv")

master = pd.read_csv(MASTER_TABLE, dtype=str)
rderi = pd.read_csv(RDERI_CSV, dtype=str)

# 标准化 lsoa_code
master["lsoa_code"] = master["lsoa_code"].astype(str).str.strip().str.upper()
rderi["lsoa_code"] = rderi["lsoa_code"].astype(str).str.strip().str.upper()

print("Master shape:", master.shape)
print("rDERI shape:", rderi.shape)

# 合并 rDERI 分数
master_with_rderi = master.merge(
    rderi[["lsoa_code", "rDERI_score_0_10", "rDERI_decile", 
           "broadband_score_z", "deprivation_score_z", "demography_score_z"]],
    on="lsoa_code",
    how="left",
    indicator="_merge_rderi"
)

print("\n合并结果:")
print(master_with_rderi["_merge_rderi"].value_counts())
miss_rderi = master_with_rderi["rDERI_score_0_10"].isna().sum()
print(f"缺失 rDERI 分数: {miss_rderi} ({miss_rderi/len(master_with_rderi):.2%})")

# 转换数值列
rderi_cols = ["rDERI_score_0_10", "rDERI_decile", 
              "broadband_score_z", "deprivation_score_z", "demography_score_z"]
for c in rderi_cols:
    master_with_rderi[c] = pd.to_numeric(master_with_rderi[c], errors="coerce")

# 保存整合后的表
OUT_MASTER_FINAL = CLEAN_DIR / "analysis_table_final_with_rderi.csv"
master_with_rderi = master_with_rderi.drop(columns=["_merge_rderi"])
master_with_rderi.to_csv(OUT_MASTER_FINAL, index=False)
print(f"\n保存完整分析表: {OUT_MASTER_FINAL}")
print(f"最终表形状: {master_with_rderi.shape}")
print(f"总列数: {len(master_with_rderi.columns)}")

Master shape: (4813, 37)
rDERI shape: (4835, 7)

合并结果:
_merge_rderi
both          4659
left_only      154
right_only       0
Name: count, dtype: int64
缺失 rDERI 分数: 154 (3.20%)

保存完整分析表: QM data/clean/analysis_table_final_with_rderi.csv
最终表形状: (4813, 42)
总列数: 42


In [22]:
## 数据质量检查
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN_DIR = Path("QM data/clean")
MASTER_FINAL = CLEAN_DIR / "analysis_table_final_with_rderi.csv"

df = pd.read_csv(MASTER_FINAL, dtype=str)

print("=" * 60)
print("数据质量检查报告")
print("=" * 60)
print(f"\n数据集形状: {df.shape[0]} 行 × {df.shape[1]} 列")

# 1. 数据类型和缺失值检查
print("\n【1. 缺失值分析】")
print("-" * 60)

# 识别数值列和分类列
numeric_cols = []
categorical_cols = []
for col in df.columns:
    if col in ["lsoa_code", "lsoa21nm", "msoa21cd", "msoa21nm", "lad22cd", "lad22nm"]:
        categorical_cols.append(col)
        continue
    # 尝试转换为数值
    try:
        pd.to_numeric(df[col], errors="raise")
        numeric_cols.append(col)
    except:
        categorical_cols.append(col)

print(f"数值列数量: {len(numeric_cols)}")
print(f"分类列数量: {len(categorical_cols)}")

# 缺失值统计
missing_summary = []
for col in df.columns:
    missing_count = df[col].isna().sum()
    missing_pct = missing_count / len(df) * 100
    if missing_count > 0:
        missing_summary.append({
            "列名": col,
            "缺失数": missing_count,
            "缺失比例(%)": f"{missing_pct:.2f}"
        })

if missing_summary:
    missing_df = pd.DataFrame(missing_summary)
    print("\n有缺失值的列:")
    print(missing_df.to_string(index=False))
else:
    print("\n✓ 所有列都没有缺失值！")

# 2. 关键变量检查
print("\n【2. 关键变量检查】")
print("-" * 60)

key_vars = ["lsoa_code", "dpi_score", "imd_score", "rDERI_score_0_10", 
            "crime_count_window", "bb_q_dl_core", "pct_no_qual", 
            "pct_disabled", "fuel_poverty_pct"]

for var in key_vars:
    if var in df.columns:
        missing = df[var].isna().sum()
        if missing == 0:
            print(f"✓ {var}: 无缺失")
        else:
            print(f"✗ {var}: {missing} 缺失 ({missing/len(df):.2%})")

# 3. 异常值检查（使用 IQR 方法）
print("\n【3. 异常值检测（IQR方法）】")
print("-" * 60)

outlier_summary = []
for col in numeric_cols[:15]:  # 只检查前15个数值列
    try:
        values = pd.to_numeric(df[col], errors="coerce").dropna()
        if len(values) == 0:
            continue
        Q1 = values.quantile(0.25)
        Q3 = values.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = ((values < lower_bound) | (values > upper_bound)).sum()
        if outliers > 0:
            outlier_summary.append({
                "列名": col,
                "异常值数量": outliers,
                "异常值比例(%)": f"{outliers/len(values)*100:.2f}"
            })
    except:
        pass

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary)
    print("\n检测到异常值的列:")
    print(outlier_df.to_string(index=False))
    print("\n注意: 某些异常值可能是合理的（如极端贫困区域）")
else:
    print("\n✓ 未检测到明显异常值")

print("\n" + "=" * 60)

数据质量检查报告

数据集形状: 4813 行 × 42 列

【1. 缺失值分析】
------------------------------------------------------------
数值列数量: 36
分类列数量: 6

有缺失值的列:
                 列名  缺失数 缺失比例(%)
       dpi_ci_lower 4753   98.75
       dpi_ci_upper 4753   98.75
   rDERI_score_0_10  154    3.20
       rDERI_decile  154    3.20
  broadband_score_z  154    3.20
deprivation_score_z  154    3.20
 demography_score_z  154    3.20

【2. 关键变量检查】
------------------------------------------------------------
✓ lsoa_code: 无缺失
✓ dpi_score: 无缺失
✓ imd_score: 无缺失
✗ rDERI_score_0_10: 154 缺失 (3.20%)
✓ crime_count_window: 无缺失
✓ bb_q_dl_core: 无缺失
✓ pct_no_qual: 无缺失
✓ pct_disabled: 无缺失
✓ fuel_poverty_pct: 无缺失

【3. 异常值检测（IQR方法）】
------------------------------------------------------------

检测到异常值的列:
                列名  异常值数量 异常值比例(%)
         dpi_score     96     1.99
      dpi_ci_lower      5     8.33
      dpi_ci_upper      5     8.33
         imd_score      7     0.15
  imd_income_score     12     0.25
  imd_employ_score     44     0.91

In [23]:
## 探索性数据分析（描述性统计）
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN_DIR = Path("QM data/clean")
MASTER_FINAL = CLEAN_DIR / "analysis_table_final_with_rderi.csv"

df = pd.read_csv(MASTER_FINAL)

print("=" * 60)
print("探索性数据分析 - 描述性统计")
print("=" * 60)

# 选择关键数值变量
key_vars = {
    "数字倾向指数": "dpi_score",
    "IMD总分": "imd_score",
    "rDERI总分": "rDERI_score_0_10",
    "宽带下载速度(Mbps)": "bb_q_dl_core",
    "犯罪计数(12个月)": "crime_count_window",
    "犯罪率(每千人)": "crime_rate_per_1000_window",
    "无学历比例(%)": "pct_no_qual",
    "Level 4+比例(%)": "pct_level4plus",
    "残疾比例(%)": "pct_disabled",
    "燃料贫困比例(%)": "fuel_poverty_pct"
}

# 转换为数值并计算描述性统计
desc_stats = []
for label, col in key_vars.items():
    if col in df.columns:
        values = pd.to_numeric(df[col], errors="coerce").dropna()
        if len(values) > 0:
            desc_stats.append({
                "变量": label,
                "均值": f"{values.mean():.2f}",
                "中位数": f"{values.median():.2f}",
                "标准差": f"{values.std():.2f}",
                "最小值": f"{values.min():.2f}",
                "最大值": f"{values.max():.2f}",
                "缺失数": int(df[col].isna().sum())
            })

desc_df = pd.DataFrame(desc_stats)
print("\n关键变量描述性统计:")
print(desc_df.to_string(index=False))

# 相关系数矩阵（关键变量之间）
print("\n" + "=" * 60)
print("关键变量相关性分析")
print("=" * 60)

corr_vars = ["dpi_score", "imd_score", "rDERI_score_0_10", 
             "bb_q_dl_core", "crime_rate_per_1000_window",
             "pct_no_qual", "pct_disabled", "fuel_poverty_pct"]

# 确保所有变量都存在
corr_vars = [v for v in corr_vars if v in df.columns]

if len(corr_vars) >= 3:
    corr_data = df[corr_vars].apply(pd.to_numeric, errors="coerce")
    corr_matrix = corr_data.corr()
    
    print("\n相关系数矩阵 (仅显示绝对值>0.3的相关性):")
    print("-" * 60)
    
    # 显示显著相关性
    for i, var1 in enumerate(corr_vars):
        for var2 in corr_vars[i+1:]:
            corr_val = corr_matrix.loc[var1, var2]
            if abs(corr_val) > 0.3:
                direction = "正相关" if corr_val > 0 else "负相关"
                print(f"{var1} ↔ {var2}: {corr_val:.3f} ({direction})")

print("\n" + "=" * 60)

探索性数据分析 - 描述性统计

关键变量描述性统计:
           变量     均值    中位数    标准差   最小值      最大值  缺失数
       数字倾向指数   0.96   0.96   0.02  0.75     1.00    0
        IMD总分  21.48  20.36  10.90  2.33    64.68    0
      rDERI总分   1.13   1.12   0.16  0.56     2.08  154
 宽带下载速度(Mbps) 129.11 129.96  31.25 27.32   270.32    0
   犯罪计数(12个月) 170.66 112.00 345.67  0.00 13223.00    0
     犯罪率(每千人)  92.26  63.25 165.56  0.00  5335.74    0
     无学历比例(%)  16.30  16.10   6.35  1.52    39.92    0
Level 4+比例(%)  46.33  44.34  13.66 20.22    87.28    0
      残疾比例(%)  13.36  13.08   3.05  4.60    28.43    0
    燃料贫困比例(%)   9.63   8.98   3.17  1.86    22.79    0

关键变量相关性分析

相关系数矩阵 (仅显示绝对值>0.3的相关性):
------------------------------------------------------------
dpi_score ↔ rDERI_score_0_10: -0.394 (负相关)
dpi_score ↔ pct_no_qual: -0.338 (负相关)
dpi_score ↔ pct_disabled: -0.444 (负相关)
imd_score ↔ rDERI_score_0_10: 0.744 (正相关)
imd_score ↔ pct_no_qual: 0.634 (正相关)
imd_score ↔ pct_disabled: 0.537 (正相关)
imd_score ↔ fuel_poverty_pct: 0.

In [24]:
## 创建最终建模数据集（清理和标准化）
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN_DIR = Path("QM data/clean")
MASTER_FINAL = CLEAN_DIR / "analysis_table_final_with_rderi.csv"

df = pd.read_csv(MASTER_FINAL)

print("=" * 60)
print("创建建模数据集")
print("=" * 60)

# 1. 识别并转换所有数值列
numeric_cols = []
for col in df.columns:
    if col in ["lsoa_code", "lsoa21nm", "msoa21cd", "msoa21nm", "lad22cd", "lad22nm"]:
        continue
    try:
        pd.to_numeric(df[col], errors="raise")
        numeric_cols.append(col)
    except:
        pass

# 转换数值列
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"\n已转换 {len(numeric_cols)} 个数值列")

# 2. 处理缺失值策略
print("\n缺失值处理策略:")
print("-" * 60)

# 对于关键变量，使用中位数填补（如果缺失较少）
key_vars_to_impute = ["dpi_score", "imd_score", "rDERI_score_0_10", 
                      "bb_q_dl_core", "bb_q_ul_core", "crime_count_window",
                      "pct_no_qual", "pct_level4plus", "pct_disabled", 
                      "fuel_poverty_pct"]

for var in key_vars_to_impute:
    if var in df.columns:
        missing_before = df[var].isna().sum()
        if missing_before > 0 and missing_before < len(df) * 0.1:  # 缺失<10%才填补
            median_val = df[var].median()
            df[var] = df[var].fillna(median_val)
            print(f"  {var}: {missing_before} 个缺失值 → 用中位数 {median_val:.2f} 填补")

# 3. 创建建模用的数据集（移除分类列，保留编码）
modeling_df = df.drop(columns=["lsoa21nm", "msoa21cd", "msoa21nm", "lad22cd", "lad22nm"], errors="ignore")

# 4. 移除仍有大量缺失的列（缺失>50%）
print("\n移除缺失值过多的列:")
print("-" * 60)
cols_to_drop = []
for col in modeling_df.columns:
    if col == "lsoa_code":
        continue
    missing_pct = modeling_df[col].isna().sum() / len(modeling_df)
    if missing_pct > 0.5:
        cols_to_drop.append(col)
        print(f"  移除 {col}: 缺失 {missing_pct:.1%}")

modeling_df = modeling_df.drop(columns=cols_to_drop)

# 5. 最终缺失值检查
print("\n最终缺失值统计:")
print("-" * 60)
final_missing = modeling_df.isna().sum()
final_missing = final_missing[final_missing > 0].sort_values(ascending=False)
if len(final_missing) > 0:
    print(f"\n仍有 {len(final_missing)} 列存在缺失值:")
    for col, count in final_missing.head(10).items():
        print(f"  {col}: {count} ({count/len(modeling_df):.2%})")
else:
    print("✓ 所有列都已完整！")

# 6. 保存建模数据集
OUT_MODELING = CLEAN_DIR / "modeling_dataset_clean.csv"
modeling_df.to_csv(OUT_MODELING, index=False)

print(f"\n✓ 建模数据集已保存: {OUT_MODELING}")
print(f"  形状: {modeling_df.shape[0]} 行 × {modeling_df.shape[1]} 列")
print(f"  可用变量数: {len(modeling_df.columns) - 1} 个（不含lsoa_code）")

# 7. 保存变量说明
var_info = []
for col in modeling_df.columns:
    if col == "lsoa_code":
        var_info.append({"变量名": col, "类型": "标识符", "说明": "LSOA21代码"})
    else:
        dtype = "数值型" if pd.api.types.is_numeric_dtype(modeling_df[col]) else "分类型"
        missing = modeling_df[col].isna().sum()
        var_info.append({
            "变量名": col,
            "类型": dtype,
            "缺失数": missing,
            "缺失比例(%)": f"{missing/len(modeling_df)*100:.2f}"
        })

var_info_df = pd.DataFrame(var_info)
VAR_INFO_PATH = CLEAN_DIR / "modeling_dataset_variables_info.csv"
var_info_df.to_csv(VAR_INFO_PATH, index=False)
print(f"✓ 变量说明已保存: {VAR_INFO_PATH}")

print("\n" + "=" * 60)
print("数据准备完成！可以开始建模分析。")
print("=" * 60)

创建建模数据集

已转换 36 个数值列

缺失值处理策略:
------------------------------------------------------------
  rDERI_score_0_10: 154 个缺失值 → 用中位数 1.12 填补

移除缺失值过多的列:
------------------------------------------------------------
  移除 dpi_ci_lower: 缺失 98.8%
  移除 dpi_ci_upper: 缺失 98.8%

最终缺失值统计:
------------------------------------------------------------

仍有 4 列存在缺失值:
  rDERI_decile: 154 (3.20%)
  broadband_score_z: 154 (3.20%)
  deprivation_score_z: 154 (3.20%)
  demography_score_z: 154 (3.20%)

✓ 建模数据集已保存: QM data/clean/modeling_dataset_clean.csv
  形状: 4813 行 × 35 列
  可用变量数: 34 个（不含lsoa_code）
✓ 变量说明已保存: QM data/clean/modeling_dataset_variables_info.csv

数据准备完成！可以开始建模分析。


In [25]:
import re
import numpy as np
import pandas as pd

# -----------------------------
# 1) Helpers
# -----------------------------
def _clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def _to_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def _median_impute(df: pd.DataFrame, cols: list[str], group_col: str | None = None) -> pd.DataFrame:
    df = df.copy()
    if group_col is None:
        for c in cols:
            med = df[c].median(skipna=True)
            df[c] = df[c].fillna(med)
    else:
        for c in cols:
            df[c] = df.groupby(group_col)[c].transform(lambda s: s.fillna(s.median(skipna=True)))
    return df

def _zscore(s: pd.Series) -> pd.Series:
    mu = s.mean()
    sd = s.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd

def _minmax_0_10(s: pd.Series) -> pd.Series:
    mn = s.min()
    mx = s.max()
    if mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn) * 10

# -----------------------------
# 0) Paths (EDIT THESE)
# -----------------------------
DERI_CSV = "QM data/02/DERIdataset_v1.6.csv"   # <- your downloaded file from GitHub (Version 1.6)

# -----------------------------
# 2) Load DERI indicator table (ROBUST)
# -----------------------------
deri = pd.read_csv(DERI_CSV, dtype=str)
deri = _clean_cols(deri)

def _norm(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[%()\-–—,:+]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

col_norm = {c: _norm(c) for c in deri.columns}

def pick_col(must=(), any_of=(), not_any=()):
    """
    Return the first column whose normalized name:
      - contains all tokens in `must`
      - contains at least one token in `any_of` (if provided)
      - contains none of tokens in `not_any`
    """
    must = [m.lower() for m in must]
    any_of = [a.lower() for a in any_of]
    not_any = [n.lower() for n in not_any]
    for c, n in col_norm.items():
        if any(x in n for x in not_any):
            continue
        if not all(x in n for x in must):
            continue
        if any_of and not any(x in n for x in any_of):
            continue
        return c
    return None

# ---- Key column (LSOA code) ----
KEY = (
    pick_col(must=("lsoa", "code"), any_of=("2011",))  # e.g. "LSOA code (2011)"
    or pick_col(must=("lsoa", "code"))                 # e.g. "LSOA code"
)

if KEY is None:
    raise ValueError(f"Cannot find LSOA code column. First 30 cols: {list(deri.columns)[:30]}")

# ---- Region column (optional) ----
REGION_COL = pick_col(must=("region",), any_of=("name",))  # usually "Region name"

# -----------------------------
# 3) Resolve indicators (FUZZY MATCH)
# -----------------------------
# Broadband / quality
COL_UNABLE_30 = pick_col(must=("unable", "30"), any_of=("mbit", "mbps", "broadband"))
COL_BELOW_10  = pick_col(must=("less", "10"), any_of=("mbit", "mbps", "broadband", "connections"))
COL_AVG_DL    = pick_col(must=("average", "download", "speed"))

# Deprivation / socioecon
COL_IMD_SCORE = (
    pick_col(must=("imd", "score")) or
    pick_col(must=("index", "multiple", "deprivation"), any_of=("score",))
)
COL_UNEMP  = pick_col(must=("unemployment",), any_of=("rate",))
COL_NOQUAL = pick_col(must=("no", "qualifications"), any_of=("percentage", "residents", "16"))

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# IMPORTANT FIX: choose DE indicator correctly
# Priority:
#   1) "Percentage of population in social grade DE"
#   2) other percentage-based DE
#   3) "Social grade - DE" (but explicitly NOT "all HRP")
# Avoid:
#   - "Social grade - all HRP" (this is NOT DE share)
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
COL_DE = (
    pick_col(must=("percentage", "population", "social", "grade"), any_of=("de",)) or
    pick_col(must=("percentage", "social", "grade"), any_of=("de",)) or
    pick_col(must=("population", "social", "grade"), any_of=("de",)) or
    pick_col(must=("social", "grade", "de"), not_any=("all hrp",))
)

# Demography / vulnerability
COL_65PLUS  = pick_col(must=("65", "over"), any_of=("percentage", "population"))
COL_PENSION = pick_col(must=("pension", "credit"), any_of=("rate", "1000"))

# Disability / limitation
COL_LIMITED_T = pick_col(must=("day", "activities", "limited"), any_of=("percentage", "residents"))
COL_LIMITED_A_LOT = pick_col(must=("limited", "a", "lot"), any_of=("day",))
COL_LIMITED_A_LIT = pick_col(must=("limited", "a", "little"), any_of=("day",))

needed_core = {
    "unable_30": COL_UNABLE_30,
    "below_10":  COL_BELOW_10,
    "avg_dl":    COL_AVG_DL,
    "imd":       COL_IMD_SCORE,
    "unemp":     COL_UNEMP,
    "noqual":    COL_NOQUAL,
    "de":        COL_DE,
    "pct65":     COL_65PLUS,
    "pension":   COL_PENSION,
}

missing_core = [k for k, v in needed_core.items() if v is None]
if missing_core:
    raise ValueError(
        "DERI CSV missing required indicators (fuzzy matching failed): "
        f"{missing_core}\n"
        "Tip: print(deri.columns) and check if column wording is very different."
    )

# limited: accept either total column or (a lot + a little)
if COL_LIMITED_T is None and (COL_LIMITED_A_LOT is None or COL_LIMITED_A_LIT is None):
    raise ValueError(
        "Cannot find disability/day-to-day limitation indicator.\n"
        "Need either:\n"
        " - a total % limited column, OR\n"
        " - both 'limited a lot' and 'limited a little' columns."
    )

print("DERI key column:", KEY)
print("Resolved columns:")
for k, v in needed_core.items():
    print(f"  {k}: {v}")
print("  limited_total:", COL_LIMITED_T)
print("  limited_a_lot:", COL_LIMITED_A_LOT)
print("  limited_a_little:", COL_LIMITED_A_LIT)

# Safety check: ensure we did NOT accidentally pick "all HRP" for DE
if isinstance(COL_DE, str) and "all hrp" in _norm(COL_DE):
    raise ValueError(
        f"DE column wrongly resolved to '{COL_DE}'.\n"
        "You must use 'Percentage of population in social grade DE' (or similar), not 'all HRP'."
    )

# -----------------------------
# 4) Numeric conversion + impute
# -----------------------------
use_cols = [KEY]
if REGION_COL:
    use_cols.append(REGION_COL)

use_cols += [
    COL_UNABLE_30, COL_BELOW_10, COL_AVG_DL,
    COL_IMD_SCORE, COL_UNEMP, COL_NOQUAL, COL_DE,
    COL_65PLUS, COL_PENSION
]

if COL_LIMITED_T:
    use_cols.append(COL_LIMITED_T)
else:
    use_cols += [COL_LIMITED_A_LOT, COL_LIMITED_A_LIT]

df = deri[use_cols].copy()

# standardise key
df[KEY] = df[KEY].astype(str).str.strip().str.upper()

# numeric conversion
num_cols = [c for c in df.columns if c != KEY and c != REGION_COL]
df = _to_numeric(df, num_cols)

# build limited_total if needed
if COL_LIMITED_T is None:
    df["__limited_total"] = df[COL_LIMITED_A_LOT].fillna(0) + df[COL_LIMITED_A_LIT].fillna(0)
    LIMITED_COL = "__limited_total"
else:
    LIMITED_COL = COL_LIMITED_T

# impute (on scaling dataset)
df = _median_impute(df, [c for c in num_cols if c in df.columns])

# -----------------------------
# 5) Scoring (risk direction: higher = higher risk)
# -----------------------------
# scaling_mode:
#   "national": compute scaling on full dataset, THEN filter London for output
#   "london":   filter London first, THEN compute scaling
scaling_mode = "national"  # change to "london" if you prefer London-only scaling

if scaling_mode == "london":
    if REGION_COL:
        df = df[df[REGION_COL].astype(str).str.strip().str.lower().eq("london")].copy()
    else:
        print("[WARN] No Region column found; cannot filter London. Using full dataset.")

z = pd.DataFrame(index=df.index)

# Broadband component
z["unable_30_z"] = _zscore(df[COL_UNABLE_30])
z["below_10_z"]  = _zscore(df[COL_BELOW_10])
z["avg_dl_z"]    = _zscore(-df[COL_AVG_DL])  # reverse speed: higher speed => lower risk

# Deprivation component
z["imd_z"]    = _zscore(df[COL_IMD_SCORE])
z["unemp_z"]  = _zscore(df[COL_UNEMP])
z["noqual_z"] = _zscore(df[COL_NOQUAL])
z["de_z"]     = _zscore(df[COL_DE])

# Demography component
z["pct65_z"]   = _zscore(df[COL_65PLUS])
z["pension_z"] = _zscore(df[COL_PENSION])
z["limited_z"] = _zscore(df[LIMITED_COL])

df["broadband_score_z"]   = z[["unable_30_z", "below_10_z", "avg_dl_z"]].mean(axis=1)
df["deprivation_score_z"] = z[["imd_z", "unemp_z", "noqual_z", "de_z"]].mean(axis=1)
df["demography_score_z"]  = z[["pct65_z", "pension_z", "limited_z"]].mean(axis=1)

df["rDERI_overall_z"] = df[["broadband_score_z", "deprivation_score_z", "demography_score_z"]].mean(axis=1)
df["rDERI_score_0_10"] = _minmax_0_10(df["rDERI_overall_z"])
df["rDERI_decile"] = pd.qcut(df["rDERI_score_0_10"], 10, labels=False, duplicates="drop") + 1

# If national scaling, filter London now for output
if scaling_mode == "national" and REGION_COL:
    out = df[df[REGION_COL].astype(str).str.strip().str.lower().eq("london")].copy()
else:
    out = df.copy()

# -----------------------------
# 6) Output compact table
# -----------------------------
out = out.rename(columns={KEY: "lsoa_code"})  # standardise key name
rderi_out = out[["lsoa_code"] + ([REGION_COL] if REGION_COL else []) + [
    "rDERI_score_0_10",
    "rDERI_decile",
    "broadband_score_z",
    "deprivation_score_z",
    "demography_score_z",
]].copy()

rderi_out.to_csv("rDERI_LSOA_London_v1.6.csv", index=False)
print("Saved:", "rDERI_LSOA_London_v1.6.csv", "| rows:", len(rderi_out))

DERI key column: LSOA code
Resolved columns:
  unable_30: Percentage of homes unable to receive at least 30Mbit/s broadband
  below_10: Percentage of connections receiving less than 10Mbit/s broadband
  avg_dl: Average download speed (Mbit/s)
  imd: Index of Multiple Deprivation 2019 score - England base
  unemp: Unemployment rate
  noqual: Percentage of residents aged 16+ with no qualifications
  de: Percentage of population in social grade DE
  pct65: Percentage of population aged 65 and over
  pension: Guaranteed pension credit (rate per 1,000 aged 65+)
  limited_total: Percentage of residents whose day-to-day activities are limited
  limited_a_lot: Day-to-day activities limited a lot
  limited_a_little: Day-to-day activities limited a little
Saved: rDERI_LSOA_London_v1.6.csv | rows: 4835


In [26]:
import pandas as pd
from pathlib import Path

MASTER_IN = Path("QM data/clean/analysis_table_raw_plus_quality_plus_crime_plus_edu_plus_pop10_plus_afford.csv")
RDERI_IN  = Path("rDERI_LSOA_London_v1.6.csv")  # 你刚生成的

master = pd.read_csv(MASTER_IN, dtype={"lsoa_code": str})
rderi  = pd.read_csv(RDERI_IN, dtype={"lsoa_code": str})

master["lsoa_code"] = master["lsoa_code"].str.strip().str.upper()
rderi["lsoa_code"]  = rderi["lsoa_code"].str.strip().str.upper()

# 只拿需要列，避免重复/污染
rderi = rderi[["lsoa_code", "rDERI_score_0_10", "rDERI_decile"]].drop_duplicates("lsoa_code")

out = master.merge(rderi, on="lsoa_code", how="left")

print("master rows:", len(master))
print("rderi rows:", len(rderi))
print("missing rDERI after join:", out["rDERI_score_0_10"].isna().sum(), f"({out['rDERI_score_0_10'].isna().mean():.2%})")

OUT_PATH = Path("QM data/clean/analysis_table_raw_plus_all.csv")
out.to_csv(OUT_PATH, index=False)
print("saved:", OUT_PATH, "shape:", out.shape)

master rows: 4813
rderi rows: 4835
missing rDERI after join: 154 (3.20%)
saved: QM data/clean/analysis_table_raw_plus_all.csv shape: (4813, 39)


In [27]:
import pandas as pd
from pathlib import Path

MASTER_IN = Path("QM data/clean/analysis_table_raw_plus_all.csv")
RDERI_IN  = Path("rDERI_LSOA_London_v1.6.csv")

m = pd.read_csv(MASTER_IN, dtype={"lsoa_code": str})
r = pd.read_csv(RDERI_IN,  dtype={"lsoa_code": str})

m_codes = set(m["lsoa_code"].str.strip().str.upper())
r_codes = set(r["lsoa_code"].str.strip().str.upper())

print("only in master:", len(m_codes - r_codes))
print("only in rDERI :", len(r_codes - m_codes))
print("examples only in master:", list(m_codes - r_codes)[:15])
print("examples only in rDERI :", list(r_codes - m_codes)[:15])

only in master: 154
only in rDERI : 176
examples only in master: ['E01034383', 'E01035642', 'E01034592', 'E01035485', 'E01035695', 'E01034612', 'E01035670', 'E01034474', 'E01035650', 'E01034140', 'E01035697', 'E01035721', 'E01035678', 'E01035689', 'E01035716']
examples only in rDERI : ['E01033585', 'E01033698', 'E01000940', 'E01004562', 'E01004246', 'E01033576', 'E01004188', 'E01004318', 'E01001748', 'E01032805', 'E01001667', 'E01001045', 'E01001752', 'E01033574', 'E01002080']


In [28]:
import numpy as np
import pandas as pd
from pathlib import Path

RDERI_IN   = Path("rDERI_LSOA_London_v1.6.csv")   # 你刚生成的（London）
LOOKUP_CSV = Path("QM data/LSOA11_LSOA21_LAD22_EW_LU_v2_-7939559626189777291.csv")  # 改成你的真实路径
OUT_RDERI21 = Path("QM data/clean/clean_rderi_lsoa21.csv")

# 1) 读 rDERI（注意：你文件里不一定有 overall_z；如果没有，就用三组件平均）
r = pd.read_csv(RDERI_IN, dtype={"lsoa_code": str})
r["lsoa_code"] = r["lsoa_code"].str.strip().str.upper()

# 如果你在生成 rDERI 时保留了三组件 z，就可以更稳
# 你当前 rDERI 输出包含：
# broadband_score_z, deprivation_score_z, demography_score_z
if {"broadband_score_z","deprivation_score_z","demography_score_z"}.issubset(r.columns):
    r["rDERI_overall_z"] = r[["broadband_score_z","deprivation_score_z","demography_score_z"]].mean(axis=1)
else:
    # 退而求其次：用 0-10 分当作 proxy（不如 z 稳）
    r["rDERI_overall_z"] = pd.to_numeric(r["rDERI_score_0_10"], errors="coerce")

r = r.dropna(subset=["lsoa_code","rDERI_overall_z"]).copy()

# 2) 读 lookup（LSOA11CD -> LSOA21CD）
lu = pd.read_csv(LOOKUP_CSV, dtype=str)
lu.columns = [c.strip() for c in lu.columns]

need = {"LSOA11CD","LSOA21CD"}
if not need.issubset(lu.columns):
    raise ValueError(f"lookup 缺列：需要 {need}，但只有：{list(lu.columns)}")

lu["LSOA11CD"] = lu["LSOA11CD"].str.strip().str.upper()
lu["LSOA21CD"] = lu["LSOA21CD"].str.strip().str.upper()

# 3) 把 rDERI(L11) 映射到 L21
tmp = r.merge(lu[["LSOA11CD","LSOA21CD"]], left_on="lsoa_code", right_on="LSOA11CD", how="left")

print("unmapped rDERI rows (no LSOA21CD):", tmp["LSOA21CD"].isna().sum())

# best-fit：对同一个 LSOA21CD 的多个 LSOA11 取均值（你 lookup 没权重列时这是最常用做法）
r21 = (
    tmp.dropna(subset=["LSOA21CD"])
       .groupby("LSOA21CD", as_index=False)
       .agg(rDERI_overall_z=("rDERI_overall_z","mean"))
       .rename(columns={"LSOA21CD":"lsoa_code"})
)

# 4) 在 LSOA21（London 子集）上重新缩放到 0–10 + decile
mn, mx = r21["rDERI_overall_z"].min(), r21["rDERI_overall_z"].max()
r21["rDERI_score_0_10"] = (r21["rDERI_overall_z"] - mn) / (mx - mn) * 10 if mx != mn else 0.0
r21["rDERI_decile"] = pd.qcut(r21["rDERI_score_0_10"], 10, labels=False, duplicates="drop") + 1

r21.to_csv(OUT_RDERI21, index=False)
print("saved:", OUT_RDERI21, "shape:", r21.shape)

unmapped rDERI rows (no LSOA21CD): 0
saved: QM data/clean/clean_rderi_lsoa21.csv shape: (4813, 4)


In [29]:
import pandas as pd
from pathlib import Path

RDERI21_IN = Path("QM data/clean/clean_rderi_lsoa21.csv")
r21 = pd.read_csv(RDERI21_IN, dtype=str)

print("r21 columns:", list(r21.columns))
print(r21.head(3))

r21 columns: ['lsoa_code', 'rDERI_overall_z', 'rDERI_score_0_10', 'rDERI_decile']
   lsoa_code       rDERI_overall_z    rDERI_score_0_10 rDERI_decile
0  E01000001  -0.49485929495882813  2.8076960548232917            2
1  E01000002     0.202220828682695  4.9824173332275326            9
2  E01000003   -0.1612333860312738   3.848528146522441            6


In [30]:
import numpy as np
import pandas as pd
from pathlib import Path

MASTER_IN = Path("QM data/clean/analysis_table_raw_plus_all.csv")
RDERI21_IN = Path("QM data/clean/clean_rderi_lsoa21.csv")

m = pd.read_csv(MASTER_IN, dtype={"lsoa_code": str})
r21 = pd.read_csv(RDERI21_IN, dtype={"lsoa_code": str})

m["lsoa_code"] = m["lsoa_code"].str.strip().str.upper()
r21["lsoa_code"] = r21["lsoa_code"].str.strip().str.upper()

# 1) If rDERI_score_0_10 not present, create it from overall_z
if "rDERI_score_0_10" not in r21.columns:
    if "rDERI_overall_z" not in r21.columns:
        raise ValueError(
            "clean_rderi_lsoa21.csv has neither 'rDERI_score_0_10' nor 'rDERI_overall_z'. "
            f"Columns are: {list(r21.columns)}"
        )
    r21["rDERI_overall_z"] = pd.to_numeric(r21["rDERI_overall_z"], errors="coerce")
    mn, mx = r21["rDERI_overall_z"].min(), r21["rDERI_overall_z"].max()
    r21["rDERI_score_0_10"] = (r21["rDERI_overall_z"] - mn) / (mx - mn) * 10 if mx != mn else 0.0

# 2) If decile not present, build it
if "rDERI_decile" not in r21.columns:
    r21["rDERI_decile"] = pd.qcut(r21["rDERI_score_0_10"], 10, labels=False, duplicates="drop") + 1

# keep only needed, dedupe
r21 = r21[["lsoa_code", "rDERI_score_0_10", "rDERI_decile"]].copy()
r21 = r21.drop_duplicates("lsoa_code")

# 3) Remove old rDERI columns in master to avoid suffixes
for c in ["rDERI_score_0_10", "rDERI_decile"]:
    if c in m.columns:
        m = m.drop(columns=[c])

# 4) Merge
out = m.merge(r21, on="lsoa_code", how="left")

miss = out["rDERI_score_0_10"].isna().sum()
print("missing rDERI after LSOA21 mapping:", miss, f"({miss/len(out):.2%})")

OUT_PATH = Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv")
out.to_csv(OUT_PATH, index=False)
print("saved:", OUT_PATH, "shape:", out.shape)

missing rDERI after LSOA21 mapping: 0 (0.00%)
saved: QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv shape: (4813, 39)


In [31]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW_IN  = Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv")
RAW_OUT = Path("QM data/clean/analysis_table_raw.csv")
STD_OUT = Path("QM data/clean/analysis_table_std.csv")

df = pd.read_csv(RAW_IN, dtype={"lsoa_code": str})
df["lsoa_code"] = df["lsoa_code"].str.strip().str.upper()

# 你不想进入 PCA/聚类/ICS 的列（id + 类别/分位）
exclude = {"lsoa_code", "rDERI_decile"}  # decile 不进 PCA
# 如果你还有 rank/decile 类列，也可以加进 exclude

num_cols = [c for c in df.columns if c not in exclude]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 标准化（z-score）
std = df[["lsoa_code"]].copy()
for c in num_cols:
    x = df[c]
    mu = x.mean(skipna=True)
    sd = x.std(skipna=True, ddof=0)
    std[c] = 0.0 if (sd == 0 or np.isnan(sd)) else (x - mu) / sd

df.to_csv(RAW_OUT, index=False)
std.to_csv(STD_OUT, index=False)

print("saved raw:", RAW_OUT, "shape:", df.shape)
print("saved std:", STD_OUT, "shape:", std.shape)

saved raw: QM data/clean/analysis_table_raw.csv shape: (4813, 39)
saved std: QM data/clean/analysis_table_std.csv shape: (4813, 38)


In [32]:
# =========================
# PCA (PC1/PC2) + loadings + maps
# =========================
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import geopandas as gpd
import matplotlib.pyplot as plt

# -------------------------
# 0) Paths (EDIT)
# -------------------------
DATA_IN = Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv")

# 你的 LSOA21 边界（建议用 clean 的）
GEO_GPKG = Path("QM data/12/lsoa_boundary_clean.gpkg")
GEO_KEY_CANDIDATES = ["lsoa_code", "LSOA21CD", "lsoa21cd", "LSOA code", "LSOA Code"]

OUT_DIR = Path("QM data/output/4_1_")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# 1) Load master table
# -------------------------
df = pd.read_csv(DATA_IN, dtype={"lsoa_code": str})
df["lsoa_code"] = df["lsoa_code"].str.strip().str.upper()

print("master shape:", df.shape)

# -------------------------
# 2) Choose columns for PCA (robust)
#    - keep numeric "features"
#    - drop IDs / names / ranks / deciles / region texts etc.
# -------------------------
exclude_exact = {
    "lsoa_code",
    "rDERI_decile",  # 这种分箱列不进 PCA
}

exclude_keywords = [
    "name", "region", "nation", "local authority",
    "rank", "decile", "code",  # 注意：这里不影响 lsoa_code 因为上面已经排掉
    "_merge", "geometry"
]

def should_exclude(col: str) -> bool:
    c = col.strip().lower()
    if col in exclude_exact:
        return True
    return any(k in c for k in exclude_keywords)

# 候选列：非排除列
cand_cols = [c for c in df.columns if not should_exclude(c)]

# 转数值（不能转的会变 NaN，后面会筛掉）
X = df[cand_cols].apply(pd.to_numeric, errors="coerce")

# 删掉缺失太多的列（阈值你可调：0.2=允许最多20%缺失）
missing_rate = X.isna().mean().sort_values(ascending=False)
drop_cols = missing_rate[missing_rate > 0.20].index.tolist()
X = X.drop(columns=drop_cols)

# 对剩余列：用列中位数填补
X = X.fillna(X.median(numeric_only=True))

feature_cols = X.columns.tolist()
print("PCA features:", len(feature_cols))
print("Dropped (missing>20%):", len(drop_cols))
print("Top missing rates (kept):")
print(missing_rate.drop(index=drop_cols).head(10))

# -------------------------
# 3) Standardize + PCA
# -------------------------
scaler = StandardScaler()
X_std = scaler.fit_transform(X.values)

pca = PCA(n_components=5, random_state=0)
scores = pca.fit_transform(X_std)

evr = pca.explained_variance_ratio_
print("Explained variance ratio:", evr[:5])

# -------------------------
# 4) Loadings (PC1/PC2)
#    sklearn: components_ shape = (n_components, n_features)
#    loading = component weight (方向 + 强度)
# -------------------------
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

# 输出：PC1/PC2 loadings + 绝对值排序
L12 = loadings[["PC1","PC2"]].copy()
L12["abs_PC1"] = L12["PC1"].abs()
L12["abs_PC2"] = L12["PC2"].abs()
L12 = L12.sort_values("abs_PC1", ascending=False)

L12_out = OUT_DIR / "pca_loadings_PC1_PC2.csv"
L12.to_csv(L12_out)
print("saved:", L12_out)

# -------------------------
# 5) Scores table (lsoa + PC1/PC2)
# -------------------------
scores_df = pd.DataFrame({
    "lsoa_code": df["lsoa_code"].values,
    "PC1": scores[:, 0],
    "PC2": scores[:, 1],
})

scores_out = OUT_DIR / "pca_scores_PC1_PC2.csv"
scores_df.to_csv(scores_out, index=False)
print("saved:", scores_out)

# -------------------------
# 6) Auto-interpretation helper (top +/- loadings)
# -------------------------
def top_loadings(pc: str, k: int = 10):
    s = loadings[pc].sort_values(ascending=False)
    pos = s.head(k)
    neg = s.tail(k)
    return pos, neg

pc1_pos, pc1_neg = top_loadings("PC1", 10)
pc2_pos, pc2_neg = top_loadings("PC2", 10)

summary_txt = OUT_DIR / "pca_summary.txt"
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write("PCA summary\n")
    f.write(f"n_rows={len(df)} n_features={len(feature_cols)}\n\n")
    f.write("Explained variance ratio:\n")
    for i, r in enumerate(evr[:5], start=1):
        f.write(f"  PC{i}: {r:.4f}\n")

    f.write("\nTop + loadings (PC1):\n")
    for k, v in pc1_pos.items():
        f.write(f"  {k}: {v:.4f}\n")
    f.write("\nTop - loadings (PC1):\n")
    for k, v in pc1_neg.items():
        f.write(f"  {k}: {v:.4f}\n")

    f.write("\nTop + loadings (PC2):\n")
    for k, v in pc2_pos.items():
        f.write(f"  {k}: {v:.4f}\n")
    f.write("\nTop - loadings (PC2):\n")
    for k, v in pc2_neg.items():
        f.write(f"  {k}: {v:.4f}\n")

print("saved:", summary_txt)

# -------------------------
# 7) Map PC1/PC2 (join to geo)
# -------------------------
g = gpd.read_file(GEO_GPKG)

# 找几何表 key
geo_key = None
for c in GEO_KEY_CANDIDATES:
    if c in g.columns:
        geo_key = c
        break
if geo_key is None:
    raise ValueError(f"Cannot find LSOA key in geo file. Columns: {list(g.columns)[:50]}")

g = g.rename(columns={geo_key: "lsoa_code"})
g["lsoa_code"] = g["lsoa_code"].astype(str).str.strip().str.upper()

g2 = g.merge(scores_df, on="lsoa_code", how="left")

print("geo rows:", len(g2), "| missing PC1:", g2["PC1"].isna().mean(), "| missing PC2:", g2["PC2"].isna().mean())

# Plot PC1
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
g2.plot(column="PC1", ax=ax, legend=True)  # 默认配色即可
ax.set_title("PCA PC1 (standardized features)")
ax.set_axis_off()
pc1_png = OUT_DIR / "map_PC1.png"
plt.savefig(pc1_png, dpi=300, bbox_inches="tight")
plt.close()
print("saved:", pc1_png)

# Plot PC2
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
g2.plot(column="PC2", ax=ax, legend=True)
ax.set_title("PCA PC2 (standardized features)")
ax.set_axis_off()
pc2_png = OUT_DIR / "map_PC2.png"
plt.savefig(pc2_png, dpi=300, bbox_inches="tight")
plt.close()
print("saved:", pc2_png)


master shape: (4813, 39)
PCA features: 30
Dropped (missing>20%): 7
Top missing rates (kept):
pct_limited_a_lot                   0.0
pct_level4plus                      0.0
pct_low_skill                       0.0
risk_edu_low                        0.0
risk_edu_highskill_inverse          0.0
pct_disabled                        0.0
pct_no_long_term_condition          0.0
pct_limited_a_little                0.0
pct_long_term_not_limited           0.0
crime_rate_per_1000_window_log1p    0.0
dtype: float64
Explained variance ratio: [0.37846031 0.14161431 0.12409545 0.05998872 0.05524605]
saved: QM data/output/4_1_/pca_loadings_PC1_PC2.csv
saved: QM data/output/4_1_/pca_scores_PC1_PC2.csv
saved: QM data/output/4_1_/pca_summary.txt
geo rows: 4994 | missing PC1: 0.03624349219062875 | missing PC2: 0.03624349219062875
saved: QM data/output/4_1_/map_PC1.png
saved: QM data/output/4_1_/map_PC2.png


In [33]:
# 4.2-*- coding: utf-8 -*-
"""
PCA full outputs (variance + scree + loadings + PC1/PC2 maps + 4-quadrant map + result CSV)
NO fiona required.

Input:
  - QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv  (no geometry)
  - a GeoPackage with LSOA geometry + lsoa_code (your master gpkg)

Outputs (in QM data/output/pca/):
  - pca_explained_variance.csv
  - pca_loadings_pc1_pc2.csv
  - pca_scores_pc1_pc2.csv
  - pca_scores_with_quadrant.csv
  - figures: scree.png, loadings_pc1_top.png, loadings_pc2_top.png,
             map_pc1.png, map_pc2.png, map_quadrants.png, scatter_pc1_pc2.png
  - optional: pca_scores_maps.gpkg
"""

from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# =========================
# 0) Paths (EDIT if needed)
# =========================
ANALYSIS_CSV = Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv")

# Change this to your real gpkg if needed:
# e.g. "QM data/clean/master_lsoa21_plus_dpi_analysis.gpkg"
GEO_GPKG = Path("QM data/clean/master_lsoa21_dpi_imd.gpkg")

# If your gpkg contains multiple layers and geopandas fails / reads wrong,
# set this explicitly (common: "lsoa", "lsoa21", "layer1"...). Otherwise keep None.
GEO_LAYER = None

OUT_DIR = Path("QM data/output/4_2_pca")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


# =========================
# 1) Load analysis table
# =========================
df = pd.read_csv(ANALYSIS_CSV, dtype={"lsoa_code": str})
df["lsoa_code"] = df["lsoa_code"].astype(str).str.strip().str.upper()
df = df.drop_duplicates(subset=["lsoa_code"]).copy()
print("analysis rows:", len(df), "cols:", df.shape[1])


# =========================
# 2) Load geometry (no fiona)
# =========================
if not GEO_GPKG.exists():
    raise FileNotFoundError(f"GeoPackage not found: {GEO_GPKG}")

if GEO_LAYER is None:
    g = gpd.read_file(GEO_GPKG)  # default layer
else:
    g = gpd.read_file(GEO_GPKG, layer=GEO_LAYER)

# Ensure key column
if "lsoa_code" not in g.columns:
    for cand in ["lsoa21cd", "LSOA21CD", "LSOA code", "lsoa11cd", "LSOA11CD"]:
        if cand in g.columns:
            g = g.rename(columns={cand: "lsoa_code"})
            break

if "lsoa_code" not in g.columns:
    raise ValueError(f"Geo layer has no lsoa_code. Columns: {list(g.columns)[:30]}")

g["lsoa_code"] = g["lsoa_code"].astype(str).str.strip().str.upper()
g = g.drop_duplicates(subset=["lsoa_code"]).copy()

print("geo rows:", len(g), "cols:", g.shape[1])


# =========================
# 3) Choose PCA features
#    (Keep interpretable; skip missing automatically)
# =========================
FEATURES = [
    # Deprivation / socioecon
    "imd_score",
    "imd_income_score",
    "imd_employ_score",
    "imd_edu_score",
    "imd_health_score",
    "imd_barriers_score",
    "imd_crime_score",

    # Education structure
    "pct_no_qual",
    "pct_level4plus",

    # Digital inclusion / exclusion
    "dpi_score",
    "rDERI_score_0_10",

    # Broadband quality
    "bb_q_dl_core",
    "bb_q_ul_core",
    "bb_q_sfu_core",

    # Crime environment
    "crime_rate_per_1000_window_log1p",

    # Health/disability
    "pct_disabled",
    "pct_limited_a_lot",
    "pct_limited_a_little",

    # Affordability
    "fuel_poverty_pct",
]

# Drop some duplicates if present
DROP_IF_PRESENT = [
    "risk_edu_low", "risk_edu_highskill_inverse",
    "risk_disability_limited", "risk_fuel_poverty"
]

use_cols = [c for c in FEATURES if c in df.columns and c not in DROP_IF_PRESENT]
missing = [c for c in FEATURES if c not in df.columns]
print("Using features n=", len(use_cols))
if missing:
    print("[WARN] Missing features (skipped):", missing)

if len(use_cols) < 5:
    raise ValueError("Too few PCA features found. Check column names in your analysis CSV.")

X = df[["lsoa_code"] + use_cols].copy()

# numeric conversion
for c in use_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")

# median impute
med = X[use_cols].median(numeric_only=True)
X[use_cols] = X[use_cols].fillna(med)

print("PCA input shape:", X.shape)


# =========================
# 4) Standardize + PCA
# =========================
scaler = StandardScaler(with_mean=True, with_std=True)
Z = scaler.fit_transform(X[use_cols].to_numpy(dtype=float))

pca = PCA(n_components=2, random_state=RANDOM_STATE)
PC = pca.fit_transform(Z)

scores = pd.DataFrame({
    "lsoa_code": X["lsoa_code"].values,
    "PC1": PC[:, 0],
    "PC2": PC[:, 1],
})

# Explained variance
evr = pca.explained_variance_ratio_
ev_tbl = pd.DataFrame({
    "component": ["PC1", "PC2"],
    "explained_variance_ratio": evr,
    "explained_variance_ratio_pct": evr * 100
})
ev_path = OUT_DIR / "pca_explained_variance.csv"
ev_tbl.to_csv(ev_path, index=False)
print("Saved:", ev_path)

# Loadings
loadings = pd.DataFrame({
    "feature": use_cols,
    "PC1": pca.components_[0, :],
    "PC2": pca.components_[1, :],
})
loadings["abs_PC1"] = loadings["PC1"].abs()
loadings["abs_PC2"] = loadings["PC2"].abs()
load_path = OUT_DIR / "pca_loadings_pc1_pc2.csv"
loadings.sort_values("abs_PC1", ascending=False).to_csv(load_path, index=False)
print("Saved:", load_path)

# Save scores
score_path = OUT_DIR / "pca_scores_pc1_pc2.csv"
scores.to_csv(score_path, index=False)
print("Saved:", score_path)


# =========================
# 5) Quadrants
# =========================
def quadrant(pc1, pc2):
    if pc1 >= 0 and pc2 >= 0:
        return "Q1 (+PC1,+PC2)"
    if pc1 < 0 and pc2 >= 0:
        return "Q2 (-PC1,+PC2)"
    if pc1 < 0 and pc2 < 0:
        return "Q3 (-PC1,-PC2)"
    return "Q4 (+PC1,-PC2)"

scores["quadrant"] = [quadrant(a, b) for a, b in zip(scores["PC1"], scores["PC2"])]

quad_path = OUT_DIR / "pca_scores_with_quadrant.csv"
scores.to_csv(quad_path, index=False)
print("Saved:", quad_path)


# =========================
# 6) Plots: scree + loadings + scatter
# =========================
# Scree (PC1/PC2)
plt.figure()
plt.plot([1, 2], evr * 100, marker="o")
plt.xticks([1, 2], ["PC1", "PC2"])
plt.ylabel("Explained variance (%)")
plt.title("PCA explained variance (PC1, PC2)")
scree_path = OUT_DIR / "scree.png"
plt.savefig(scree_path, dpi=200, bbox_inches="tight")
plt.close()
print("Saved:", scree_path)

# Loadings barplots (top-k)
def plot_top_loadings(df_load, col, k=20, outname="loadings.png"):
    d = df_load.copy().sort_values(col, key=lambda s: s.abs(), ascending=False).head(k)
    plt.figure(figsize=(7, 5))
    plt.barh(d["feature"], d[col])
    plt.gca().invert_yaxis()
    plt.title(f"Top {k} loadings: {col}")
    plt.xlabel("loading")
    outpath = OUT_DIR / outname
    plt.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", outpath)

plot_top_loadings(loadings, "PC1", k=20, outname="loadings_pc1_top.png")
plot_top_loadings(loadings, "PC2", k=20, outname="loadings_pc2_top.png")

# Scatter
plt.figure(figsize=(6, 6))
for q, sub in scores.groupby("quadrant"):
    plt.scatter(sub["PC1"], sub["PC2"], s=8, alpha=0.6, label=q)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA scores (PC1 vs PC2)")
plt.legend(markerscale=2, fontsize=8, frameon=True)
scatter_path = OUT_DIR / "scatter_pc1_pc2.png"
plt.savefig(scatter_path, dpi=200, bbox_inches="tight")
plt.close()
print("Saved:", scatter_path)


# =========================
# 7) Join to geometry + maps
# =========================
geo = g[["lsoa_code", "geometry"]].copy()
geo = geo.merge(scores, on="lsoa_code", how="inner")
print("Geo+scores rows:", len(geo))

# Optional: project to Web Mercator for nicer plotting
try:
    geo_plot = geo.to_crs(3857)
except Exception:
    geo_plot = geo.copy()

# Map PC1
ax = geo_plot.plot(column="PC1", legend=True, figsize=(8, 6))
ax.set_axis_off()
ax.set_title("PCA PC1 (standardized features)")
pc1_map_path = OUT_DIR / "map_pc1.png"
plt.savefig(pc1_map_path, dpi=200, bbox_inches="tight")
plt.close()
print("Saved:", pc1_map_path)

# Map PC2
ax = geo_plot.plot(column="PC2", legend=True, figsize=(8, 6))
ax.set_axis_off()
ax.set_title("PCA PC2 (standardized features)")
pc2_map_path = OUT_DIR / "map_pc2.png"
plt.savefig(pc2_map_path, dpi=200, bbox_inches="tight")
plt.close()
print("Saved:", pc2_map_path)

# Quadrant map (categorical)
ax = geo_plot.plot(column="quadrant", categorical=True, legend=True, figsize=(8, 6))
ax.set_axis_off()
ax.set_title("PCA quadrants (sign of PC1 & PC2)")
quad_map_path = OUT_DIR / "map_quadrants.png"
plt.savefig(quad_map_path, dpi=200, bbox_inches="tight")
plt.close()
print("Saved:", quad_map_path)

# Optional: save a gpkg for QGIS
gpkg_out = OUT_DIR / "pca_scores_maps.gpkg"
try:
    geo.to_file(gpkg_out, driver="GPKG", layer="pca_scores")
    print("Saved:", gpkg_out)
except Exception as e:
    print("[WARN] Could not write gpkg (still fine). Reason:", e)

print("\nDONE. Outputs in:", OUT_DIR.resolve())

analysis rows: 4813 cols: 39
geo rows: 4813 cols: 20
Using features n= 19
PCA input shape: (4813, 20)
Saved: QM data/output/4_2_pca/pca_explained_variance.csv
Saved: QM data/output/4_2_pca/pca_loadings_pc1_pc2.csv
Saved: QM data/output/4_2_pca/pca_scores_pc1_pc2.csv
Saved: QM data/output/4_2_pca/pca_scores_with_quadrant.csv
Saved: QM data/output/4_2_pca/scree.png
Saved: QM data/output/4_2_pca/loadings_pc1_top.png
Saved: QM data/output/4_2_pca/loadings_pc2_top.png
Saved: QM data/output/4_2_pca/scatter_pc1_pc2.png
Geo+scores rows: 4813
Saved: QM data/output/4_2_pca/map_pc1.png
Saved: QM data/output/4_2_pca/map_pc2.png
Saved: QM data/output/4_2_pca/map_quadrants.png
Saved: QM data/output/4_2_pca/pca_scores_maps.gpkg

DONE. Outputs in: /home/jovyan/work/QM data/output/4_2_pca


In [34]:
# =========================
# 4.3 聚类分析（k=6 固定；用 PCA Top15 得分聚类）
# - 不改动 4.2 PCA 既有代码与输出
# - 避免 notebook 挂：不做全量 silhouette；n_init 降低
# =========================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEED = 42
K = 6
TOP_M = 15

# -------------------------
# 0) 准备输入矩阵：优先复用 4.2 已经算好的 X_std / feature_cols
# -------------------------
use_existing = ("X_std" in globals()) and ("feature_cols" in globals()) and ("df" in globals())
if use_existing:
    # 复用你 PCA cell 里的 df / feature_cols / X_std（不影响 4.2 输出）
    df_clu = df.copy()
    feats = list(feature_cols)
    Xz = np.asarray(X_std)
    # 防御：确保维度一致
    if Xz.shape[0] != len(df_clu) or Xz.shape[1] != len(feats):
        use_existing = False

if not use_existing:
    # 如果 4.2 没留变量/维度对不上，就从清洗表重建一份（也不会影响 4.2）
    RAW_CANDIDATES = [
        Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv"),
        Path("/mnt/data/analysis_table_raw_plus_all_plus_rderi.csv"),
    ]
    RAW_IN = next((p for p in RAW_CANDIDATES if p.exists()), None)
    if RAW_IN is None:
        raise FileNotFoundError(f"找不到输入数据文件，尝试过：{RAW_CANDIDATES}")

    df_clu = pd.read_csv(RAW_IN, dtype={"lsoa_code": str})
    df_clu["lsoa_code"] = df_clu["lsoa_code"].astype(str).str.strip().str.upper()

    # 跟你 PCA/聚类逻辑一致：排除 ID/分位等
    exclude = {"lsoa_code", "lsoa21nm", "msoa21cd", "msoa21nm", "lad22cd", "lad22nm", "rDERI_decile"}
    feats = [c for c in df_clu.columns if c not in exclude]

    X = df_clu[feats].apply(pd.to_numeric, errors="coerce")

    # 缺失：用中位数填补（只用于聚类，不改你之前 PCA 输出）
    for c in X.columns:
        X[c] = X[c].fillna(X[c].median(skipna=True))

    scaler = StandardScaler()
    Xz = scaler.fit_transform(X.values)

print("Clustering base matrix:", Xz.shape, "| features:", len(feats))

# -------------------------
# 1) PCA（取 Top15 主成分得分用于聚类输入）
# -------------------------
m = min(TOP_M, Xz.shape[1])
pca15 = PCA(n_components=m, random_state=SEED)
PC = pca15.fit_transform(Xz)

pc_cols = [f"PC{i}" for i in range(1, m + 1)]
pc_scores = pd.DataFrame(PC, columns=pc_cols)
pc_scores.insert(0, "lsoa_code", df_clu["lsoa_code"].values)

print(f"PCA for clustering fitted: n_components={m} | explained_var_top{m}=",
      round(pca15.explained_variance_ratio_.sum(), 4))

# -------------------------
# 2) KMeans（k=6 固定）
# -------------------------
km = KMeans(
    n_clusters=K,
    random_state=SEED,
    n_init=20,          # 比 50 轻很多，稳定性也够用
    init="k-means++"
)
labels0 = km.fit_predict(pc_scores[pc_cols].values)
pc_scores["cluster"] = labels0 + 1   # 写作更直观：1–6

# -------------------------
# 3) （可选）抽样 silhouette：避免 O(n^2) 把 kernel 打挂
# -------------------------
try:
    n = len(pc_scores)
    sample_n = min(1200, n)  # 抽样最多 1200
    rng = np.random.default_rng(SEED)
    idx = rng.choice(n, size=sample_n, replace=False)
    sil = silhouette_score(pc_scores.loc[idx, pc_cols].values, labels0[idx])
    print("Silhouette (sampled):", round(sil, 4), f"| sample_n={sample_n}")
except Exception as e:
    print("Silhouette skipped:", repr(e))

# -------------------------
# 4) 输出（label + profile）
# -------------------------
OUT_DIR = Path("QM data/output/4_3_clustering")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 4.1 labels + PC scores
OUT_LABEL = OUT_DIR / "cluster_labels_k6_pca15.csv"
pc_scores.to_csv(OUT_LABEL, index=False)
print("Saved:", OUT_LABEL)

# 4.2 profile：把 cluster 回连到原始表，做“原单位均值画像”
df_join = df_clu.merge(pc_scores[["lsoa_code", "cluster"]], on="lsoa_code", how="left")

# 只对数值列求均值（避免字符串列）
num_part = df_join.drop(columns=[c for c in df_join.columns if c in ["lsoa21nm","msoa21cd","msoa21nm","lad22cd","lad22nm"]], errors="ignore")
for c in num_part.columns:
    if c not in ["lsoa_code", "cluster"]:
        num_part[c] = pd.to_numeric(num_part[c], errors="coerce")

profile_raw = num_part.groupby("cluster", as_index=False).mean(numeric_only=True)
OUT_PROFILE = OUT_DIR / "cluster_profile_means_rawunits.csv"
profile_raw.to_csv(OUT_PROFILE, index=False)
print("Saved:", OUT_PROFILE)

# 4.3 cluster sizes
sizes = pc_scores["cluster"].value_counts().sort_index().rename_axis("cluster").reset_index(name="n")
OUT_SIZE = OUT_DIR / "cluster_sizes.csv"
sizes.to_csv(OUT_SIZE, index=False)
print("Saved:", OUT_SIZE)

print("\nDone: 4.3 clustering (k=6, PCA top15) complete.")


Clustering base matrix: (4813, 30) | features: 30
PCA for clustering fitted: n_components=15 | explained_var_top15= 0.9643
Silhouette (sampled): 0.1758 | sample_n=1200
Saved: QM data/output/4_3_clustering/cluster_labels_k6_pca15.csv
Saved: QM data/output/4_3_clustering/cluster_profile_means_rawunits.csv
Saved: QM data/output/4_3_clustering/cluster_sizes.csv

Done: 4.3 clustering (k=6, PCA top15) complete.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ---- paths（按你之前默认目录；如你改过目录就改这里）----
OUT_DIR = Path("QM data/output/4_3_clustering")
LABELS_FP = OUT_DIR / "cluster_labels_k6_pca15.csv"
RAW_FP = Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv")

# 兼容你这次上传的路径（如果你在本地目录没放）
if not RAW_FP.exists():
    RAW_FP = Path("/mnt/data/analysis_table_raw_plus_all_plus_rderi.csv")

# ---- load ----
labels = pd.read_csv(LABELS_FP, dtype={"lsoa_code": str})
labels["lsoa_code"] = labels["lsoa_code"].astype(str).str.strip().str.upper()

df = pd.read_csv(RAW_FP, dtype={"lsoa_code": str})
df["lsoa_code"] = df["lsoa_code"].astype(str).str.strip().str.upper()

# ---- join cluster back ----
dfj = df.merge(labels[["lsoa_code", "cluster"]], on="lsoa_code", how="inner")
if dfj["cluster"].isna().any():
    raise ValueError("有 lsoa_code 没 join 到 cluster，请检查主键/大小写/空格。")

# ---- rebuild the SAME feature selection logic used for clustering ----
exclude = {
    "lsoa_code","lsoa21nm","msoa21cd","msoa21nm","lad22cd","lad22nm",
    "dpi_score","dpi_ci_lower","dpi_ci_upper",
    "rDERI_score_0_10","rDERI_decile",
}
candidate = [c for c in dfj.columns if c not in exclude and c != "cluster"]

X = dfj[candidate].apply(pd.to_numeric, errors="coerce")

# drop too-sparse / near-constant columns (same spirit as clustering)
valid_cols = []
for c in X.columns:
    s = X[c]
    if s.notna().sum() < 50:
        continue
    if s.nunique(dropna=True) <= 2:
        continue
    valid_cols.append(c)

X = X[valid_cols].copy()

# median impute
for c in X.columns:
    X[c] = X[c].fillna(X[c].median(skipna=True))

# ---- standardize to z ----
scaler = StandardScaler()
Xz = scaler.fit_transform(X.values)

# ---- cluster mean z-profile ----
zdf = pd.DataFrame(Xz, columns=[f"{c}__z" for c in valid_cols])
zdf.insert(0, "cluster", dfj["cluster"].values)

profile_z = zdf.groupby("cluster", as_index=False).mean(numeric_only=True)

# ---- save ----
OUT_Z = OUT_DIR / "cluster_profile_means_z.csv"
profile_z.to_csv(OUT_Z, index=False)

print("Saved:", OUT_Z, "| shape:", profile_z.shape)
print("Top cols:", profile_z.columns[:10].tolist())


Saved: QM data/output/4_3_clustering/cluster_profile_means_z.csv | shape: (6, 29)
Top cols: ['cluster', 'imd_score__z', 'imd_income_score__z', 'imd_employ_score__z', 'imd_edu_score__z', 'imd_health_score__z', 'imd_crime_score__z', 'imd_barriers_score__z', 'imd_livenv_score__z', 'imd_pop2015__z']


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 你的 4.3 输出目录（按你之前代码的默认位置）
OUT_DIR = Path("QM data/output/4_3_clustering")

# 4.3 输出文件（你已生成）
labels_fp  = OUT_DIR / "cluster_labels_k6_pca15.csv"
profile_fp = OUT_DIR / "cluster_profile_means_rawunits.csv"
zprof_fp   = OUT_DIR / "cluster_profile_means_z.csv"
sizes_fp   = OUT_DIR / "cluster_sizes.csv"

labels  = pd.read_csv(labels_fp, dtype={"lsoa_code": str})
profile = pd.read_csv(profile_fp)
zprof   = pd.read_csv(zprof_fp)
sizes   = pd.read_csv(sizes_fp)

# 你原始清洗表（用于画“DPI分布/与基础设施关系”等 outcome 图）
RAW_IN = Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv")
df_raw = pd.read_csv(RAW_IN, dtype={"lsoa_code": str})
df_raw["lsoa_code"] = df_raw["lsoa_code"].astype(str).str.strip().str.upper()

# join cluster 回原始表
dfp = df_raw.merge(labels[["lsoa_code","cluster","PC1","PC2"]], on="lsoa_code", how="left")

# 存图
FIG_DIR = OUT_DIR / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name):
    p = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(p, dpi=220)
    plt.close()
    print("Saved:", p)

# -------------------------
# 图1：Cluster size（每类多少个 LSOA）
# -------------------------
plt.figure()
plt.bar(sizes["cluster"].astype(str), sizes["n"])
plt.xlabel("Cluster")
plt.ylabel("Number of LSOAs")
plt.title("Cluster sizes (k=6, PCA top15)")
savefig("fig_cluster_sizes.png")

# -------------------------
# 图2：PC1–PC2 平面：看“类型在结构轴线空间里怎么分布”
# -------------------------
plt.figure()
for k in sorted(dfp["cluster"].dropna().unique()):
    sub = dfp[dfp["cluster"] == k]
    plt.scatter(sub["PC1"], sub["PC2"], s=10, alpha=0.7, label=f"C{k}")
plt.xlabel("PC1 score")
plt.ylabel("PC2 score")
plt.title("Clusters in PCA space (PC1 vs PC2)")
plt.legend(markerscale=2, fontsize=8)
savefig("fig_pc1_pc2_clusters0.png")

# -------------------------
# 图2-更新：PC1–PC2 平面：看“类型在结构轴线空间里怎么分布”
# -------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# -----------------------------
# Config you SHOULD edit (human-readable axis + type names)
# -----------------------------
TITLE_MAIN = "Six Neighbourhood Types Emerge from Multivariate Patterns"
TITLE_SUB  = "Clusters Shown in PCA Space (PC1 vs PC2)"  # keep the technical bit as a subtitle

# Replace these with your interpreted PC meanings from loadings
PC1_HUMAN = "PC1 (higher = higher risk)"
PC2_HUMAN = "PC2 (higher = higher risk)"

# Replace these with your real typology names once finalised
CLUSTER_NAMES = {
    1: "C1 – Engaged despite constraints",
    2: "C2 – Missing middle (good broadband, low participation)",
    3: "C3 – Mid-risk mix (mixed pressures)",
    4: "C4 – Stacked disadvantage (low participation + high pressure)",
    5: "C5 – Connected and engaged (high broadband, high participation)",
    6: "C6 – Connectivity constrained (slower broadband)",
}

# Overplotting controls
POINT_SIZE = 6
POINT_ALPHA = 0.12  # lower = less overplotting
CENTROID_SIZE = 80

# -----------------------------
# Helper: pull explained variance if a PCA model exists
# -----------------------------
def _get_pca_evr():
    for obj_name in ["pca", "pca_model", "pca_fit", "pca_"]:
        obj = globals().get(obj_name, None)
        if obj is not None and hasattr(obj, "explained_variance_ratio_"):
            evr = np.asarray(obj.explained_variance_ratio_).ravel()
            if evr.size >= 2:
                return float(evr[0]), float(evr[1])
    # If you stored it somewhere else, add another branch here.
    return None

# -----------------------------
# Helper: 95% confidence ellipse (no scipy dependency)
# -----------------------------
def add_confidence_ellipse(ax, x, y, edgecolor=None, lw=1.6, alpha=0.9):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size < 10 or np.any(~np.isfinite(x)) or np.any(~np.isfinite(y)):
        return

    cov = np.cov(x, y)
    if not np.all(np.isfinite(cov)) or np.linalg.det(cov) <= 0:
        return

    # eigen-decomposition for ellipse axis lengths / rotation
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals = vals[order]
    vecs = vecs[:, order]

    # 95% chi-square for 2 DOF ≈ 5.991
    chi2_95 = 5.991
    width, height = 2.0 * np.sqrt(vals * chi2_95)

    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    ell = Ellipse(
        (np.mean(x), np.mean(y)),
        width=width,
        height=height,
        angle=angle,
        fill=False,
        edgecolor=edgecolor,
        linewidth=lw,
        alpha=alpha,
    )
    ax.add_patch(ell)

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(9, 6.5))

# Keep a stable colour per cluster
# (tab10 is fine for <=10 clusters)
cmap = plt.get_cmap("tab10")
clusters = sorted(dfp["cluster"].dropna().unique())

# Plot all points faintly (NO legend labels here to avoid huge legend clutter)
for i, k in enumerate(clusters):
    sub = dfp[dfp["cluster"] == k]
    color = cmap(i % 10)

    ax.scatter(
        sub["PC1"], sub["PC2"],
        s=POINT_SIZE,
        alpha=POINT_ALPHA,
        color=color,
        linewidths=0,
        rasterized=True
    )

# Overlay centroids + 95% ellipse (legend uses these, not the full point cloud)
legend_handles = []
legend_labels = []

for i, k in enumerate(clusters):
    sub = dfp[dfp["cluster"] == k]
    color = cmap(i % 10)

    x = sub["PC1"].to_numpy()
    y = sub["PC2"].to_numpy()

    # centroid
    cx, cy = np.nanmean(x), np.nanmean(y)
    h = ax.scatter(
        [cx], [cy],
        s=CENTROID_SIZE,
        color=color,
        edgecolor="black",
        linewidth=0.6,
        zorder=5
    )

    # ellipse (optional but recommended)
    add_confidence_ellipse(ax, x, y, edgecolor=color, lw=1.6, alpha=0.95)

    legend_handles.append(h)
    legend_labels.append(CLUSTER_NAMES.get(int(k), f"C{k}"))

# Labels in "human" form
ax.set_xlabel(PC1_HUMAN)
ax.set_ylabel(PC2_HUMAN)

# Titles: conclusion-first + technical subtitle
fig.suptitle(TITLE_MAIN, fontsize=14, y=0.98)
ax.set_title(TITLE_SUB, fontsize=11, pad=10)

# Add explained variance annotation (top-left inside plot) if available
evr = _get_pca_evr()
if evr is not None:
    pc1_pct, pc2_pct = evr[0] * 100, evr[1] * 100
    ax.text(
        0.02, 0.98,
        f"Variance explained: PC1 {pc1_pct:.1f}%, PC2 {pc2_pct:.1f}%",
        transform=ax.transAxes,
        va="top", ha="left",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.9),
    )

# Cleaner legend (centroids only) with human-readable type names
ax.legend(
    legend_handles, legend_labels,
    title="Neighbourhood types",
    markerscale=1.0,
    fontsize=8,
    title_fontsize=9,
    frameon=True,
    loc="upper right"
)

ax.grid(True, alpha=0.25)
fig.tight_layout()

savefig("fig_pc1_pc2_clusters.png")



# -------------------------
# 图3：Outcome（DPI）按 cluster 的分布（箱线图）
# 用来支撑“同一结构类型下数字倾向分布差异”
# -------------------------
if "dpi_score" in dfp.columns:
    data = [dfp.loc[dfp["cluster"] == k, "dpi_score"].dropna().values
            for k in sorted(dfp["cluster"].dropna().unique())]
    plt.figure()
    plt.boxplot(data, labels=[str(k) for k in sorted(dfp["cluster"].dropna().unique())], showfliers=False)
    plt.xlabel("Cluster")
    plt.ylabel("DPI score")
    plt.title("Outcome (DPI) distribution by cluster")
    savefig("fig_dpi_by_cluster_boxplot.png")

# -------------------------
# 图4：DPI vs 基础设施质量（例：download proxy）按 cluster 上色
# 对应你研究方案的“网好≠参与一定高”的验证图思路 :contentReference[oaicite:2]{index=2}
# -------------------------
speed_col = "bb_q_dl_core" if "bb_q_dl_core" in dfp.columns else None
if speed_col and "dpi_score" in dfp.columns:
    plt.figure()
    for k in sorted(dfp["cluster"].dropna().unique()):
        sub = dfp[dfp["cluster"] == k]
        plt.scatter(sub[speed_col], sub["dpi_score"], s=10, alpha=0.7, label=f"C{k}")
    plt.xlabel(speed_col)
    plt.ylabel("DPI score")
    plt.title("DPI vs broadband quality (download proxy), by cluster")
    plt.legend(markerscale=2, fontsize=8)
    savefig("fig_scatter_dpi_vs_dl_by_cluster0.png")

# -------------------------
# 图4更新：DPI vs 基础设施质量（例：download proxy）按 cluster 上色
# 对应你研究方案的“网好≠参与一定高”的验证图思路 :contentReference[oaicite:2]{index=2}
# -------------------------
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Inputs + sanity checks
# -----------------------------
speed_col = "bb_q_dl_core" if "bb_q_dl_core" in dfp.columns else None
dpi_col = "dpi_score" if "dpi_score" in dfp.columns else None
cluster_col = "cluster" if "cluster" in dfp.columns else None

if not (speed_col and dpi_col and cluster_col):
    raise ValueError("Missing required columns: need cluster, dpi_score, and bb_q_dl_core in dfp.")

# Optional: cluster names (improves legend/annotations)
CLUSTER_NAMES = {
    1: "C1",
    2: "C2 – Missing middle",
    3: "C3",
    4: "C4",
    5: "C5",
    6: "C6",
}

# Thresholds for “missing middle” quadrant
dpi_p10 = float(dfp[dpi_col].quantile(0.10))               # lowest 10% DPI
spd_med = float(dfp[speed_col].quantile(0.50))             # median speed
spd_p75 = float(dfp[speed_col].quantile(0.75))             # upper quartile speed (optional alt line)

# Use one of these as the "good broadband" threshold
GOOD_SPEED = spd_p75  # change to spd_med if you prefer median

# -----------------------------
# Option A (recommended): per-cluster box + jitter (two clear comparisons)
# -----------------------------
import numpy as np
import matplotlib.pyplot as plt

speed_col = "bb_q_dl_core" if "bb_q_dl_core" in dfp.columns else None
dpi_col = "dpi_score" if "dpi_score" in dfp.columns else None
cluster_col = "cluster" if "cluster" in dfp.columns else None

if not (speed_col and dpi_col and cluster_col):
    raise ValueError("Missing required columns: need cluster, dpi_score, and bb_q_dl_core in dfp.")

CLUSTER_NAMES = {
    1: "C1",
    2: "C2 – Missing middle",
    3: "C3",
    4: "C4",
    5: "C5",
    6: "C6",
}

dpi_p10 = float(dfp[dpi_col].quantile(0.10))
spd_p75 = float(dfp[speed_col].quantile(0.75))
GOOD_SPEED = spd_p75

clusters = sorted(dfp[cluster_col].dropna().unique())
pos = np.arange(1, len(clusters) + 1)

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10, 7.8),
    gridspec_kw={"height_ratios": [1, 1]},
    sharex=True
)

# tighter title spacing (less whitespace)
fig.subplots_adjust(top=0.90, hspace=0.08)

rng = np.random.default_rng(42)

# ---- Panel 1: DPI by cluster
dpi_data = [dfp.loc[dfp[cluster_col] == k, dpi_col].dropna().to_numpy() for k in clusters]
ax1.boxplot(dpi_data, positions=pos, widths=0.55, showfliers=False)

for i, k in enumerate(clusters, start=1):
    sub = dfp.loc[dfp[cluster_col] == k, dpi_col].dropna()
    if len(sub) > 1200:
        sub = sub.sample(1200, random_state=42)
    xj = i + rng.normal(0, 0.08, size=len(sub))
    ax1.scatter(xj, sub.to_numpy(), s=8, alpha=0.18, linewidths=0, rasterized=True)

ax1.set_ylabel("Digital participation (DPI, 0–1)")
ax1.set_title("Who is digitally excluded despite decent broadband?", fontsize=13, pad=6)

# threshold line (keep the line)
ax1.axhline(dpi_p10, linestyle="--", linewidth=1)

# move text to top-left (NO overlap with the line/box)
ax1.text(
    0.02, 0.96,
    f"Low DPI threshold (bottom 10%): {dpi_p10:.3f}",
    transform=ax1.transAxes,
    va="top", ha="left", fontsize=10,
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.75", alpha=0.85)
)

ax1.grid(True, alpha=0.25)

# optional: one minimal annotation (keep it clean)
if 2 in [int(k) for k in clusters]:
    idx = [int(k) for k in clusters].index(2) + 1
    y_med = float(np.median(dpi_data[idx-1]))
    ax1.annotate(
        "Overlooked barriers:\nlow DPI despite decent broadband",
        xy=(idx, y_med),
        xytext=(idx + 0.9, y_med - 0.02),
        arrowprops=dict(arrowstyle="->", linewidth=1),
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.75", alpha=0.85),
        ha="left", va="top"
    )

# ---- Panel 2: speed by cluster
spd_data = [dfp.loc[dfp[cluster_col] == k, speed_col].dropna().to_numpy() for k in clusters]
ax2.boxplot(spd_data, positions=pos, widths=0.55, showfliers=False)

for i, k in enumerate(clusters, start=1):
    sub = dfp.loc[dfp[cluster_col] == k, speed_col].dropna()
    if len(sub) > 1200:
        sub = sub.sample(1200, random_state=42)
    xj = i + rng.normal(0, 0.08, size=len(sub))
    ax2.scatter(xj, sub.to_numpy(), s=8, alpha=0.18, linewidths=0, rasterized=True)

ax2.set_ylabel("Download speed (Mbps, proxy)")
ax2.set_xlabel("Neighbourhood type (cluster)")

ax2.axhline(GOOD_SPEED, linestyle="--", linewidth=1)

# move text to top-left (again, avoid line overlap)
ax2.text(
    0.02, 0.96,
    f"'Good broadband' threshold (p75): {GOOD_SPEED:.1f} Mbps",
    transform=ax2.transAxes,
    va="top", ha="left", fontsize=10,
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.75", alpha=0.85)
)

ax2.grid(True, alpha=0.25)

xticklabels = [CLUSTER_NAMES.get(int(k), f"C{k}") for k in clusters]
ax2.set_xticks(pos)
ax2.set_xticklabels(xticklabels)

savefig("fig_box_jitter_dpi_and_speed_by_cluster.png")



# -------------------------
# 图5：Cluster profile 热力图（z均值）——看“哪些变量最区分这些类型”
# -------------------------
z_only = zprof.set_index("cluster")
ranges = (z_only.max(axis=0) - z_only.min(axis=0)).sort_values(ascending=False)
top_vars = list(ranges.head(12).index)  # 取最能区分的 12 个变量

mat = z_only.loc[sorted(z_only.index), top_vars].values
plt.figure(figsize=(10, 3.8))
plt.imshow(mat, aspect="auto")
plt.yticks(range(len(z_only.index)), [str(i) for i in sorted(z_only.index)])
plt.xticks(range(len(top_vars)), [v.replace("__z","") for v in top_vars], rotation=60, ha="right")
plt.colorbar(label="Mean z-score")
plt.xlabel("Most distinguishing features (by range across clusters)")
plt.ylabel("Cluster")
plt.title("Cluster profiles (z-mean) – key separating features")
savefig("fig_cluster_profile_heatmap_topvars.png")

# -------------------------
# 表：用于写作的“每类均值汇总”（DPI/速度/IMD/rDERI）
# -------------------------
cols = [c for c in ["dpi_score","bb_q_dl_core","imd_score","rDERI_score_0_10"] if c in dfp.columns]
summary = dfp.groupby("cluster")[cols].mean(numeric_only=True).round(4).reset_index()
summary.to_csv(OUT_DIR / "table_cluster_summary_keymeans.csv", index=False)
print("Saved:", OUT_DIR / "table_cluster_summary_keymeans.csv")
display(summary)


Saved: QM data/output/4_3_clustering/figs/fig_cluster_sizes.png
Saved: QM data/output/4_3_clustering/figs/fig_pc1_pc2_clusters0.png
Saved: QM data/output/4_3_clustering/figs/fig_pc1_pc2_clusters.png
Saved: QM data/output/4_3_clustering/figs/fig_dpi_by_cluster_boxplot.png


/tmp/ipykernel_89378/2731699973.py:242: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=[str(k) for k in sorted(dfp["cluster"].dropna().unique())], showfliers=False)


Saved: QM data/output/4_3_clustering/figs/fig_scatter_dpi_vs_dl_by_cluster0.png
Saved: QM data/output/4_3_clustering/figs/fig_box_jitter_dpi_and_speed_by_cluster.png
Saved: QM data/output/4_3_clustering/figs/fig_cluster_profile_heatmap_topvars.png
Saved: QM data/output/4_3_clustering/table_cluster_summary_keymeans.csv


,cluster,dpi_score,bb_q_dl_core,imd_score,rDERI_score_0_10
0,1,0.9606,129.2414,25.7942,4.1106
1,2,0.9521,142.2317,11.3103,3.3323
2,3,0.9618,119.0106,24.4563,3.8367
3,4,0.9457,117.5059,35.2088,5.0707
4,5,0.9689,137.1807,11.2115,2.5217
5,6,0.9631,76.7010,23.8012,4.1244


In [ ]:
# # =========================
# # 4.4 ICS (Integrated Composite Score) — Final (Scheme A, kernel-safe)
# # - 先生成 ICS 表：ics_table_lsoa.csv
# # - 再生成带 geometry 的地图数据：ics_map.gpkg（不在同一块里画图，避免 kernel 挂）
# # - 如需出图：运行本块末尾提供的 “(Optional) Map rendering cell” 单独绘图
# # =========================

# from pathlib import Path
# import numpy as np
# import pandas as pd

# # 可选：地图数据文件（读边界+merge）需要 geopandas
# try:
#     import geopandas as gpd
#     HAS_GPD = True
# except Exception:
#     HAS_GPD = False

# # -------------------------
# # 0) 路径配置
# # -------------------------
# RAW_CANDIDATES = [
#     Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv"),
#     Path("/mnt/data/analysis_table_raw_plus_all_plus_rderi.csv"),
# ]
# RAW_FP = next((p for p in RAW_CANDIDATES if p.exists()), None)
# if RAW_FP is None:
#     raise FileNotFoundError(f"找不到主表：{RAW_CANDIDATES}")

# OUT_DIR = Path("QM data/output/4_4_ics")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# # 你已确认使用这个边界文件
# BOUNDARY_FP = Path("QM data/12/lsoa_boundary_clean.gpkg")
# BOUNDARY_LAYER = "lsoa"  # 推荐显式写上更稳（gpkg 可能有多个 layer）

# # -------------------------
# # 1) 读取主表
# # -------------------------
# df = pd.read_csv(RAW_FP, dtype={"lsoa_code": str})
# df["lsoa_code"] = df["lsoa_code"].astype(str).str.strip().str.upper()

# print("Loaded:", RAW_FP, "| shape:", df.shape)

# # -------------------------
# # 2) 工具函数：winsorize + zscore（方向可控）
# # -------------------------
# def winsorize_series(s: pd.Series, q_low=0.01, q_high=0.99) -> pd.Series:
#     s = pd.to_numeric(s, errors="coerce")
#     lo = s.quantile(q_low)
#     hi = s.quantile(q_high)
#     return s.clip(lower=lo, upper=hi)

# def zscore(s: pd.Series) -> pd.Series:
#     s = pd.to_numeric(s, errors="coerce")
#     mu = s.mean(skipna=True)
#     sd = s.std(skipna=True, ddof=0)
#     if sd == 0 or np.isnan(sd):
#         return pd.Series(np.zeros(len(s)), index=s.index)
#     return (s - mu) / sd

# def make_risk_z(s: pd.Series, higher_is_risk: bool, do_winsor=True) -> pd.Series:
#     """
#     统一成：值越大 = 风险越高；返回 z 分数
#     higher_is_risk=True  ->  z
#     higher_is_risk=False -> -z
#     """
#     if do_winsor:
#         s = winsorize_series(s)
#     z = zscore(s)
#     return z if higher_is_risk else (-z)

# # -------------------------
# # 3) 变量篮子（自动检测可用列）
# # -------------------------

# # 3.1 Participation（低参与=高风险）
# participation_candidates = [
#     ("dpi_score", False),  # DPI 越高越好 => 取反作为风险
# ]

# # 3.2 Vulnerability / Pressure（越高越风险）
# vulnerability_candidates = [
#     ("imd_score", True),
#     ("imd_income_score", True),
#     ("imd_employ_score", True),
#     ("imd_edu_score", True),
#     ("imd_health_score", True),
#     ("imd_crime_score", True),
#     ("imd_livenv_score", True),

#     ("crime_rate_per_1000_window", True),
#     ("crime_count_window", True),
#     ("fuel_poverty_pct", True),
#     ("pct_no_qual", True),
#     ("pct_low_skill", True),
#     ("risk_edu_low", True),
#     ("pct_disabled", True),
#     ("pct_limited_a_lot", True),
#     ("risk_disability_limited", True),
# ]

# # 3.3 Infrastructure constraint（越差越风险；网速/质量越高越好，所以 False）
# infra_candidates = [
#     ("bb_q_dl_core", False),
#     ("bb_q_ul_core", False),
#     ("bb_q_sfu_core", False),
# ]

# def pick_available(df: pd.DataFrame, candidates):
#     out = []
#     for col, higher_is_risk in candidates:
#         if col in df.columns:
#             out.append((col, higher_is_risk))
#     return out

# P_cols = pick_available(df, participation_candidates)
# V_cols = pick_available(df, vulnerability_candidates)
# I_cols = pick_available(df, infra_candidates)

# print("Participation cols:", [c for c,_ in P_cols])
# print("Vulnerability cols:", [c for c,_ in V_cols])
# print("Infra cols:", [c for c,_ in I_cols])

# # -------------------------
# # 4) 计算三个子分数 + ICS 合成
# # -------------------------
# # 权重（强调“被忽略高风险”的软约束：P 与 V 更大，I 更小）
# wP, wV, wI = 0.45, 0.45, 0.10  # 你可在论文中解释为“透明且主题导向的加权”

# work = pd.DataFrame({"lsoa_code": df["lsoa_code"]})

# # Participation risk
# P_z_list = []
# for col, higher_is_risk in P_cols:
#     work[f"{col}__riskz"] = make_risk_z(df[col], higher_is_risk=higher_is_risk)
#     P_z_list.append(f"{col}__riskz")
# work["risk_participation"] = work[P_z_list].mean(axis=1) if P_z_list else 0.0

# # Vulnerability risk
# V_z_list = []
# for col, higher_is_risk in V_cols:
#     work[f"{col}__riskz"] = make_risk_z(df[col], higher_is_risk=higher_is_risk)
#     V_z_list.append(f"{col}__riskz")
# work["risk_vulnerability"] = work[V_z_list].mean(axis=1) if V_z_list else 0.0

# # Infra risk (optional)
# I_z_list = []
# for col, higher_is_risk in I_cols:
#     work[f"{col}__riskz"] = make_risk_z(df[col], higher_is_risk=higher_is_risk)
#     I_z_list.append(f"{col}__riskz")
# work["risk_infra"] = work[I_z_list].mean(axis=1) if I_z_list else 0.0


# # -------------------------
# # 4.4) ICS 合成（阈值用真实有数据的 LSOA；缺失仅为绘图填补）
# # -------------------------

# # 1) 先算一个未填补版本（会有 NaN）
# ics_raw = (wP * work["risk_participation"] +
#            wV * work["risk_vulnerability"] +
#            wI * work["risk_infra"])

# # 2) 阈值只在非缺失上算（避免 fillna(0) 影响 Top10/分位数切分）
# q90 = ics_raw.dropna().quantile(0.90)
# q75 = ics_raw.dropna().quantile(0.75)
# q50 = ics_raw.dropna().quantile(0.50)

# # 3) 再把缺失填补为 0（只为不留白；0 代表“平均风险水平”）
# work["ics_score_z"] = ics_raw.fillna(0.0)

# # Map to 0–100 for interpretability（用填补后的值做 min/max，地图不会 NaN）
# mn, mx = work["ics_score_z"].min(), work["ics_score_z"].max()
# work["ics_score_0_100"] = (work["ics_score_z"] - mn) / (mx - mn) * 100

# # Rank (1 = highest risk)
# work["ics_rank"] = work["ics_score_z"].rank(ascending=False, method="min").astype(int)

# # Bands for policy-friendly map（用上面“未缺失分布”算出来的阈值）
# def band(x):
#     if x >= q90: return "High (Top 10%)"
#     if x >= q75: return "Upper (10–25%)"
#     if x >= q50: return "Mid (25–50%)"
#     return "Lower (50–100%)"

# work["ics_band"] = work["ics_score_z"].apply(band)
# work["ics_high_risk_flag"] = (work["ics_score_z"] >= q90).astype(int)


# # # ICS (higher = higher risk)
# # work["ics_score_z"] = (wP * work["risk_participation"] +
# #                        wV * work["risk_vulnerability"] +
# #                        wI * work["risk_infra"])

# # # Map to 0–100 for interpretability
# # mn, mx = work["ics_score_z"].min(), work["ics_score_z"].max()
# # work["ics_score_0_100"] = (work["ics_score_z"] - mn) / (mx - mn) * 100

# # # Rank (1 = highest risk)
# # work["ics_rank"] = work["ics_score_z"].rank(ascending=False, method="min").astype(int)

# # # Bands for policy-friendly map
# # q90 = work["ics_score_z"].quantile(0.90)
# # q75 = work["ics_score_z"].quantile(0.75)
# # q50 = work["ics_score_z"].quantile(0.50)

# # def band(x):
# #     if x >= q90: return "High (Top 10%)"
# #     if x >= q75: return "Upper (10–25%)"
# #     if x >= q50: return "Mid (25–50%)"
# #     return "Lower (50–100%)"

# # work["ics_band"] = work["ics_score_z"].apply(band)
# # work["ics_high_risk_flag"] = (work["ics_score_z"] >= q90).astype(int)

# # Proxy overlooked flag (正式版本在 4.5 用 rDERI 四象限)
# if "rDERI_score_0_10" in df.columns:
#     med_rderi = pd.to_numeric(df["rDERI_score_0_10"], errors="coerce").median()
#     work["overlooked_proxy_flag"] = ((work["ics_high_risk_flag"] == 1) &
#                                      (pd.to_numeric(df["rDERI_score_0_10"], errors="coerce") < med_rderi)).astype(int)
# else:
#     work["overlooked_proxy_flag"] = 0

# # -------------------------
# # 5) 输出 ICS 表
# # -------------------------
# OUT_TABLE = OUT_DIR / "ics_table_lsoa.csv"
# keep_cols = [
#     "lsoa_code",
#     "ics_score_z", "ics_score_0_100",
#     "ics_rank", "ics_band", "ics_high_risk_flag",
#     "risk_participation", "risk_vulnerability", "risk_infra",
#     "overlooked_proxy_flag",
# ]
# work[keep_cols].to_csv(OUT_TABLE, index=False)
# print("Saved:", OUT_TABLE)


# # 6) Scheme A: 只生成“带 geometry 的地图数据文件”，不在此处画图（防 kernel 挂）
# if HAS_GPD:
#     if not BOUNDARY_FP.exists():
#         print(f"[WARN] 边界文件不存在，跳过地图数据输出：{BOUNDARY_FP}")
#     else:
#         # 只读取必要字段，降低内存压力
#         gdf = gpd.read_file(BOUNDARY_FP, layer=BOUNDARY_LAYER)[["lsoa_code", "geometry"]].copy()
#         gdf["lsoa_code"] = gdf["lsoa_code"].astype(str).str.strip().str.upper()

#         mg = gdf.merge(work[keep_cols], on="lsoa_code", how="left")

#         # =========================
#         # 空间邻居均值插补（把 merge 不上的 181 个也“上色”）
#         # =========================
#         try:
#             from libpysal.weights import Queen
#             import numpy as np

#             wq = Queen.from_dataframe(mg)   # Queen 邻接
#             wq.transform = "R"

#             vals = mg["ics_score_z"].to_numpy(dtype=float)

#             # 迭代两轮：先用有值的邻居填，再让“刚填的值”继续传播一点
#             for _ in range(2):
#                 miss_idx = np.where(np.isnan(vals))[0]
#                 for i in miss_idx:
#                     neigh = wq.neighbors.get(i, [])
#                     neigh_vals = [vals[j] for j in neigh if not np.isnan(vals[j])]
#                     if len(neigh_vals):
#                         vals[i] = float(np.mean(neigh_vals))

#             # 仍填不上的（孤立/无邻居可用）最后用全局均值兜底
#             global_mean = np.nanmean(vals)
#             vals = np.where(np.isnan(vals), global_mean, vals)

#             mg["ics_score_z"] = vals

#         except Exception as e:
#             print("[WARN] libpysal 不可用或 Queen 构建失败，退回均值填补：", e)
#             mean_ics = mg["ics_score_z"].mean(skipna=True)
#             mg["ics_score_z"] = mg["ics_score_z"].fillna(mean_ics)

#         # 用同一套阈值重新分箱（q50/q75/q90 要在前面已经算好）
#         def band2(x):
#             if x >= q90: return "High (Top 10%)"
#             if x >= q75: return "Upper (10–25%)"
#             if x >= q50: return "Mid (25–50%)"
#             return "Lower (50–100%)"

#         mg["ics_band"] = mg["ics_score_z"].apply(band2)
#         mg["ics_high_risk_flag"] = (mg["ics_score_z"] >= q90).astype(int)

#         mn, mx = mg["ics_score_z"].min(), mg["ics_score_z"].max()
#         mg["ics_score_0_100"] = (mg["ics_score_z"] - mn) / (mx - mn) * 100

#         # 现在 missing 应该是 0
#         missing = int(mg["ics_score_z"].isna().sum())

#         OUT_GPKG = OUT_DIR / "ics_map.gpkg"
#         mg.to_file(OUT_GPKG, layer="ics_map", driver="GPKG")

#         print("Saved:", OUT_GPKG, "| rows:", len(mg), "| missing ICS:", missing)
#         print("Next step: 单独运行下方 Optional Map rendering cell 来出图（更稳）")
# else:
#     print("[INFO] Geopandas 不可用，已跳过地图数据输出。你仍可使用 ics_table_lsoa.csv 继续后续分析。")

Loaded: QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv | shape: (4813, 39)
Participation cols: ['dpi_score']
Vulnerability cols: ['imd_score', 'imd_income_score', 'imd_employ_score', 'imd_edu_score', 'imd_health_score', 'imd_crime_score', 'imd_livenv_score', 'crime_rate_per_1000_window', 'crime_count_window', 'fuel_poverty_pct', 'pct_no_qual', 'pct_low_skill', 'risk_edu_low', 'pct_disabled', 'pct_limited_a_lot', 'risk_disability_limited']
Infra cols: ['bb_q_dl_core', 'bb_q_ul_core', 'bb_q_sfu_core']
Saved: QM data/output/4_4_ics/ics_table_lsoa.csv


/tmp/ipykernel_89704/590435563.py:257: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  wq = Queen.from_dataframe(mg)   # Queen 邻接


In [ ]:
# =========================
# 4.4 ICS (Integrated Composite Score) — Final (clean, reproducible)
# Outputs:
#  - QM data/output/4_4_ics/ics_table_lsoa.csv
#  - QM data/output/4_4_ics/ics_map.gpkg  (layer: ics_map)
# Notes:
#  - Quantile thresholds q50/q75/q90 computed on non-missing ics_raw
#  - For boundary-only LSOAs (merge-missing), impute ics_score_z by Queen-neighbour mean (visual + completeness)
# =========================

from pathlib import Path
import numpy as np
import pandas as pd

# geopandas optional (for gpkg)
try:
    import geopandas as gpd
    HAS_GPD = True
except Exception:
    HAS_GPD = False

# -------------------------
# 0) Paths
# -------------------------
RAW_CANDIDATES = [
    Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv"),
    Path("/mnt/data/analysis_table_raw_plus_all_plus_rderi.csv"),
]
RAW_FP = next((p for p in RAW_CANDIDATES if p.exists()), None)
if RAW_FP is None:
    raise FileNotFoundError(f"Cannot find main table: {RAW_CANDIDATES}")

OUT_DIR = Path("QM data/output/4_4_ics")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY_FP = Path("QM data/12/lsoa_boundary_clean.gpkg")
BOUNDARY_LAYER = "lsoa"  # adjust if needed

# -------------------------
# 1) Read main table
# -------------------------
df = pd.read_csv(RAW_FP, dtype={"lsoa_code": str})
df["lsoa_code"] = df["lsoa_code"].astype(str).str.strip().str.upper()

print("Loaded:", RAW_FP, "| shape:", df.shape)

# -------------------------
# 2) Helpers: winsorize + zscore + risk direction
# -------------------------
def winsorize_series(s: pd.Series, q_low=0.01, q_high=0.99) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    lo = s.quantile(q_low)
    hi = s.quantile(q_high)
    return s.clip(lower=lo, upper=hi)

def zscore(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    mu = s.mean(skipna=True)
    sd = s.std(skipna=True, ddof=0)
    if sd == 0 or np.isnan(sd):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd

def make_risk_z(s: pd.Series, higher_is_risk: bool, do_winsor=True) -> pd.Series:
    """
    Return z-score with unified direction: higher value => higher risk
    higher_is_risk=True  ->  z
    higher_is_risk=False -> -z
    """
    if do_winsor:
        s = winsorize_series(s)
    z = zscore(s)
    return z if higher_is_risk else (-z)

# -------------------------
# 3) Variable baskets (auto-pick existing cols)
# -------------------------
participation_candidates = [
    ("dpi_score", False),  # higher DPI is better => risk uses -z
]

vulnerability_candidates = [
    ("imd_score", True),
    ("imd_income_score", True),
    ("imd_employ_score", True),
    ("imd_edu_score", True),
    ("imd_health_score", True),
    ("imd_crime_score", True),
    ("imd_livenv_score", True),

    ("crime_rate_per_1000_window", True),
    ("crime_count_window", True),
    ("fuel_poverty_pct", True),
    ("pct_no_qual", True),
    ("pct_low_skill", True),
    ("risk_edu_low", True),
    ("pct_disabled", True),
    ("pct_limited_a_lot", True),
    ("risk_disability_limited", True),
]

infra_candidates = [
    ("bb_q_dl_core", False),   # higher speed/quality => lower risk => -z
    ("bb_q_ul_core", False),
    ("bb_q_sfu_core", False),
]

def pick_available(_df: pd.DataFrame, candidates):
    out = []
    for col, higher_is_risk in candidates:
        if col in _df.columns:
            out.append((col, higher_is_risk))
    return out

P_cols = pick_available(df, participation_candidates)
V_cols = pick_available(df, vulnerability_candidates)
I_cols = pick_available(df, infra_candidates)

print("Participation cols:", [c for c,_ in P_cols])
print("Vulnerability cols:", [c for c,_ in V_cols])
print("Infra cols:", [c for c,_ in I_cols])

# -------------------------
# 4) Compute sub-scores + ICS
# -------------------------
wP, wV, wI = 0.45, 0.45, 0.10

work = pd.DataFrame({"lsoa_code": df["lsoa_code"]})

# Participation risk
P_z_list = []
for col, higher_is_risk in P_cols:
    work[f"{col}__riskz"] = make_risk_z(df[col], higher_is_risk=higher_is_risk)
    P_z_list.append(f"{col}__riskz")
work["risk_participation"] = work[P_z_list].mean(axis=1) if P_z_list else 0.0

# Vulnerability risk
V_z_list = []
for col, higher_is_risk in V_cols:
    work[f"{col}__riskz"] = make_risk_z(df[col], higher_is_risk=higher_is_risk)
    V_z_list.append(f"{col}__riskz")
work["risk_vulnerability"] = work[V_z_list].mean(axis=1) if V_z_list else 0.0

# Infra risk
I_z_list = []
for col, higher_is_risk in I_cols:
    work[f"{col}__riskz"] = make_risk_z(df[col], higher_is_risk=higher_is_risk)
    I_z_list.append(f"{col}__riskz")
work["risk_infra"] = work[I_z_list].mean(axis=1) if I_z_list else 0.0

# 4.4) ICS (thresholds computed on non-missing only)
ics_raw = (wP * work["risk_participation"] +
           wV * work["risk_vulnerability"] +
           wI * work["risk_infra"])

q90 = ics_raw.dropna().quantile(0.90)
q75 = ics_raw.dropna().quantile(0.75)
q50 = ics_raw.dropna().quantile(0.50)

# Fill missing inside-table rows with 0 for stability (not the merge-missing ones)
work["ics_score_z"] = ics_raw.fillna(0.0)

# Policy bands — IMPORTANT: use plain "-" (NOT "–") to avoid mapping issues
def band_label(x):
    if x >= q90: return "High (Top 10%)"
    if x >= q75: return "Upper (10-25%)"
    if x >= q50: return "Mid (25-50%)"
    return "Lower (50-100%)"

work["ics_band"] = work["ics_score_z"].apply(band_label)
work["ics_high_risk_flag"] = (work["ics_score_z"] >= q90).astype(int)

# 0–100 scale (for interpretability)
mn, mx = work["ics_score_z"].min(), work["ics_score_z"].max()
work["ics_score_0_100"] = (work["ics_score_z"] - mn) / (mx - mn) * 100

# Rank (1 = highest risk)
work["ics_rank"] = work["ics_score_z"].rank(ascending=False, method="min").astype(int)

# Proxy overlooked flag (optional)
if "rDERI_score_0_10" in df.columns:
    med_rderi = pd.to_numeric(df["rDERI_score_0_10"], errors="coerce").median()
    work["overlooked_proxy_flag"] = (
        (work["ics_high_risk_flag"] == 1) &
        (pd.to_numeric(df["rDERI_score_0_10"], errors="coerce") < med_rderi)
    ).astype(int)
else:
    work["overlooked_proxy_flag"] = 0

# -------------------------
# 5) Save ICS table
# -------------------------
keep_cols = [
    "lsoa_code",
    "ics_score_z", "ics_score_0_100",
    "ics_rank", "ics_band", "ics_high_risk_flag",
    "risk_participation", "risk_vulnerability", "risk_infra",
    "overlooked_proxy_flag",
]
OUT_TABLE = OUT_DIR / "ics_table_lsoa.csv"
work[keep_cols].to_csv(OUT_TABLE, index=False)
print("Saved:", OUT_TABLE)

# -------------------------
# 6) Save map gpkg (with geometry) + impute merge-missing using Queen neighbours
# -------------------------
if HAS_GPD:
    if not BOUNDARY_FP.exists():
        print(f"[WARN] Boundary missing, skip gpkg: {BOUNDARY_FP}")
    else:
        gdf = gpd.read_file(BOUNDARY_FP, layer=BOUNDARY_LAYER)[["lsoa_code", "geometry"]].copy()
        gdf["lsoa_code"] = gdf["lsoa_code"].astype(str).str.strip().str.upper()

        mg = gdf.merge(work[keep_cols], on="lsoa_code", how="left")

        merge_missing = int(mg["ics_score_z"].isna().sum())
        print("Merge-missing ICS (boundary has code, table doesn't):", merge_missing, "out of", len(mg))
        if merge_missing > 0:
            print("Example unmatched codes:", mg.loc[mg["ics_score_z"].isna(), "lsoa_code"].head(10).tolist())

        # --- spatial neighbour mean imputation for merge-missing polygons ---
        # If libpysal not available, fallback to global mean.
        try:
            from libpysal.weights import Queen

            wq = Queen.from_dataframe(mg)
            wq.transform = "R"

            vals = mg["ics_score_z"].to_numpy(dtype=float)

            for _ in range(2):
                miss_idx = np.where(np.isnan(vals))[0]
                for i in miss_idx:
                    neigh = wq.neighbors.get(i, [])
                    neigh_vals = [vals[j] for j in neigh if not np.isnan(vals[j])]
                    if len(neigh_vals):
                        vals[i] = float(np.mean(neigh_vals))

            global_mean = np.nanmean(vals)
            vals = np.where(np.isnan(vals), global_mean, vals)
            mg["ics_score_z"] = vals

        except Exception as e:
            print("[WARN] Queen imputation failed; fallback to global mean. Reason:", repr(e))
            mean_ics = mg["ics_score_z"].mean(skipna=True)
            mg["ics_score_z"] = mg["ics_score_z"].fillna(mean_ics)

        # Recompute band + flags + 0-100 on mg after imputation
        mg["ics_band"] = mg["ics_score_z"].apply(band_label)
        mg["ics_high_risk_flag"] = (mg["ics_score_z"] >= q90).astype(int)

        mn2, mx2 = mg["ics_score_z"].min(), mg["ics_score_z"].max()
        mg["ics_score_0_100"] = (mg["ics_score_z"] - mn2) / (mx2 - mn2) * 100

        missing_after = int(mg["ics_score_z"].isna().sum())
        print("Missing ICS after imputation:", missing_after)

        OUT_GPKG = OUT_DIR / "ics_map.gpkg"
        mg.to_file(OUT_GPKG, layer="ics_map", driver="GPKG")
        print("Saved:", OUT_GPKG, "| rows:", len(mg))
else:
    print("[INFO] geopandas not available; skipped gpkg. Use ics_table_lsoa.csv only.")


Loaded: QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv | shape: (4813, 39)
Participation cols: ['dpi_score']
Vulnerability cols: ['imd_score', 'imd_income_score', 'imd_employ_score', 'imd_edu_score', 'imd_health_score', 'imd_crime_score', 'imd_livenv_score', 'crime_rate_per_1000_window', 'crime_count_window', 'fuel_poverty_pct', 'pct_no_qual', 'pct_low_skill', 'risk_edu_low', 'pct_disabled', 'pct_limited_a_lot', 'risk_disability_limited']
Infra cols: ['bb_q_dl_core', 'bb_q_ul_core', 'bb_q_sfu_core']
Saved: QM data/output/4_4_ics/ics_table_lsoa.csv
Merge-missing ICS (boundary has code, table doesn't): 181 out of 4994
Example unmatched codes: ['E01034472', 'E01034473', 'E01034477', 'E01034469', 'E01034476', 'E01034478', 'E01033917', 'E01033919', 'E01033921', 'E01033922']


/tmp/ipykernel_89751/2069256845.py:226: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  wq = Queen.from_dataframe(mg)


In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import ListedColormap

OUT_DIR = Path("QM data/output/4_4_ics")
g = gpd.read_file(OUT_DIR / "ics_map.gpkg", layer="ics_map").copy()

# (Optional) simplify for speed
try:
    g_m = g.to_crs(27700)
    g_m["geometry"] = g_m["geometry"].simplify(tolerance=30, preserve_topology=True)
    g = g_m
except Exception as e:
    print("CRS/simplify skipped:", repr(e))

# -------------------------
# 1) Continuous ICS score map (0–100)
# -------------------------
fig, ax = plt.subplots(figsize=(8, 9))

g.plot(
    column="ics_score_0_100",
    cmap="magma_r",
    vmin=0, vmax=100,
    linewidth=0,
    ax=ax
)
ax.set_axis_off()
ax.set_title("ICS risk score (0–100): higher = higher risk")

# Smaller colorbar
norm = mpl.colors.Normalize(vmin=0, vmax=100)
sm = mpl.cm.ScalarMappable(norm=norm, cmap="magma_r")
sm._A = []
cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02, shrink=0.75)
cbar.ax.tick_params(labelsize=9)

fig1 = OUT_DIR / "fig_ics_score_map.png"
fig.savefig(fig1, dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved:", fig1)

# -------------------------
# 2) ICS band map (policy-oriented)
#    IMPORTANT: we standardised labels to use "-" in Cell A
# -------------------------
g2 = g.copy()

# Hard-normalise any stray long dashes just in case
def norm_band(s):
    return (s.astype(str)
            .str.replace("\u2013", "-", regex=False)
            .str.replace("\u2014", "-", regex=False)
            .str.replace("\xa0", " ", regex=False)
            .str.strip())

g2["ics_band"] = norm_band(g2["ics_band"])

order = ["High (Top 10%)", "Upper (10-25%)", "Mid (25-50%)", "Lower (50-100%)"]

# Diagnose unexpected labels (this is where white holes come from)
bad = g2.loc[~g2["ics_band"].isin(order), "ics_band"].value_counts()
if len(bad) > 0:
    print("[WARN] Unexpected band labels detected, coercing to Mid:", bad.head(20).to_dict())
    g2.loc[~g2["ics_band"].isin(order), "ics_band"] = "Mid (25-50%)"

g2["ics_band"] = pd.Categorical(g2["ics_band"], categories=order, ordered=True)

# Your custom palette: High purple, Upper red, Mid yellow, Lower very light yellow
base = plt.get_cmap("magma_r")
c_high  = base(0.68)
c_upper = base(0.42)
c_mid   = base(0.18)
c_lower = base(0.06)
cmap_band = ListedColormap([c_high, c_upper, c_mid, c_lower])

ax = g2.plot(
    column="ics_band",
    categorical=True,
    cmap=cmap_band,
    legend=True,
    linewidth=0,
    figsize=(8, 9),
    legend_kwds={"loc": "upper right", "frameon": True}
)
ax.set_axis_off()
ax.set_title("ICS risk band (policy-oriented)")

# Make legend patches borderless
leg = ax.get_legend()
if leg is not None:
    for h in leg.legend_handles:
        try:
            h.set_edgecolor("none")
        except Exception:
            pass

fig2 = OUT_DIR / "fig_ics_band_map_reversed.png"
ax.figure.savefig(fig2, dpi=200, bbox_inches="tight")
plt.close(ax.figure)
print("Saved:", fig2)


In [3]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import ListedColormap

OUT_DIR = Path("QM data/output/4_4_ics")
g = gpd.read_file(OUT_DIR / "ics_map.gpkg", layer="ics_map").copy()

# (Optional) simplify for speed
try:
    g_m = g.to_crs(27700)
    g_m["geometry"] = g_m["geometry"].simplify(tolerance=30, preserve_topology=True)
    g = g_m
except Exception as e:
    print("CRS/simplify skipped:", repr(e))

# -------------------------
# 1) Continuous ICS score map (0–100)
# -------------------------
fig, ax = plt.subplots(figsize=(8, 9))

g.plot(
    column="ics_score_0_100",
    cmap="magma_r",
    vmin=0, vmax=100,
    linewidth=0,
    ax=ax
)
ax.set_axis_off()
ax.set_title("ICS risk score (0–100): higher = higher risk")

# Smaller colorbar
norm = mpl.colors.Normalize(vmin=0, vmax=100)
sm = mpl.cm.ScalarMappable(norm=norm, cmap="magma_r")
sm._A = []
cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02, shrink=0.75)
cbar.ax.tick_params(labelsize=9)

fig1 = OUT_DIR / "fig_ics_score_map.png"
fig.savefig(fig1, dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved:", fig1)

# -------------------------
# 2) ICS band map (policy-oriented)
#    IMPORTANT: we standardised labels to use "-" in Cell A
# -------------------------
g2 = g.copy()

# Hard-normalise any stray long dashes just in case
def norm_band(s):
    return (s.astype(str)
            .str.replace("\u2013", "-", regex=False)
            .str.replace("\u2014", "-", regex=False)
            .str.replace("\xa0", " ", regex=False)
            .str.strip())

g2["ics_band"] = norm_band(g2["ics_band"])

order = ["High (Top 10%)", "Upper (10-25%)", "Mid (25-50%)", "Lower (50-100%)"]

# Diagnose unexpected labels (this is where white holes come from)
bad = g2.loc[~g2["ics_band"].isin(order), "ics_band"].value_counts()
if len(bad) > 0:
    print("[WARN] Unexpected band labels detected, coercing to Mid:", bad.head(20).to_dict())
    g2.loc[~g2["ics_band"].isin(order), "ics_band"] = "Mid (25-50%)"

g2["ics_band"] = pd.Categorical(g2["ics_band"], categories=order, ordered=True)

# Your custom palette: High purple, Upper red, Mid yellow, Lower very light yellow
base = plt.get_cmap("magma_r")
c_high  = base(0.68)
c_upper = base(0.42)
c_mid   = base(0.18)
c_lower = base(0.06)
cmap_band = ListedColormap([c_high, c_upper, c_mid, c_lower])

ax = g2.plot(
    column="ics_band",
    categorical=True,
    cmap=cmap_band,
    legend=True,
    linewidth=0,
    figsize=(8, 9),
    legend_kwds={"loc": "upper right", "frameon": True}
)
ax.set_axis_off()
ax.set_title("ICS risk band (policy-oriented)")

# Make legend patches borderless
leg = ax.get_legend()
if leg is not None:
    for h in leg.legend_handles:
        try:
            h.set_edgecolor("none")
        except Exception:
            pass

fig2 = OUT_DIR / "fig_ics_band_map_reversed.png"
ax.figure.savefig(fig2, dpi=200, bbox_inches="tight")
plt.close(ax.figure)
print("Saved:", fig2)

Saved: QM data/output/4_4_ics/fig_ics_score_map.png
[WARN] Unexpected band labels detected, coercing to Mid: {'Missing': 181}
Saved: QM data/output/4_4_ics/fig_ics_band_map_reversed.png


In [1]:
# def norm_band(s):
#     return (s.astype(str)
#             .str.replace("\u2013", "-", regex=False)  # en dash
#             .str.replace("\u2014", "-", regex=False)  # em dash
#             .str.replace("\xa0", " ", regex=False)    # 不换行空格
#             .str.strip())

# g["_band"] = norm_band(g["ics_band"])

# # =========================
# # (Optional) Map rendering cell (run separately to avoid kernel crash)
# # =========================
# import geopandas as gpd
# import matplotlib.pyplot as plt

# OUT_DIR = Path("QM data/output/4_4_ics")
# g = gpd.read_file(OUT_DIR / "ics_map.gpkg", layer="ics_map")

# # 简化几何降低绘图负担（tolerance可调 10~50，越小越清晰，太小可能挂）
# try:
#     g_m = g.to_crs(27700)
#     g_m["geometry"] = g_m["geometry"].simplify(tolerance=30, preserve_topology=True)
#     g = g_m
# except Exception as e:
#     print("CRS/simplify skipped:", repr(e))

# # -------------------------
# # 1) Continuous ICS score map (0–100) with reversed colormap
# #    viridis_r makes low=yellow, high=purple  ✅
# # -------------------------
# # ax = g.plot(
# #     column="ics_score_0_100",
# #     cmap="magma_r",
# #     vmin=0, vmax=100,
# #     legend=True, linewidth=0, figsize=(8, 9)
# # )

# # ax.set_axis_off()
# # ax.set_title("ICS risk score (0–100): higher = higher risk")
# # fig1 = OUT_DIR / "fig_ics_score_map_reversed.png"
# # ax.figure.savefig(fig1, dpi=200, bbox_inches="tight")
# # plt.close(ax.figure)
# # print("Saved:", fig1)

# import matplotlib.pyplot as plt
# import matplotlib as mpl

# fig, ax = plt.subplots(figsize=(8, 9))

# # 1) 先画地图（先不要 legend）
# g.plot(
#     column="ics_score_0_100",
#     cmap="magma_r",
#     vmin=0, vmax=100,
#     linewidth=0,
#     ax=ax
# )
# ax.set_axis_off()
# ax.set_title("ICS risk score (0–100): higher = higher risk")

# # 2) 手动创建更小的色标（控制宽度/高度/间距）
# norm = mpl.colors.Normalize(vmin=0, vmax=100)
# sm = mpl.cm.ScalarMappable(norm=norm, cmap="magma_r")
# sm._A = []  # 兼容旧写法，避免 warning

# cbar = fig.colorbar(
#     sm,
#     ax=ax,
#     fraction=0.035,   # ✅ 色标宽度（越小越窄） 0.02~0.05 之间调
#     pad=0.02,         # ✅ 和地图的间距
#     shrink=0.75       # ✅ 色标高度（越小越短） 0.6~0.9 之间调
# )

# # 可选：把刻度字体也缩小一点
# cbar.ax.tick_params(labelsize=9)

# fig1 = OUT_DIR / "fig_ics_score_map.png"
# fig.savefig(fig1, dpi=200, bbox_inches="tight")
# plt.close(fig)
# print("Saved:", fig1)

# # -------------------------
# # 2) ICS band map (policy-oriented) — also use reversed palette
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from matplotlib.colors import ListedColormap

# # =========================
# # ICS band map (pretty + ordered legend)
# # High top, Lower bottom
# # High uses old Upper purple; Upper uses old Mid red; Mid uses old Lower yellow; Lower uses very light yellow
# # =========================

# g2 = g.copy()

# # 1) 统一标签（兼容不同破折号写法）
# rep = {
#     "Lower (50-100%)": "Lower (50–100%)",
#     "Mid (25-50%)": "Mid (25–50%)",
#     "Upper (10-25%)": "Upper (10–25%)",
#     "High (Top10%)": "High (Top 10%)",
#     "High (Top10%)": "High (Top 10%)",
#     "High (Top 10%)": "High (Top 10%)",
# }
# if g2["ics_band"].dtype == "object":
#     g2["ics_band"] = g2["ics_band"].replace(rep)

# # 2) 图例顺序：High -> Upper -> Mid -> Lower
# order = ["High (Top 10%)", "Upper (10–25%)", "Mid (25–50%)", "Lower (50–100%)"]
# if set(order).issubset(set(g2["ics_band"].dropna().unique())):
#     g2["ics_band"] = pd.Categorical(g2["ics_band"], categories=order, ordered=True)

# # 3) 自定义离散配色（来自 magma_r，但避开最黑/最亮）
# # 取 4 个点：Lower(很淡黄) / Mid(淡黄) / Upper(红) / High(紫)
# base = plt.get_cmap("magma_r")
# c_lower = base(0.06)  # 非常淡黄（你的“淡淡淡黄”）
# c_mid   = base(0.18)  # 原来 Lower 那档的黄
# c_upper = base(0.42)  # 原来 Mid 那档的红
# c_high  = base(0.68)  # 原来 Upper 那档的紫（避免接近黑）

# # 注意：ListedColormap 的顺序要和 categories 顺序一致（High, Upper, Mid, Lower）
# cmap_band = ListedColormap([c_high, c_upper, c_mid, c_lower])

# ax = g2.plot(
#     column="ics_band",
#     categorical=True,
#     cmap=cmap_band,
#     legend=True,
#     linewidth=0,
#     figsize=(8, 9),
#     legend_kwds={"loc": "upper right", "frameon": True}
# )
# ax.set_axis_off()
# ax.set_title("ICS risk band (policy-oriented)")

# # （可选）让图例色块更干净：去掉边框
# leg = ax.get_legend()
# if leg is not None:
#     for h in leg.legend_handles:
#         try:
#             h.set_edgecolor("none")
#         except Exception:
#             pass

# fig2 = OUT_DIR / "fig_ics_band_map_reversed.png"  # 覆盖旧图
# ax.figure.savefig(fig2, dpi=200, bbox_inches="tight")
# plt.close(ax.figure)
# print("Saved:", fig2)


NameError: name 'g' is not defined

In [10]:
# =========================
# 4.4.1 ICS maps — blue-toned (using your provided colourbar palette)
# FIXED:
# - No MultiIndex error (colors are HEX strings)
# - No Matplotlib get_cmap deprecation warning
# =========================
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
from matplotlib.colors import to_rgb, to_hex

OUT_DIR = Path("QM data/output/4_4_ics")
g = gpd.read_file(OUT_DIR / "ics_map.gpkg", layer="ics_map")

# --- simplify (optional) ---
try:
    g_m = g.to_crs(27700)
    g_m["geometry"] = g_m.geometry.simplify(tolerance=30, preserve_topology=True)
    g = g_m
except Exception as e:
    print("CRS/simplify skipped:", repr(e))

# ---------- helpers ----------
def pick_col(cols, candidates):
    cols_low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_low:
            return cols_low[cand.lower()]
    for cand in candidates:
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

def normalize_band(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = str(v).strip().lower()
    if "top" in s or "10%" in s or "high" in s:
        return "High (Top 10%)"
    if ("10" in s and "25" in s) or ("upper" in s):
        return "Upper (10–25%)"
    if ("25" in s and "50" in s) or ("mid" in s):
        return "Mid (25–50%)"
    if ("50" in s) or ("lower" in s):
        return "Lower (50–100%)"
    return str(v)

def lighten_hex(color, amount=0.25):
    """Return HEX string, blend with white. amount=0 -> original, 1 -> white."""
    r, g, b = to_rgb(color)
    r = r + (1 - r) * amount
    g = g + (1 - g) * amount
    b = b + (1 - b) * amount
    return to_hex((r, g, b))

def nice_bounds(ax, geom, pad=0.015):
    minx, miny, maxx, maxy = geom.bounds
    dx = (maxx - minx) * pad
    dy = (maxy - miny) * pad
    ax.set_xlim(minx - dx, maxx + dx)
    ax.set_ylim(miny - dy, maxy + dy)

# ---------- outer London outline only ----------
if hasattr(g.geometry, "union_all"):
    london_geom = g.geometry.union_all()
else:
    london_geom = g.geometry.unary_union
london_outline = london_geom.boundary

# ---------- columns ----------
band_col = pick_col(g.columns, ["ics_band", "band"])
score_col = pick_col(g.columns, ["ics_score_0_100", "ics_score", "score"])
if band_col is None:
    raise ValueError(f"Cannot find ics_band column. Available: {list(g.columns)}")

ORDER = ["High (Top 10%)", "Upper (10–25%)", "Mid (25–50%)", "Lower (50–100%)"]
g["_band_str"] = g[band_col].map(normalize_band)
g["_band"] = pd.Categorical(g["_band_str"], categories=ORDER, ordered=True)

# ---------- palette (use your colourbar) ----------
CMAP_NAME = "viridis"  # <- 你那条色带；想更纯蓝就改成 "Blues"
cmap = mpl.colormaps.get_cmap(CMAP_NAME)  # no deprecation warning

# 取 viridis 偏蓝/青的一段，并做“去饱和/提亮”，避免“脏”
sample_positions = {
    "High (Top 10%)": 0.20,
    "Upper (10–25%)": 0.32,
    "Mid (25–50%)":   0.42,
    "Lower (50–100%)":0.55,
}
lighten_amt = {
    "High (Top 10%)": 0.10,
    "Upper (10–25%)": 0.22,
    "Mid (25–50%)":   0.32,
    "Lower (50–100%)":0.45,
}
band_colors = {
    k: lighten_hex(to_hex(cmap(v)), lighten_amt[k])   # ✅ HEX string (no MultiIndex)
    for k, v in sample_positions.items()
}

# style
OUTLINE_COLOR = "0.60"   # 更浅
OUTLINE_LW = 0.8         # 更细

# =========================
# A) ICS band map
# =========================
fig, ax = plt.subplots(figsize=(8.6, 7.6))

fill = g["_band_str"].astype(str).map(band_colors).fillna("#f7f7f7")
g.plot(ax=ax, color=fill, linewidth=0)

top = g[g["_band_str"] == "High (Top 10%)"]
if not top.empty:
    top.plot(
        ax=ax,
        facecolor=band_colors["High (Top 10%)"],
        edgecolor="0.35",
        linewidth=0.6,
        hatch="///",
        zorder=6
    )

gpd.GeoSeries([london_outline], crs=g.crs).plot(
    ax=ax, color=OUTLINE_COLOR, linewidth=OUTLINE_LW, zorder=10
)

handles = [
    mpatches.Patch(facecolor=band_colors["High (Top 10%)"], edgecolor="0.35", hatch="///", label="High (Top 10%)"),
    mpatches.Patch(facecolor=band_colors["Upper (10–25%)"], edgecolor="none", label="Upper (10–25%)"),
    mpatches.Patch(facecolor=band_colors["Mid (25–50%)"], edgecolor="none", label="Mid (25–50%)"),
    mpatches.Patch(facecolor=band_colors["Lower (50–100%)"], edgecolor="none", label="Lower (50–100%)"),
]
ax.legend(handles=handles, title="Quantile bands", loc="lower left", frameon=True)

ax.set_title("ICS risk band (policy-oriented)", fontsize=20, pad=12)
ax.set_axis_off()
nice_bounds(ax, london_geom, pad=0.015)
fig.tight_layout()
fig_band = OUT_DIR / "fig_ics_band_map_blue.png"
fig.savefig(fig_band, dpi=220)
plt.close(fig)
print("Saved:", fig_band)

# =========================
# B) Priority map (Top 10% only)
# =========================
fig, ax = plt.subplots(figsize=(8.6, 7.6))

other_grey = "#eeeeee"
g.plot(ax=ax, color=other_grey, linewidth=0)

n_top = int((g["_band_str"] == "High (Top 10%)").sum())
n_total = int(g["_band_str"].notna().sum())

if not top.empty:
    top.plot(
        ax=ax,
        facecolor=band_colors["High (Top 10%)"],
        edgecolor="0.35",
        linewidth=0.7,
        hatch="///",
        zorder=6
    )

gpd.GeoSeries([london_outline], crs=g.crs).plot(
    ax=ax, color=OUTLINE_COLOR, linewidth=OUTLINE_LW, zorder=10
)

handles = [
    mpatches.Patch(facecolor=band_colors["High (Top 10%)"], edgecolor="0.35", hatch="///",
                   label=f"High risk (Top 10%) — {n_top} LSOAs (of {n_total} with valid ICS)"),
    mpatches.Patch(facecolor=other_grey, edgecolor="none", label="All other areas"),
]
ax.legend(handles=handles, loc="lower left", frameon=True)

ax.set_title("Priority areas: highest-risk 10% of neighbourhoods (ICS)", fontsize=22, pad=12)
ax.set_axis_off()
nice_bounds(ax, london_geom, pad=0.015)
fig.tight_layout()
fig_top = OUT_DIR / "fig_ics_top10_map_blue.png"
fig.savefig(fig_top, dpi=220)
plt.close(fig)
print("Saved:", fig_top)

# =========================
# C) Continuous map (optional)
# =========================
if score_col is not None:
    fig, ax = plt.subplots(figsize=(8.6, 7.6))
    g.plot(
        ax=ax,
        column=score_col,
        cmap=CMAP_NAME,
        linewidth=0,
        legend=True,
        legend_kwds={"shrink": 0.75, "label": "ICS score (0–100)"},
    )
    gpd.GeoSeries([london_outline], crs=g.crs).plot(
        ax=ax, color=OUTLINE_COLOR, linewidth=OUTLINE_LW, zorder=10
    )
    ax.set_title("ICS risk score (0–100) — higher = higher risk", fontsize=20, pad=12)
    ax.set_axis_off()
    nice_bounds(ax, london_geom, pad=0.015)
    fig.tight_layout()
    fig_cont = OUT_DIR / "fig_ics_score_map_blue.png"
    fig.savefig(fig_cont, dpi=220)
    plt.close(fig)
    print("Saved:", fig_cont)
else:
    print("Skip continuous map: cannot find ics_score_0_100 column.")


Saved: QM data/output/4_4_ics/fig_ics_band_map_blue.png
Saved: QM data/output/4_4_ics/fig_ics_top10_map_blue.png
Saved: QM data/output/4_4_ics/fig_ics_score_map_blue.png


In [41]:
# =========================
# 4.5 DERI validation & Overlooked areas (Percentile + Z-score + Quadrants)
# Outputs:
# - QM data/output/4_5_deri/ics_deri_compare_lsoa.csv
# - QM data/output/4_5_deri/metrics_consistency.csv
# - QM data/output/4_5_deri/overlooked_list.csv
# (Optional) scatter plot saved if matplotlib is available
# =========================

from pathlib import Path
import numpy as np
import pandas as pd

# -------------------------
# 0) Paths & config
# -------------------------
ICS_FP = Path("QM data/output/4_4_ics/ics_table_lsoa.csv")
RAW_CANDIDATES = [
    Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv"),
    Path("/mnt/data/analysis_table_raw_plus_all_plus_rderi.csv"),
]
RAW_FP = next((p for p in RAW_CANDIDATES if p.exists()), None)

OUT_DIR = Path("QM data/output/4_5_deri")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not ICS_FP.exists():
    raise FileNotFoundError(f"找不到 ICS 表：{ICS_FP}")
if RAW_FP is None:
    raise FileNotFoundError(f"找不到主表（含 rDERI/DERI）：{RAW_CANDIDATES}")

# 你要对比的 DERI 列名（优先用 rDERI 0-10）
DERI_COL_CANDIDATES = ["rDERI_score_0_10", "DERI_score_0_10", "deri_score_0_10", "DERI", "rDERI"]
# ICS 用哪个列做“核心风险”（推荐 z 分数）
ICS_COL = "ics_score_z"  # 或改成 "ics_score_0_100"

# 百分位阈值：Top10% 作为 High
HIGH_PCT = 0.90  # Top10%
# 你也可以做敏感性：0.80 (Top20%)，后面 metrics 会一起算

# -------------------------
# 1) Load tables
# -------------------------
ics = pd.read_csv(ICS_FP, dtype={"lsoa_code": str})
ics["lsoa_code"] = ics["lsoa_code"].astype(str).str.strip().str.upper()

raw = pd.read_csv(RAW_FP, dtype={"lsoa_code": str})
raw["lsoa_code"] = raw["lsoa_code"].astype(str).str.strip().str.upper()

DERI_COL = next((c for c in DERI_COL_CANDIDATES if c in raw.columns), None)
if DERI_COL is None:
    raise ValueError(f"主表里找不到 DERI/rDERI 列，尝试过：{DERI_COL_CANDIDATES}")

if ICS_COL not in ics.columns:
    raise ValueError(f"ICS 表里找不到列：{ICS_COL}（现有列：{list(ics.columns)[:20]} ...）")

# join (inner to ensure comparable rows)
df = ics.merge(raw[["lsoa_code", DERI_COL]], on="lsoa_code", how="inner")
df[ICS_COL] = pd.to_numeric(df[ICS_COL], errors="coerce")
df[DERI_COL] = pd.to_numeric(df[DERI_COL], errors="coerce")

# drop NA pairs for metrics (but keep rows for outputs)
valid = df[[ICS_COL, DERI_COL]].dropna()
print("Joined rows:", len(df), "| valid pairs:", len(valid), "| DERI col:", DERI_COL)

# -------------------------
# 2) Same-scale step: A) Percentile / ranking
# -------------------------
# percentile rank in [0,1]; higher = higher risk
# method="average" gives stable ties handling
df["ics_pct"]  = df[ICS_COL].rank(pct=True, method="average")
df["deri_pct"] = df[DERI_COL].rank(pct=True, method="average")

# also keep ranks (1=lowest by default); we want higher = riskier, so descending rank:
df["ics_rank_desc"]  = df[ICS_COL].rank(ascending=False, method="min").astype("Int64")
df["deri_rank_desc"] = df[DERI_COL].rank(ascending=False, method="min").astype("Int64")

# -------------------------
# 3) Same-scale step: B) Z-score
# -------------------------
def zscore(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    mu = s.mean(skipna=True)
    sd = s.std(skipna=True, ddof=0)
    if sd == 0 or np.isnan(sd):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd

df["ics_z"]  = zscore(df[ICS_COL])
df["deri_z"] = zscore(df[DERI_COL])

# -------------------------
# 4) Consistency metrics: correlations + overlaps
# -------------------------
def pearson(a, b):
    return pd.Series(a).corr(pd.Series(b), method="pearson")

def spearman(a, b):
    return pd.Series(a).corr(pd.Series(b), method="spearman")

metrics = []

# correlations on percentile scale
metrics.append(("corr_percentile", "pearson", pearson(df["ics_pct"], df["deri_pct"])))
metrics.append(("corr_percentile", "spearman", spearman(df["ics_pct"], df["deri_pct"])))

# correlations on z-score scale
metrics.append(("corr_zscore", "pearson", pearson(df["ics_z"], df["deri_z"])))
metrics.append(("corr_zscore", "spearman", spearman(df["ics_z"], df["deri_z"])))

# top-k overlaps (policy-oriented)
def top_overlap(pct_threshold: float):
    A = df["ics_pct"]  >= pct_threshold
    B = df["deri_pct"] >= pct_threshold
    inter = int((A & B).sum())
    sizeA = int(A.sum())
    sizeB = int(B.sum())
    union = int((A | B).sum())
    # from ICS perspective + Jaccard
    return {
        "threshold_pct": pct_threshold,
        "n_ics_high": sizeA,
        "n_deri_high": sizeB,
        "n_intersection": inter,
        "overlap_rate_inter_over_ics": (inter / sizeA) if sizeA else np.nan,
        "jaccard_inter_over_union": (inter / union) if union else np.nan,
    }

ov10 = top_overlap(0.90)
ov20 = top_overlap(0.80)

metrics_df = pd.DataFrame(metrics, columns=["metric_group", "method", "value"])
overlap_df = pd.DataFrame([ov10, ov20])

# save metrics
METRICS_FP = OUT_DIR / "metrics_consistency.csv"
metrics_out = pd.concat(
    [metrics_df, pd.DataFrame({
        "metric_group": ["top_overlap", "top_overlap"],
        "method": [f"Top {int((1-ov10['threshold_pct'])*100)}%", f"Top {int((1-ov20['threshold_pct'])*100)}%"],
        "value": [ov10["overlap_rate_inter_over_ics"], ov20["overlap_rate_inter_over_ics"]],
    })],
    ignore_index=True
)
metrics_out.to_csv(METRICS_FP, index=False)
print("Saved:", METRICS_FP)

# also save overlap detail table
OVERLAP_FP = OUT_DIR / "metrics_top_overlap_detail.csv"
overlap_df.to_csv(OVERLAP_FP, index=False)
print("Saved:", OVERLAP_FP)

# -------------------------
# 5) Quadrant classification (percentile thresholds)
# -------------------------
# Define "High" using percentile threshold (default top10%)
df["ics_high"]  = (df["ics_pct"]  >= HIGH_PCT).astype(int)
df["deri_high"] = (df["deri_pct"] >= HIGH_PCT).astype(int)

def quadrant_label(row):
    ih, dh = row["ics_high"], row["deri_high"]
    if ih == 1 and dh == 1:
        return "Q1: High ICS & High DERI (consensus high risk)"
    if ih == 1 and dh == 0:
        return "Q2: High ICS & Low DERI (OVERLOOKED areas)"
    if ih == 0 and dh == 1:
        return "Q3: Low ICS & High DERI (DERI-flagged, not ICS)"
    return "Q4: Low ICS & Low DERI (consensus low risk)"

df["quadrant"] = df.apply(quadrant_label, axis=1)

# The one you care about most:
df["overlooked_flag"] = ((df["ics_high"] == 1) & (df["deri_high"] == 0)).astype(int)

# -------------------------
# 6) Outputs: full compare table + overlooked list
# -------------------------
COMPARE_FP = OUT_DIR / "ics_deri_compare_lsoa.csv"
keep = [
    "lsoa_code",
    ICS_COL, DERI_COL,
    "ics_pct", "deri_pct",
    "ics_z", "deri_z",
    "ics_rank_desc", "deri_rank_desc",
    "ics_high", "deri_high",
    "quadrant", "overlooked_flag",
]
df[keep].to_csv(COMPARE_FP, index=False)
print("Saved:", COMPARE_FP)

# Overlooked list (sorted by ICS risk, highest first)
OVERLOOKED_FP = OUT_DIR / "overlooked_list.csv"
overlooked = df[df["overlooked_flag"] == 1].copy()
overlooked = overlooked.sort_values(by=ICS_COL, ascending=False)
overlooked[keep].to_csv(OVERLOOKED_FP, index=False)
print("Saved:", OVERLOOKED_FP, "| n:", len(overlooked))

# quick counts for sanity
counts = df["quadrant"].value_counts(dropna=False)
print("\nQuadrant counts:\n", counts)

# -------------------------
# 7) (Optional) Scatter plot with quadrants (safe)
# -------------------------
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6.5, 5.5))
    # use percentile (more policy-friendly)
    # No manual colors required; simple markers by label
    for lab in df["quadrant"].dropna().unique():
        sub = df[df["quadrant"] == lab]
        plt.scatter(sub["ics_pct"], sub["deri_pct"], s=10, alpha=0.6, label=lab)
    plt.axvline(HIGH_PCT, linestyle="--")
    plt.axhline(HIGH_PCT, linestyle="--")
    plt.xlabel("ICS percentile (higher = higher risk)")
    plt.ylabel(f"{DERI_COL} percentile (higher = higher risk)")
    plt.title("ICS vs DERI (percentile) with quadrant thresholds")
    plt.legend(fontsize=7, loc="best")
    FIG_FP = OUT_DIR / "fig_scatter_ics_vs_deri_percentile_quadrants.png"
    plt.tight_layout()
    plt.savefig(FIG_FP, dpi=160)
    plt.close()
    print("Saved:", FIG_FP)
except Exception as e:
    print("[INFO] Scatter plot skipped:", repr(e))


Joined rows: 4813 | valid pairs: 4813 | DERI col: rDERI_score_0_10
Saved: QM data/output/4_5_deri/metrics_consistency.csv
Saved: QM data/output/4_5_deri/metrics_top_overlap_detail.csv
Saved: QM data/output/4_5_deri/ics_deri_compare_lsoa.csv
Saved: QM data/output/4_5_deri/overlooked_list.csv | n: 256

Quadrant counts:
 quadrant
Q4: Low ICS & Low DERI (consensus low risk)        4075
Q3: Low ICS & High DERI (DERI-flagged, not ICS)     256
Q2: High ICS & Low DERI (OVERLOOKED areas)          256
Q1: High ICS & High DERI (consensus high risk)      226
Name: count, dtype: int64
Saved: QM data/output/4_5_deri/fig_scatter_ics_vs_deri_percentile_quadrants.png


In [42]:
# =========================
# Top-k overlap visualizations (Top 5/10/20/30) — standalone
# Inputs:
# - QM data/output/4_4_ics/ics_table_lsoa.csv
# - analysis_table_raw_plus_all_plus_rderi.csv (contains rDERI_score_0_10)
# Outputs (in QM data/output/4_5_deri/):
# - metrics_top_overlap_detail_top5_10_20_30.csv
# - fig_topk_overlap_rate_line.png
# - fig_topk_overlap_rate_bar.png
# - fig_topk_jaccard_line.png
# - fig_topk_intersection_count_line.png
# =========================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- paths ----
ICS_FP = Path("QM data/output/4_4_ics/ics_table_lsoa.csv")

RAW_CANDIDATES = [
    Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv"),
    Path("/mnt/data/analysis_table_raw_plus_all_plus_rderi.csv"),
]
RAW_FP = next((p for p in RAW_CANDIDATES if p.exists()), None)

OUT_DIR = Path("QM data/output/4_5_deri")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not ICS_FP.exists():
    raise FileNotFoundError(f"Missing: {ICS_FP}")
if RAW_FP is None:
    raise FileNotFoundError(f"Missing raw table. Tried: {RAW_CANDIDATES}")

# ---- load ----
ics = pd.read_csv(ICS_FP, dtype={"lsoa_code": str})
ics["lsoa_code"] = ics["lsoa_code"].astype(str).str.strip().str.upper()

raw = pd.read_csv(RAW_FP, dtype={"lsoa_code": str})
raw["lsoa_code"] = raw["lsoa_code"].astype(str).str.strip().str.upper()

# choose columns
ICS_COL = "ics_score_z"  # 更推荐用 z 分数（与前面一致）
DERI_COL = "rDERI_score_0_10" if "rDERI_score_0_10" in raw.columns else None
if DERI_COL is None:
    raise ValueError("raw 表里找不到 rDERI_score_0_10。请确认你用的是含 rDERI 的那张主表。")
if ICS_COL not in ics.columns:
    raise ValueError(f"ics 表里找不到 {ICS_COL}（可改用 ics_score_0_100）。")

df = ics.merge(raw[["lsoa_code", DERI_COL]], on="lsoa_code", how="inner")
df[ICS_COL] = pd.to_numeric(df[ICS_COL], errors="coerce")
df[DERI_COL] = pd.to_numeric(df[DERI_COL], errors="coerce")

# percentiles (same-scale method A)
df["ics_pct"]  = df[ICS_COL].rank(pct=True, method="average")
df["deri_pct"] = df[DERI_COL].rank(pct=True, method="average")

# ---- compute overlaps for Top-k ----
wanted_k = [5, 10, 20, 30]
rows = []
for k in wanted_k:
    thr = 1 - k / 100.0
    A = df["ics_pct"] >= thr
    B = df["deri_pct"] >= thr

    inter = int((A & B).sum())
    sizeA = int(A.sum())
    sizeB = int(B.sum())
    union = int((A | B).sum())

    rows.append({
        "top_k_pct": k,
        "threshold_pct": thr,
        "n_ics_high": sizeA,
        "n_deri_high": sizeB,
        "n_intersection": inter,
        "overlap_rate_inter_over_ics": inter / sizeA if sizeA else np.nan,    # |A∩B|/|A|
        "jaccard_inter_over_union": inter / union if union else np.nan,       # |A∩B|/|A∪B|
    })

res = pd.DataFrame(rows).sort_values("top_k_pct")

OUT_TABLE = OUT_DIR / "metrics_top_overlap_detail_top5_10_20_30.csv"
res.to_csv(OUT_TABLE, index=False)
print("Saved:", OUT_TABLE)
display(res)

# ---- plotting helpers ----
def save_line(x, y, xlabel, ylabel, title, path, y01=False):
    plt.figure(figsize=(7.2, 4.8))
    plt.plot(x, y, marker="o")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(wanted_k)
    if y01:
        plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()

def save_bar(x_labels, y, xlabel, ylabel, title, path, y01=False):
    plt.figure(figsize=(7.2, 4.8))
    plt.bar([str(v) for v in x_labels], y)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    if y01:
        plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()

# 1) Overlap rate line
fig1 = OUT_DIR / "fig_topk_overlap_rate_line.png"
save_line(
    res["top_k_pct"],
    res["overlap_rate_inter_over_ics"],
    "Top-k threshold (%)",
    "Overlap rate |A∩B| / |A| (ICS→DERI)",
    "Top-k overlap rate between ICS and DERI",
    fig1,
    y01=True
)

# 2) Overlap rate bar
fig2 = OUT_DIR / "fig_topk_overlap_rate_bar.png"
save_bar(
    res["top_k_pct"],
    res["overlap_rate_inter_over_ics"],
    "Top-k threshold (%)",
    "Overlap rate |A∩B| / |A| (ICS→DERI)",
    "Top-k overlap rate (bar)",
    fig2,
    y01=True
)

# 3) Jaccard line
fig3 = OUT_DIR / "fig_topk_jaccard_line.png"
save_line(
    res["top_k_pct"],
    res["jaccard_inter_over_union"],
    "Top-k threshold (%)",
    "Jaccard |A∩B| / |A∪B|",
    "Top-k Jaccard similarity between ICS and DERI",
    fig3,
    y01=True
)

# 4) Intersection count line
fig4 = OUT_DIR / "fig_topk_intersection_count_line.png"
plt.figure(figsize=(7.2, 4.8))
plt.plot(res["top_k_pct"], res["n_intersection"], marker="o")
plt.xlabel("Top-k threshold (%)")
plt.ylabel("Count of intersection |A∩B|")
plt.title("Top-k intersection counts (how many LSOAs overlap)")
plt.xticks(wanted_k)
plt.tight_layout()
plt.savefig(fig4, dpi=160)
plt.close()

print("Saved figures:")
print(" -", fig1)
print(" -", fig2)
print(" -", fig3)
print(" -", fig4)

Saved: QM data/output/4_5_deri/metrics_top_overlap_detail_top5_10_20_30.csv


,top_k_pct,threshold_pct,n_ics_high,n_deri_high,n_intersection,overlap_rate_inter_over_ics,jaccard_inter_over_union
0,5,0.95,241,241,95,0.394191,0.245478
1,10,0.90,482,482,226,0.468880,0.306233
2,20,0.80,963,963,555,0.576324,0.404814
3,30,0.70,1444,1444,958,0.663435,0.496373


Saved figures:
 - QM data/output/4_5_deri/fig_topk_overlap_rate_line.png
 - QM data/output/4_5_deri/fig_topk_overlap_rate_bar.png
 - QM data/output/4_5_deri/fig_topk_jaccard_line.png
 - QM data/output/4_5_deri/fig_topk_intersection_count_line.png


In [43]:
# =========================
# 4.6 Spatial statistics (NO esda, NO libpysal Queen.from_dataframe):
# - Build Queen adjacency via geopandas.sindex + touches (stable)
# - Global Moran's I (permutation p)
# - Local Moran (LISA) (permutation p + HH/LL/HL/LH)
# - Gi* (permutation z + p + hotspot/coldspot)
# - Plot as centroid points (avoid heavy polygon plotting)
# =========================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from scipy import sparse

# -------------------------
# 0) Config
# -------------------------
PERMUTATIONS = 99     # 先用 99，稳定且快；跑通后可改 199/499/999
ALPHA = 0.05
RANDOM_SEED = 42

OUT_DIR = Path("QM data/output/4_6_spatial_stats")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- paths ----
BOUNDARY_CANDIDATES = [
    Path("QM data/geo/lsoa_boundary_clean.gpkg"),
    Path("QM data/clean/lsoa_boundary_clean.gpkg"),
    Path("lsoa_boundary_clean.gpkg"),
    Path("QM data/12/lsoa_boundary_clean.gpkg"),
]
BOUNDARY_FP = next((p for p in BOUNDARY_CANDIDATES if p.exists()), None)
if BOUNDARY_FP is None:
    raise FileNotFoundError(f"找不到 lsoa_boundary_clean.gpkg。尝试过：{BOUNDARY_CANDIDATES}")

RAW_CANDIDATES = [
    Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv"),
    Path("/mnt/data/analysis_table_raw_plus_all_plus_rderi.csv"),
]
RAW_FP = next((p for p in RAW_CANDIDATES if p.exists()), None)
if RAW_FP is None:
    raise FileNotFoundError(f"找不到 analysis_table_raw_plus_all_plus_rderi.csv。尝试过：{RAW_CANDIDATES}")

ICS_CANDIDATES = [
    Path("QM data/output/4_4_ics/ics_table_lsoa.csv"),
    Path("/mnt/data/ics_table_lsoa.csv"),
]
ICS_FP = next((p for p in ICS_CANDIDATES if p.exists()), None)
if ICS_FP is None:
    raise FileNotFoundError(f"找不到 ics_table_lsoa.csv。尝试过：{ICS_CANDIDATES}")


# -------------------------
# 1) Utilities
# -------------------------
def pick_col(cols, candidates):
    cols_low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_low:
            return cols_low[cand.lower()]
    for cand in candidates:
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

def standardize(series: pd.Series) -> np.ndarray:
    x = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    m = np.nanmean(x)
    sd = np.nanstd(x)
    if not np.isfinite(sd) or sd == 0:
        return np.zeros_like(x, dtype=float)
    return (x - m) / sd

def plot_points(gdf, col, title, out_fp, categorical=True):
    # centroid point plot: much lighter than polygon
    pts = gdf.copy()
    pts["geometry"] = pts.geometry.centroid
    ax = pts.plot(column=col, legend=True, figsize=(8, 8))
    ax.set_title(title)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_fp, dpi=180)
    plt.close()

def plot_points_cont(gdf, col, title, out_fp):
    pts = gdf.copy()
    pts["geometry"] = pts.geometry.centroid
    ax = pts.plot(column=col, legend=True, figsize=(8, 8))
    ax.set_title(title)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_fp, dpi=180)
    plt.close()


# -------------------------
# 2) Build Queen contiguity via sindex + touches (NO libpysal)
# -------------------------
def build_queen_sparse_by_touches(gdf: gpd.GeoDataFrame):
    """
    Returns:
      gdf2: filtered gdf (drops islands)
      W_rs: row-standardized sparse matrix
      neighbors: list of neighbor indices per i
      islands: indices dropped (in original gdf2 indexing before drop)
    """
    gdf = gdf.copy().reset_index(drop=True)

    # fix invalid geometries (reduces chance of crashes / weird adjacency)
    # buffer(0) is classic fix; skip if geometry is empty
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()
    gdf = gdf.reset_index(drop=True)
    gdf["geometry"] = gdf.geometry.buffer(0)

    sindex = gdf.sindex
    neighbors = []
    for i, geom in enumerate(gdf.geometry):
        cand = list(sindex.intersection(geom.bounds))
        cand = [j for j in cand if j != i]
        nb = []
        for j in cand:
            # Queen contiguity
            if geom.touches(gdf.geometry.iloc[j]):
                nb.append(j)
        neighbors.append(nb)

    islands = [i for i, nb in enumerate(neighbors) if len(nb) == 0]

    if islands:
        keep = [i for i in range(len(neighbors)) if i not in set(islands)]
        gdf2 = gdf.iloc[keep].reset_index(drop=True)
        old2new = {old: new for new, old in enumerate(keep)}
        neighbors2 = []
        for old in keep:
            neighbors2.append([old2new[j] for j in neighbors[old] if j in old2new])
        neighbors = neighbors2
    else:
        gdf2 = gdf

    n = gdf2.shape[0]
    rows, cols, data = [], [], []
    for i, nb in enumerate(neighbors):
        for j in nb:
            rows.append(i); cols.append(j); data.append(1.0)

    W = sparse.csr_matrix((data, (rows, cols)), shape=(n, n))

    # row-standardize
    row_sums = np.array(W.sum(axis=1)).ravel()
    row_sums[row_sums == 0] = 1.0
    W_rs = sparse.diags(1.0 / row_sums) @ W
    return gdf2, W_rs.tocsr(), neighbors, islands


# -------------------------
# 3) Stats (NO esda)
# -------------------------
def global_moran_I(z: np.ndarray, W: sparse.csr_matrix, permutations=99, seed=42):
    rng = np.random.default_rng(seed)
    n = len(z)
    S0 = float(W.sum())
    I_obs = (n / S0) * (float(z @ (W @ z)) / float(z @ z))

    # permutation p (two-sided)
    more_extreme = 0
    for _ in range(permutations):
        zp = rng.permutation(z)
        Ip = (n / S0) * (float(zp @ (W @ zp)) / float(zp @ zp))
        if abs(Ip) >= abs(I_obs):
            more_extreme += 1
    p_sim = (more_extreme + 1) / (permutations + 1)

    # z via permutation distribution
    perms = np.empty(permutations, dtype=float)
    rng2 = np.random.default_rng(seed + 1)
    for k in range(permutations):
        zp = rng2.permutation(z)
        perms[k] = (n / S0) * (float(zp @ (W @ zp)) / float(zp @ zp))
    z_sim = (I_obs - perms.mean()) / (perms.std(ddof=0) + 1e-12)

    return {"I": I_obs, "p_sim": p_sim, "z_sim": z_sim, "n": n, "permutations": permutations}

def local_moran(z: np.ndarray, W: sparse.csr_matrix, permutations=99, seed=42, alpha=0.05):
    rng = np.random.default_rng(seed)
    n = len(z)
    m2 = float((z @ z) / n)
    lag = np.asarray(W @ z)
    I_obs = (z * lag) / (m2 + 1e-12)

    more_extreme = np.zeros(n, dtype=int)
    abs_obs = np.abs(I_obs)

    for _ in range(permutations):
        zp = rng.permutation(z)
        lag_p = np.asarray(W @ zp)
        I_p = (zp * lag_p) / (float((zp @ zp) / n) + 1e-12)
        more_extreme += (np.abs(I_p) >= abs_obs)

    p = (more_extreme + 1) / (permutations + 1)

    cls = np.array(["Not sig"] * n, dtype=object)
    sig = p < alpha
    posz = z > 0
    posl = lag > 0
    cls[sig & posz & posl] = "HH (hotspot)"
    cls[sig & (~posz) & (~posl)] = "LL (coldspot)"
    cls[sig & posz & (~posl)] = "HL (high-low)"
    cls[sig & (~posz) & posl] = "LH (low-high)"

    return {"I_local": I_obs, "p": p, "lag": lag, "class": cls}

def gistar_perm(y: np.ndarray, W: sparse.csr_matrix, permutations=99, seed=42, alpha=0.05):
    rng = np.random.default_rng(seed)
    n = len(y)

    W_star = W + sparse.identity(n, format="csr")
    row_sums = np.array(W_star.sum(axis=1)).ravel()
    row_sums[row_sums == 0] = 1.0
    W_star = sparse.diags(1.0 / row_sums) @ W_star

    g_obs = np.asarray(W_star @ y)

    mean = np.zeros(n, dtype=float)
    M2 = np.zeros(n, dtype=float)
    more_extreme = np.zeros(n, dtype=int)
    abs_obs = np.abs(g_obs)

    for t in range(1, permutations + 1):
        yp = rng.permutation(y)
        g_p = np.asarray(W_star @ yp)

        delta = g_p - mean
        mean += delta / t
        delta2 = g_p - mean
        M2 += delta * delta2

        more_extreme += (np.abs(g_p) >= abs_obs)

    var = M2 / max(permutations, 1)
    sd = np.sqrt(var) + 1e-12
    z = (g_obs - mean) / sd
    p = (more_extreme + 1) / (permutations + 1)

    cls = np.array(["Not sig"] * n, dtype=object)
    sig = p < alpha
    cls[sig & (z > 0)] = "Hotspot (Gi*)"
    cls[sig & (z < 0)] = "Coldspot (Gi*)"
    return {"g": g_obs, "z": z, "p": p, "class": cls}


# -------------------------
# 4) Load & join
# -------------------------
bnd = gpd.read_file(BOUNDARY_FP)
key_bnd = pick_col(bnd.columns, ["lsoa_code", "lsoa21cd", "LSOA21CD", "lsoa11cd", "LSOA11CD"])
if key_bnd is None:
    raise ValueError(f"边界文件找不到 LSOA code 字段。当前字段：{list(bnd.columns)}")
bnd["lsoa_code"] = bnd[key_bnd].astype(str).str.strip().str.upper()

raw = pd.read_csv(RAW_FP, dtype={"lsoa_code": str})
raw["lsoa_code"] = raw["lsoa_code"].astype(str).str.strip().str.upper()

ics = pd.read_csv(ICS_FP, dtype={"lsoa_code": str})
ics["lsoa_code"] = ics["lsoa_code"].astype(str).str.strip().str.upper()

dpi_col = pick_col(raw.columns, ["dpi_score", "dpi", "DPI"])
ics_col = pick_col(ics.columns, ["ics_score_z", "ics_score_0_100", "ics_score"])
if dpi_col is None:
    raise ValueError("raw 表找不到 dpi 列")
if ics_col is None:
    raise ValueError("ics 表找不到 ics 分数列")

gdf = bnd.merge(raw[["lsoa_code", dpi_col]], on="lsoa_code", how="left") \
         .merge(ics[["lsoa_code", ics_col]], on="lsoa_code", how="left")

print("Boundary rows:", len(bnd), "| Joined rows:", len(gdf))
print("DPI col:", dpi_col, "| ICS col:", ics_col)


# -------------------------
# 5) Run for one variable
# -------------------------
def run_var(gdf_in, value_col, tag):
    g0 = gdf_in.dropna(subset=[value_col]).copy().reset_index(drop=True)

    # build weights safely
    g1, W, neighbors, islands = build_queen_sparse_by_touches(g0)

    y_raw = pd.to_numeric(g1[value_col], errors="coerce").to_numpy(dtype=float)
    z = standardize(g1[value_col])

    glob = global_moran_I(z, W, permutations=PERMUTATIONS, seed=RANDOM_SEED)
    lisa = local_moran(z, W, permutations=PERMUTATIONS, seed=RANDOM_SEED + 10, alpha=ALPHA)
    gi = gistar_perm(y_raw, W, permutations=PERMUTATIONS, seed=RANDOM_SEED + 20, alpha=ALPHA)

    g1[f"{tag}_z"] = z
    g1["moran_lag_z"] = lisa["lag"]
    g1["lisa_I"] = lisa["I_local"]
    g1["lisa_p"] = lisa["p"]
    g1["lisa_class"] = lisa["class"]

    g1["gistar_g"] = gi["g"]
    g1["gistar_z"] = gi["z"]
    g1["gistar_p"] = gi["p"]
    g1["gistar_class"] = gi["class"]

    out_table = OUT_DIR / f"table_lsoa_spatial_stats_{tag}.csv"
    g1[["lsoa_code", value_col, f"{tag}_z",
        "lisa_I","lisa_p","lisa_class",
        "gistar_z","gistar_p","gistar_class"]].to_csv(out_table, index=False)

    fig_lisa = OUT_DIR / f"fig_lisa_{tag}_points.png"
    plot_points(g1, "lisa_class", f"LISA clusters (points) — {tag}", fig_lisa)

    fig_gi = OUT_DIR / f"fig_gistar_{tag}_points.png"
    plot_points(g1, "gistar_class", f"Gi* hotspots (points) — {tag}", fig_gi)

    fig_z = OUT_DIR / f"fig_{tag}_z_points.png"
    plot_points_cont(g1, f"{tag}_z", f"{tag} z-score (points)", fig_z)

    return glob, out_table, fig_lisa, fig_gi, fig_z, len(g0), len(g1), len(islands)

dpi_glob, dpi_table, dpi_lisa_fig, dpi_gi_fig, dpi_z_fig, dpi_n0, dpi_n1, dpi_islands = run_var(gdf, dpi_col, "dpi")
ics_glob, ics_table, ics_lisa_fig, ics_gi_fig, ics_z_fig, ics_n0, ics_n1, ics_islands = run_var(gdf, ics_col, "ics")

summary = pd.DataFrame([
    {"variable":"DPI", "source_col":dpi_col, "n_before":dpi_n0, "n_used":dpi_n1, "n_islands_dropped":dpi_islands, **dpi_glob},
    {"variable":"ICS", "source_col":ics_col, "n_before":ics_n0, "n_used":ics_n1, "n_islands_dropped":ics_islands, **ics_glob},
])
summary_fp = OUT_DIR / "global_moran_summary.csv"
summary.to_csv(summary_fp, index=False)

print("\nSaved:", summary_fp)
display(summary)

print("\nSaved DPI outputs:")
print(" -", dpi_table)
print(" -", dpi_lisa_fig)
print(" -", dpi_gi_fig)
print(" -", dpi_z_fig)

print("\nSaved ICS outputs:")
print(" -", ics_table)
print(" -", ics_lisa_fig)
print(" -", ics_gi_fig)
print(" -", ics_z_fig)

Boundary rows: 4994 | Joined rows: 4994
DPI col: dpi_score | ICS col: ics_score_z

Saved: QM data/output/4_6_spatial_stats/global_moran_summary.csv


,variable,source_col,n_before,n_used,n_islands_dropped,I,p_sim,z_sim,n,permutations
0,DPI,dpi_score,4813,4813,0,0.404847,0.01,44.471496,4813,99
1,ICS,ics_score_z,4813,4813,0,0.364144,0.01,42.238321,4813,99



Saved DPI outputs:
 - QM data/output/4_6_spatial_stats/table_lsoa_spatial_stats_dpi.csv
 - QM data/output/4_6_spatial_stats/fig_lisa_dpi_points.png
 - QM data/output/4_6_spatial_stats/fig_gistar_dpi_points.png
 - QM data/output/4_6_spatial_stats/fig_dpi_z_points.png

Saved ICS outputs:
 - QM data/output/4_6_spatial_stats/table_lsoa_spatial_stats_ics.csv
 - QM data/output/4_6_spatial_stats/fig_lisa_ics_points.png
 - QM data/output/4_6_spatial_stats/fig_gistar_ics_points.png
 - QM data/output/4_6_spatial_stats/fig_ics_z_points.png


In [44]:
# =========================
# 4.6.1 Spatial statistics (NO esda, NO libpysal Queen.from_dataframe):
# - Build Queen adjacency via geopandas.sindex + touches (stable)
# - Global Moran's I (permutation p)
# - Local Moran (LISA) (permutation p + HH/LL/HL/LH)
# - Gi* (permutation z + p + hotspot/coldspot)
# - Plot as centroid points (avoid heavy polygon plotting)
#   + NEWS-style Gi* plot: fade Not significant + add London outline + legend order + public title
# =========================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from scipy import sparse
from matplotlib.lines import Line2D

# -------------------------
# 0) Config
# -------------------------
PERMUTATIONS = 99     # 先用 99，稳定且快；跑通后可改 199/499/999
ALPHA = 0.05
RANDOM_SEED = 42

OUT_DIR = Path("QM data/output/4_6.1_spatial_stats")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- paths ----
BOUNDARY_CANDIDATES = [
    Path("QM data/geo/lsoa_boundary_clean.gpkg"),
    Path("QM data/clean/lsoa_boundary_clean.gpkg"),
    Path("lsoa_boundary_clean.gpkg"),
    Path("QM data/12/lsoa_boundary_clean.gpkg"),
]
BOUNDARY_FP = next((p for p in BOUNDARY_CANDIDATES if p.exists()), None)
if BOUNDARY_FP is None:
    raise FileNotFoundError(f"找不到 lsoa_boundary_clean.gpkg。尝试过：{BOUNDARY_CANDIDATES}")

RAW_CANDIDATES = [
    Path("QM data/clean/analysis_table_raw_plus_all_plus_rderi.csv"),
    Path("/mnt/data/analysis_table_raw_plus_all_plus_rderi.csv"),
]
RAW_FP = next((p for p in RAW_CANDIDATES if p.exists()), None)
if RAW_FP is None:
    raise FileNotFoundError(f"找不到 analysis_table_raw_plus_all_plus_rderi.csv。尝试过：{RAW_CANDIDATES}")

ICS_CANDIDATES = [
    Path("QM data/output/4_4_ics/ics_table_lsoa.csv"),
    Path("/mnt/data/ics_table_lsoa.csv"),
]
ICS_FP = next((p for p in ICS_CANDIDATES if p.exists()), None)
if ICS_FP is None:
    raise FileNotFoundError(f"找不到 ics_table_lsoa.csv。尝试过：{ICS_CANDIDATES}")


# -------------------------
# 1) Utilities
# -------------------------
def pick_col(cols, candidates):
    cols_low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_low:
            return cols_low[cand.lower()]
    for cand in candidates:
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

def standardize(series: pd.Series) -> np.ndarray:
    x = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    m = np.nanmean(x)
    sd = np.nanstd(x)
    if not np.isfinite(sd) or sd == 0:
        return np.zeros_like(x, dtype=float)
    return (x - m) / sd

def _save_fig(out_fp, dpi=220):
    plt.tight_layout()
    plt.savefig(out_fp, dpi=dpi)
    plt.close()

def plot_points_lisa(gdf, col, title, out_fp):
    """Generic centroid point plot for LISA categories (HH/LL/HL/LH/Not sig)."""
    pts = gdf.copy()
    pts["geometry"] = pts.geometry.centroid
    ax = pts.plot(column=col, legend=True, figsize=(8, 8), markersize=10)
    ax.set_title(title, fontsize=12)
    ax.set_axis_off()
    _save_fig(out_fp)

def plot_points_cont(gdf, col, title, out_fp):
    """Generic centroid continuous point plot (z-scores)."""
    pts = gdf.copy()
    pts["geometry"] = pts.geometry.centroid
    ax = pts.plot(column=col, legend=True, figsize=(8, 8), markersize=10)
    ax.set_title(title, fontsize=12)
    ax.set_axis_off()
    _save_fig(out_fp)

def plot_gistar_points_news(gdf_poly, cls_col, title, out_fp):
    import numpy as np
    import geopandas as gpd
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    g_poly = gdf_poly.copy()

    # centroid points
    g_pts = g_poly.copy()
    g_pts["geometry"] = g_pts.geometry.centroid

    # normalize class strings
    def _norm(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "Not significant"
        s = str(v).strip().lower()
        if "hot" in s:
            return "Hotspot"
        if "cold" in s:
            return "Coldspot"
        return "Not significant"

    g_pts["_class_norm"] = g_pts[cls_col].map(_norm)

    # dissolve London geometry (avoid unary_union warning)
    if hasattr(g_poly.geometry, "union_all"):
        london_poly = g_poly.geometry.union_all()
    else:
        london_poly = g_poly.geometry.unary_union

    london_boundary = london_poly.boundary  # LineString/MultiLineString

    # style
    COLOR = {
        "Hotspot": "#8c564b",
        "Coldspot": "#1f77b4",
        "Not significant": "#17becf"
    }
    s_sig, s_ns = 18, 12
    a_sig, a_ns = 0.95, 0.10

    outline_color = "0.35"
    outline_lw = 1.4
    outline_alpha = 0.85

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.set_title(title, fontsize=13, pad=10)

    # points first: Not significant -> Coldspot -> Hotspot
    for cls in ["Not significant", "Coldspot", "Hotspot"]:
        sub = g_pts[g_pts["_class_norm"] == cls]
        if sub.empty:
            continue
        if cls == "Not significant":
            sub.plot(ax=ax, color=COLOR[cls], markersize=s_ns, alpha=a_ns, zorder=2)
        else:
            sub.plot(ax=ax, color=COLOR[cls], markersize=s_sig, alpha=a_sig, zorder=3)

    # ✅ outline LAST and IMPORTANT: for LineString use color=..., NOT edgecolor
    gpd.GeoSeries([london_boundary], crs=g_poly.crs).plot(
        ax=ax,
        color=outline_color,          # <-- 关键修复：用 color 画线
        linewidth=outline_lw,
        alpha=outline_alpha,
        zorder=10
    )

    # legend
    handles = [
        Line2D([0], [0], marker="o", color="none",
               markerfacecolor=COLOR["Hotspot"], markersize=8, label="Hotspot"),
        Line2D([0], [0], marker="o", color="none",
               markerfacecolor=COLOR["Coldspot"], markersize=8, label="Coldspot"),
        Line2D([0], [0], marker="o", color="none",
               markerfacecolor=COLOR["Not significant"], alpha=a_ns,
               markersize=8, label="Not significant"),
    ]
    ax.legend(handles=handles, loc="upper right", frameon=True)

    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_fp, dpi=220)
    plt.close()



# -------------------------
# 2) Build Queen contiguity via sindex + touches (NO libpysal)
# -------------------------
def build_queen_sparse_by_touches(gdf: gpd.GeoDataFrame):
    """
    Returns:
      gdf2: filtered gdf (drops islands)
      W_rs: row-standardized sparse matrix
      neighbors: list of neighbor indices per i
      islands: indices dropped (in original gdf2 indexing before drop)
    """
    gdf = gdf.copy().reset_index(drop=True)

    # fix invalid geometries
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()
    gdf = gdf.reset_index(drop=True)
    gdf["geometry"] = gdf.geometry.buffer(0)

    sindex = gdf.sindex
    neighbors = []
    for i, geom in enumerate(gdf.geometry):
        cand = list(sindex.intersection(geom.bounds))
        cand = [j for j in cand if j != i]
        nb = []
        for j in cand:
            if geom.touches(gdf.geometry.iloc[j]):
                nb.append(j)
        neighbors.append(nb)

    islands = [i for i, nb in enumerate(neighbors) if len(nb) == 0]

    if islands:
        keep = [i for i in range(len(neighbors)) if i not in set(islands)]
        gdf2 = gdf.iloc[keep].reset_index(drop=True)
        old2new = {old: new for new, old in enumerate(keep)}
        neighbors2 = []
        for old in keep:
            neighbors2.append([old2new[j] for j in neighbors[old] if j in old2new])
        neighbors = neighbors2
    else:
        gdf2 = gdf

    n = gdf2.shape[0]
    rows, cols, data = [], [], []
    for i, nb in enumerate(neighbors):
        for j in nb:
            rows.append(i); cols.append(j); data.append(1.0)

    W = sparse.csr_matrix((data, (rows, cols)), shape=(n, n))

    # row-standardize
    row_sums = np.array(W.sum(axis=1)).ravel()
    row_sums[row_sums == 0] = 1.0
    W_rs = sparse.diags(1.0 / row_sums) @ W
    return gdf2, W_rs.tocsr(), neighbors, islands


# -------------------------
# 3) Stats (NO esda)
# -------------------------
def global_moran_I(z: np.ndarray, W: sparse.csr_matrix, permutations=99, seed=42):
    rng = np.random.default_rng(seed)
    n = len(z)
    S0 = float(W.sum())
    I_obs = (n / S0) * (float(z @ (W @ z)) / float(z @ z))

    # permutation p (two-sided)
    more_extreme = 0
    for _ in range(permutations):
        zp = rng.permutation(z)
        Ip = (n / S0) * (float(zp @ (W @ zp)) / float(zp @ zp))
        if abs(Ip) >= abs(I_obs):
            more_extreme += 1
    p_sim = (more_extreme + 1) / (permutations + 1)

    # z via permutation distribution
    perms = np.empty(permutations, dtype=float)
    rng2 = np.random.default_rng(seed + 1)
    for k in range(permutations):
        zp = rng2.permutation(z)
        perms[k] = (n / S0) * (float(zp @ (W @ zp)) / float(zp @ zp))
    z_sim = (I_obs - perms.mean()) / (perms.std(ddof=0) + 1e-12)

    return {"I": I_obs, "p_sim": p_sim, "z_sim": z_sim, "n": n, "permutations": permutations}

def local_moran(z: np.ndarray, W: sparse.csr_matrix, permutations=99, seed=42, alpha=0.05):
    rng = np.random.default_rng(seed)
    n = len(z)
    m2 = float((z @ z) / n)
    lag = np.asarray(W @ z)
    I_obs = (z * lag) / (m2 + 1e-12)

    more_extreme = np.zeros(n, dtype=int)
    abs_obs = np.abs(I_obs)

    for _ in range(permutations):
        zp = rng.permutation(z)
        lag_p = np.asarray(W @ zp)
        I_p = (zp * lag_p) / (float((zp @ zp) / n) + 1e-12)
        more_extreme += (np.abs(I_p) >= abs_obs)

    p = (more_extreme + 1) / (permutations + 1)

    cls = np.array(["Not sig"] * n, dtype=object)
    sig = p < alpha
    posz = z > 0
    posl = lag > 0
    cls[sig & posz & posl] = "HH (hotspot)"
    cls[sig & (~posz) & (~posl)] = "LL (coldspot)"
    cls[sig & posz & (~posl)] = "HL (high-low)"
    cls[sig & (~posz) & posl] = "LH (low-high)"

    return {"I_local": I_obs, "p": p, "lag": lag, "class": cls}

def gistar_perm(y: np.ndarray, W: sparse.csr_matrix, permutations=99, seed=42, alpha=0.05):
    rng = np.random.default_rng(seed)
    n = len(y)

    W_star = W + sparse.identity(n, format="csr")
    row_sums = np.array(W_star.sum(axis=1)).ravel()
    row_sums[row_sums == 0] = 1.0
    W_star = sparse.diags(1.0 / row_sums) @ W_star

    g_obs = np.asarray(W_star @ y)

    mean = np.zeros(n, dtype=float)
    M2 = np.zeros(n, dtype=float)
    more_extreme = np.zeros(n, dtype=int)
    abs_obs = np.abs(g_obs)

    for t in range(1, permutations + 1):
        yp = rng.permutation(y)
        g_p = np.asarray(W_star @ yp)

        delta = g_p - mean
        mean += delta / t
        delta2 = g_p - mean
        M2 += delta * delta2

        more_extreme += (np.abs(g_p) >= abs_obs)

    var = M2 / max(permutations, 1)
    sd = np.sqrt(var) + 1e-12
    z = (g_obs - mean) / sd
    p = (more_extreme + 1) / (permutations + 1)

    cls = np.array(["Not sig"] * n, dtype=object)
    sig = p < alpha
    cls[sig & (z > 0)] = "Hotspot (Gi*)"
    cls[sig & (z < 0)] = "Coldspot (Gi*)"
    return {"g": g_obs, "z": z, "p": p, "class": cls}


# -------------------------
# 4) Load & join
# -------------------------
bnd = gpd.read_file(BOUNDARY_FP)
key_bnd = pick_col(bnd.columns, ["lsoa_code", "lsoa21cd", "LSOA21CD", "lsoa11cd", "LSOA11CD"])
if key_bnd is None:
    raise ValueError(f"边界文件找不到 LSOA code 字段。当前字段：{list(bnd.columns)}")
bnd["lsoa_code"] = bnd[key_bnd].astype(str).str.strip().str.upper()

raw = pd.read_csv(RAW_FP, dtype={"lsoa_code": str})
raw["lsoa_code"] = raw["lsoa_code"].astype(str).str.strip().str.upper()

ics = pd.read_csv(ICS_FP, dtype={"lsoa_code": str})
ics["lsoa_code"] = ics["lsoa_code"].astype(str).str.strip().str.upper()

dpi_col = pick_col(raw.columns, ["dpi_score", "dpi", "DPI"])
ics_col = pick_col(ics.columns, ["ics_score_z", "ics_score_0_100", "ics_score"])
if dpi_col is None:
    raise ValueError("raw 表找不到 dpi 列")
if ics_col is None:
    raise ValueError("ics 表找不到 ics 分数列")

gdf = bnd.merge(raw[["lsoa_code", dpi_col]], on="lsoa_code", how="left") \
         .merge(ics[["lsoa_code", ics_col]], on="lsoa_code", how="left")

print("Boundary rows:", len(bnd), "| Joined rows:", len(gdf))
print("DPI col:", dpi_col, "| ICS col:", ics_col)


# -------------------------
# 5) Run for one variable
# -------------------------
def run_var(gdf_in, value_col, tag):
    g0 = gdf_in.dropna(subset=[value_col]).copy().reset_index(drop=True)

    # build weights safely
    g1, W, neighbors, islands = build_queen_sparse_by_touches(g0)

    y_raw = pd.to_numeric(g1[value_col], errors="coerce").to_numpy(dtype=float)
    z = standardize(g1[value_col])

    glob = global_moran_I(z, W, permutations=PERMUTATIONS, seed=RANDOM_SEED)
    lisa = local_moran(z, W, permutations=PERMUTATIONS, seed=RANDOM_SEED + 10, alpha=ALPHA)
    gi = gistar_perm(y_raw, W, permutations=PERMUTATIONS, seed=RANDOM_SEED + 20, alpha=ALPHA)

    g1[f"{tag}_z"] = z
    g1["moran_lag_z"] = lisa["lag"]
    g1["lisa_I"] = lisa["I_local"]
    g1["lisa_p"] = lisa["p"]
    g1["lisa_class"] = lisa["class"]

    g1["gistar_g"] = gi["g"]
    g1["gistar_z"] = gi["z"]
    g1["gistar_p"] = gi["p"]
    g1["gistar_class"] = gi["class"]

    out_table = OUT_DIR / f"table_lsoa_spatial_stats_{tag}.csv"
    g1[["lsoa_code", value_col, f"{tag}_z",
        "lisa_I","lisa_p","lisa_class",
        "gistar_z","gistar_p","gistar_class"]].to_csv(out_table, index=False)

    # LISA plot (keep as generic categorical plot)
    fig_lisa = OUT_DIR / f"fig_lisa_{tag}_points.png"
    plot_points_lisa(g1, "lisa_class", f"LISA clusters (points) — {tag.upper()}", fig_lisa)

    # Gi* plot (NEWS style, public title, better legend, faded not-sig, outline)
    fig_gi = OUT_DIR / f"fig_gistar_{tag}_points.png"
    plot_gistar_points_news(g1, "gistar_class",
                        f"Statistically significant clusters of high and low risk — {tag.upper()}",
                        fig_gi)

    # z plot
    fig_z = OUT_DIR / f"fig_{tag}_z_points.png"
    plot_points_cont(g1, f"{tag}_z", f"{tag.upper()} z-score (points)", fig_z)

    return glob, out_table, fig_lisa, fig_gi, fig_z, len(g0), len(g1), len(islands)

dpi_glob, dpi_table, dpi_lisa_fig, dpi_gi_fig, dpi_z_fig, dpi_n0, dpi_n1, dpi_islands = run_var(gdf, dpi_col, "dpi")
ics_glob, ics_table, ics_lisa_fig, ics_gi_fig, ics_z_fig, ics_n0, ics_n1, ics_islands = run_var(gdf, ics_col, "ics")

summary = pd.DataFrame([
    {"variable":"DPI", "source_col":dpi_col, "n_before":dpi_n0, "n_used":dpi_n1, "n_islands_dropped":dpi_islands, **dpi_glob},
    {"variable":"ICS", "source_col":ics_col, "n_before":ics_n0, "n_used":ics_n1, "n_islands_dropped":ics_islands, **ics_glob},
])
summary_fp = OUT_DIR / "global_moran_summary.csv"
summary.to_csv(summary_fp, index=False)

print("\nSaved:", summary_fp)
display(summary)

print("\nSaved DPI outputs:")
print(" -", dpi_table)
print(" -", dpi_lisa_fig)
print(" -", dpi_gi_fig)
print(" -", dpi_z_fig)

print("\nSaved ICS outputs:")
print(" -", ics_table)
print(" -", ics_lisa_fig)
print(" -", ics_gi_fig)
print(" -", ics_z_fig)


Boundary rows: 4994 | Joined rows: 4994
DPI col: dpi_score | ICS col: ics_score_z

Saved: QM data/output/4_6.1_spatial_stats/global_moran_summary.csv


,variable,source_col,n_before,n_used,n_islands_dropped,I,p_sim,z_sim,n,permutations
0,DPI,dpi_score,4813,4813,0,0.404847,0.01,44.471496,4813,99
1,ICS,ics_score_z,4813,4813,0,0.364144,0.01,42.238321,4813,99



Saved DPI outputs:
 - QM data/output/4_6.1_spatial_stats/table_lsoa_spatial_stats_dpi.csv
 - QM data/output/4_6.1_spatial_stats/fig_lisa_dpi_points.png
 - QM data/output/4_6.1_spatial_stats/fig_gistar_dpi_points.png
 - QM data/output/4_6.1_spatial_stats/fig_dpi_z_points.png

Saved ICS outputs:
 - QM data/output/4_6.1_spatial_stats/table_lsoa_spatial_stats_ics.csv
 - QM data/output/4_6.1_spatial_stats/fig_lisa_ics_points.png
 - QM data/output/4_6.1_spatial_stats/fig_gistar_ics_points.png
 - QM data/output/4_6.1_spatial_stats/fig_ics_z_points.png


In [45]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import to_hex

OUT_DIR = Path("QM data/output/4_4_ics")
g = gpd.read_file(OUT_DIR / "ics_map.gpkg", layer="ics_map")

# (可选) simplify 降负担
try:
    g = g.to_crs(27700)
    g["geometry"] = g["geometry"].simplify(tolerance=30, preserve_topology=True)
except Exception as e:
    print("CRS/simplify skipped:", repr(e))

# --- normalize band labels (robust) ---
def _norm_band(v):
    s = str(v).strip().lower()
    if "top" in s or s.startswith("high"):
        return "High (Top 10%)"
    if "upper" in s or "10" in s and "25" in s:
        return "Upper (10–25%)"
    if "mid" in s or "25" in s and "50" in s:
        return "Mid (25–50%)"
    return "Lower (50–100%)"

g["_band"] = g["ics_band"].map(_norm_band)

BAND_ORDER = ["High (Top 10%)", "Upper (10–25%)", "Mid (25–50%)", "Lower (50–100%)"]

# --- muted reds: sample Reds cmap then blend with white (less saturated, cleaner) ---
def _blend_rgba_with_white(rgba, amount=0.28):
    r, gg, b, a = rgba
    r = r * (1 - amount) + 1.0 * amount
    gg = gg * (1 - amount) + 1.0 * amount
    b = b * (1 - amount) + 1.0 * amount
    return (r, gg, b, 1.0)

cmap = plt.cm.Reds
# high -> lower : deeper -> lighter (but we will blend)
raw_cols = [cmap(x) for x in [0.85, 0.70, 0.52, 0.22]]
cols = [to_hex(_blend_rgba_with_white(c, amount=0.28)) for c in raw_cols]
COL = dict(zip(BAND_ORDER, cols))

# --- outer London outline (thicker) ---
if hasattr(g.geometry, "union_all"):
    london_poly = g.geometry.union_all()
else:
    london_poly = g.geometry.unary_union
london_outline = london_poly.boundary  # ok for band map (holes don't really show)

fig, ax = plt.subplots(figsize=(9, 8))
ax.set_title("ICS risk band (policy-oriented)", fontsize=18, pad=12)

# Draw from low -> high (so high sits on top)
for band in ["Lower (50–100%)", "Mid (25–50%)", "Upper (10–25%)"]:
    g[g["_band"] == band].plot(ax=ax, color=COL[band], linewidth=0)

# High band base fill
g_hi = g[g["_band"] == "High (Top 10%)"]
g_hi.plot(ax=ax, color=COL["High (Top 10%)"], linewidth=0, zorder=5)

# High band overlay hatch + subtle edge (no heavy black)
g_hi.plot(
    ax=ax,
    facecolor="none",
    edgecolor="0.20",
    linewidth=0.7,
    hatch="//",
    zorder=6
)

# London outline (slightly darker than borough lines would be)
gpd.GeoSeries([london_outline], crs=g.crs).plot(
    ax=ax, color="0.35", linewidth=1.4, alpha=0.9, zorder=10
)

# Legend (order high -> low)
handles = [
    Patch(facecolor=COL["High (Top 10%)"], edgecolor="0.20", hatch="//", label="High (Top 10%)"),
    Patch(facecolor=COL["Upper (10–25%)"], edgecolor="none", label="Upper (10–25%)"),
    Patch(facecolor=COL["Mid (25–50%)"], edgecolor="none", label="Mid (25–50%)"),
    Patch(facecolor=COL["Lower (50–100%)"], edgecolor="none", label="Lower (50–100%)"),
]
ax.legend(handles=handles, title="Quantile bands", loc="lower left", frameon=True)

ax.set_axis_off()

# Crop whitespace
minx, miny, maxx, maxy = g.total_bounds
dx = (maxx - minx) * 0.02
dy = (maxy - miny) * 0.02
ax.set_xlim(minx - dx, maxx + dx)
ax.set_ylim(miny - dy, maxy + dy)

fig_fp = OUT_DIR / "fig_ics_band_map_muted.png"
plt.tight_layout()
plt.savefig(fig_fp, dpi=220)
plt.close(fig)
print("Saved:", fig_fp)


Saved: QM data/output/4_4_ics/fig_ics_band_map_muted.png


In [46]:
# =========================
# 4.7 Typology closure (cluster × risk profiles + cluster × quadrant)
# 目标：
# 1) cluster × (risk_participation, risk_vulnerability, risk_infra) 剖面图
# 2) cluster × quadrant（Q1–Q4）交叉表 + “马赛克图”(纯 matplotlib，不依赖 statsmodels)
# 3) 各 cluster 的 “被忽略区”(Q2 / overlooked_flag) 占比 + 排序
#
# 输入（沿用你前面代码的默认输出目录）：
# - QM data/output/4_3_clustering/cluster_profile_means_z.csv
# - QM data/output/4_3_clustering/cluster_profile_means_rawunits.csv
# - QM data/output/4_3_clustering/cluster_labels_k6_pca15.csv   (LSOA→cluster)
# - QM data/output/4_5_deri/ics_deri_compare_lsoa.csv           (LSOA→quadrant/overlooked)
#
# 输出：
# - QM data/output/4_7_typology_closure/*.csv, *.png
# =========================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# -------------------------
# 0) Paths (match your existing folder convention)
# -------------------------
CLUST_DIR = Path("QM data/output/4_3_clustering")
DERI_DIR  = Path("QM data/output/4_5_deri")
OUT_DIR   = Path("QM data/output/4_7_typology_closure")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PZ_FP     = CLUST_DIR / "cluster_profile_means_z.csv"
PR_FP     = CLUST_DIR / "cluster_profile_means_rawunits.csv"
LABEL_FP  = CLUST_DIR / "cluster_labels_k6_pca15.csv"
COMP_FP   = DERI_DIR  / "ics_deri_compare_lsoa.csv"

# fallback: if you happened to run parts in /mnt/data
CANDIDATES = {
    "PZ_FP": [PZ_FP, Path("/mnt/data/cluster_profile_means_z.csv")],
    "PR_FP": [PR_FP, Path("/mnt/data/cluster_profile_means_rawunits.csv")],
    "LABEL_FP": [LABEL_FP, Path("/mnt/data/cluster_labels_k6_pca15.csv")],
    "COMP_FP": [COMP_FP, Path("/mnt/data/ics_deri_compare_lsoa.csv")],
}

def _pick_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

PZ_FP    = _pick_existing(CANDIDATES["PZ_FP"])
PR_FP    = _pick_existing(CANDIDATES["PR_FP"])
LABEL_FP = _pick_existing(CANDIDATES["LABEL_FP"])
COMP_FP  = _pick_existing(CANDIDATES["COMP_FP"])

assert PZ_FP is not None, "Missing cluster_profile_means_z.csv (4_3 output)"
assert PR_FP is not None, "Missing cluster_profile_means_rawunits.csv (4_3 output)"
assert COMP_FP is not None, "Missing ics_deri_compare_lsoa.csv (4_5 output)"
assert LABEL_FP is not None, "Missing cluster_labels_k6_pca15.csv (4_3 output: LSOA→cluster)"

print("Using:")
print("  ", PZ_FP)
print("  ", PR_FP)
print("  ", LABEL_FP)
print("  ", COMP_FP)

# -------------------------
# 1) Load tables
# -------------------------
profile_z   = pd.read_csv(PZ_FP)
profile_raw = pd.read_csv(PR_FP)
labels      = pd.read_csv(LABEL_FP, dtype={"lsoa_code": str})
compare     = pd.read_csv(COMP_FP,  dtype={"lsoa_code": str})

# normalize keys
labels["lsoa_code"]  = labels["lsoa_code"].astype(str).str.strip().str.upper()
compare["lsoa_code"] = compare["lsoa_code"].astype(str).str.strip().str.upper()

# ensure cluster int
profile_z["cluster"] = profile_z["cluster"].astype(int)
profile_raw["cluster"] = profile_raw["cluster"].astype(int)
labels["cluster"] = labels["cluster"].astype(int)

# quadrant short label: Q1..Q4
compare["quadrant_short"] = compare["quadrant"].astype(str).str.extract(r"(Q\d)").fillna(compare["quadrant"].astype(str))

# -------------------------
# 2) Build 3 risk dimensions at CLUSTER level (profile table)
#    Direction unified: higher = higher risk
# -------------------------
def _safe_mean(df, cols):
    cols2 = [c for c in cols if c in df.columns]
    if len(cols2) == 0:
        return pd.Series(np.nan, index=df.index)
    return df[cols2].mean(axis=1)

dims = pd.DataFrame({"cluster": profile_z["cluster"].astype(int)})

# (a) Infrastructure risk: low broadband quality => higher risk => NEGATE z
infra_cols = ["bb_q_dl_core__z", "bb_q_ul_core__z", "bb_q_sfu_core__z"]
dims["risk_infra"] = -_safe_mean(profile_z, infra_cols)

# (b) Vulnerability risk: higher deprivation/low education/disability/fuel poverty/crime => higher risk
vul_cols = [
    "imd_score__z",
    "risk_edu_low__z",
    "risk_disability_limited__z",
    "risk_fuel_poverty__z",
    # pick one crime (prefer log if exists)
    "crime_rate_per_1000_window_log1p__z",
    "crime_rate_per_1000_window__z",
]
dims["risk_vulnerability"] = _safe_mean(profile_z, vul_cols)

# (c) Participation risk: low DPI => higher risk
# profile_z does not include dpi; use raw and z across clusters for comparability
if "dpi_score" in profile_raw.columns:
    dpi = profile_raw["dpi_score"].astype(float)
    dpi_z_cluster = (dpi - dpi.mean()) / (dpi.std(ddof=0) + 1e-12)
    dims["risk_participation"] = -dpi_z_cluster
else:
    dims["risk_participation"] = np.nan

# overall (just for ordering)
dims["risk_overall_mean"] = dims[["risk_participation", "risk_vulnerability", "risk_infra"]].mean(axis=1, skipna=True)

dims_fp = OUT_DIR / "table_cluster_risk_dimensions.csv"
dims.to_csv(dims_fp, index=False)
print("Saved:", dims_fp)

# -------------------------
# 3) Plot A: cluster × (3 risk dimensions) profile (bar)
# -------------------------
dims_plot = dims.sort_values("risk_overall_mean", ascending=False).reset_index(drop=True)
x = np.arange(len(dims_plot))
w = 0.26

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, dims_plot["risk_participation"], width=w, label="Participation risk (low DPI)")
ax.bar(x,       dims_plot["risk_vulnerability"], width=w, label="Vulnerability risk (IMD/edu/disability/fuel/crime)")
ax.bar(x + w, dims_plot["risk_infra"],         width=w, label="Infrastructure risk (low broadband quality)")
ax.set_xticks(x)
ax.set_xticklabels([f"C{c}" for c in dims_plot["cluster"]])
ax.set_title("Cluster risk profiles (3 dimensions) — higher = higher risk")
ax.set_ylabel("Standardised score (direction unified)")
ax.legend(loc="best")
plt.tight_layout()

fig_fp = OUT_DIR / "fig_cluster_risk_profiles_3dims.png"
plt.savefig(fig_fp, dpi=180)
plt.close()
print("Saved:", fig_fp)

# -------------------------
# 4) Cluster × Quadrant join (LSOA level)
# -------------------------
# keep only needed cols from compare
need_cols = ["lsoa_code", "quadrant", "quadrant_short", "overlooked_flag", "ics_score_z", "rDERI_score_0_10", "ics_pct", "deri_pct"]
need_cols = [c for c in need_cols if c in compare.columns]
df = labels.merge(compare[need_cols], on="lsoa_code", how="inner")

print("Joined rows (LSOA with cluster+quadrant):", len(df), "| unique clusters:", df["cluster"].nunique())

# -------------------------
# 5) Crosstab tables (counts + row shares + col shares)
# -------------------------
quad_order = ["Q1", "Q2", "Q3", "Q4"]

ct_counts = pd.crosstab(df["cluster"], df["quadrant_short"])
# enforce Q1..Q4 columns order where possible
for q in quad_order:
    if q not in ct_counts.columns:
        ct_counts[q] = 0
ct_counts = ct_counts[quad_order].sort_index()

ct_row = ct_counts.div(ct_counts.sum(axis=1), axis=0)          # within cluster
ct_col = ct_counts.div(ct_counts.sum(axis=0), axis=1)          # within quadrant

ct_counts_fp = OUT_DIR / "table_cluster_x_quadrant_counts.csv"
ct_row_fp    = OUT_DIR / "table_cluster_x_quadrant_rowshare.csv"
ct_col_fp    = OUT_DIR / "table_cluster_x_quadrant_colshare.csv"
ct_counts.to_csv(ct_counts_fp)
ct_row.to_csv(ct_row_fp)
ct_col.to_csv(ct_col_fp)

print("Saved:", ct_counts_fp)
print("Saved:", ct_row_fp)
print("Saved:", ct_col_fp)

# -------------------------
# 6) Plot B1: Stacked bar (row shares) — easy to read
# -------------------------
fig, ax = plt.subplots(figsize=(10, 4))
bottom = np.zeros(len(ct_row))
clusters_sorted = ct_row.index.values

for q in quad_order:
    vals = ct_row[q].values
    ax.bar([f"C{c}" for c in clusters_sorted], vals, bottom=bottom, label=q)
    bottom += vals

ax.set_title("Cluster × Quadrant (row shares) — each bar sums to 1")
ax.set_ylabel("Share within cluster")
ax.legend(title="Quadrant", loc="upper right")
plt.xticks(rotation=0)
plt.tight_layout()

fig_fp = OUT_DIR / "fig_cluster_x_quadrant_stacked_rowshare.png"
plt.savefig(fig_fp, dpi=180)
plt.close()
print("Saved:", fig_fp)

# -------------------------
# 7) Plot B2: Mosaic (pure matplotlib rectangles)
#    - vertical height ~ cluster size share
#    - horizontal width split by quadrant shares within that cluster
# -------------------------
cluster_sizes = df["cluster"].value_counts().sort_index()
cluster_share = (cluster_sizes / cluster_sizes.sum()).sort_values(ascending=False)

# color map for quadrants (do NOT overthink; consistent legend matters more)
quad_colors = {"Q1": "#d62728", "Q2": "#ff7f0e", "Q3": "#2ca02c", "Q4": "#1f77b4"}

fig, ax = plt.subplots(figsize=(8, 6))
y0 = 0.0
for c, h in cluster_share.items():
    row = ct_row.loc[c, quad_order].copy()
    row = row / (row.sum() + 1e-12)
    x0 = 0.0
    for q in quad_order:
        wq = float(row[q])
        if wq <= 0:
            continue
        ax.add_patch(Rectangle((x0, y0), wq, h, facecolor=quad_colors.get(q, "#cccccc"), edgecolor="white", linewidth=0.8))
        x0 += wq
    # label cluster on the left margin
    ax.text(-0.01, y0 + h/2, f"C{c}", ha="right", va="center", fontsize=9)
    y0 += h

# legend
handles = [Rectangle((0,0),1,1, facecolor=quad_colors[q]) for q in quad_order]
ax.legend(handles, quad_order, title="Quadrant", loc="upper right")
ax.set_xlim(-0.08, 1.0)
ax.set_ylim(0, 1.0)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticks([])
ax.set_xlabel("Within-cluster quadrant composition (width)")
ax.set_title("Mosaic: Cluster size (height) × Quadrant composition (width)")
plt.tight_layout()

fig_fp = OUT_DIR / "fig_mosaic_cluster_x_quadrant.png"
plt.savefig(fig_fp, dpi=180)
plt.close()
print("Saved:", fig_fp)

# -------------------------
# 8) Overlooked rate by cluster (核心：Q2 / overlooked_flag)
# -------------------------
over_rate = df.groupby("cluster")["overlooked_flag"].mean().sort_values(ascending=False)
over_n    = df[df["overlooked_flag"] == 1].groupby("cluster")["lsoa_code"].count().reindex(over_rate.index).fillna(0).astype(int)
clu_n     = df.groupby("cluster")["lsoa_code"].count().reindex(over_rate.index).astype(int)

over_table = pd.DataFrame({
    "cluster": over_rate.index.astype(int),
    "n_lsoa": clu_n.values,
    "n_overlooked": over_n.values,
    "overlooked_rate": over_rate.values,
}).merge(dims[["cluster","risk_participation","risk_vulnerability","risk_infra","risk_overall_mean"]], on="cluster", how="left")

over_fp = OUT_DIR / "table_overlooked_rate_by_cluster.csv"
over_table.to_csv(over_fp, index=False)
print("Saved:", over_fp)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([f"C{c}" for c in over_rate.index], over_rate.values)
ax.set_title("Overlooked rate by cluster (Q2: High ICS & Low DERI)")
ax.set_ylabel("Share of LSOAs in Q2")
plt.tight_layout()
fig_fp = OUT_DIR / "fig_overlooked_rate_by_cluster.png"
plt.savefig(fig_fp, dpi=180)
plt.close()
print("Saved:", fig_fp)

# -------------------------
# 9) Optional: “cluster typology summary” (one table for writing)
# -------------------------
# This is the table you’ll cite when writing typology:
# - which cluster is mainly driven by infra / vulnerability / participation
# - which cluster contributes most to overlooked (Q2)
typ_fp = OUT_DIR / "table_cluster_typology_summary.csv"
over_table.sort_values(["overlooked_rate", "risk_overall_mean"], ascending=False).to_csv(typ_fp, index=False)
print("Saved:", typ_fp)

print("\nDone: 4.7 Typology closure outputs in:", OUT_DIR)


Using:
   QM data/output/4_3_clustering/cluster_profile_means_z.csv
   QM data/output/4_3_clustering/cluster_profile_means_rawunits.csv
   QM data/output/4_3_clustering/cluster_labels_k6_pca15.csv
   QM data/output/4_5_deri/ics_deri_compare_lsoa.csv
Saved: QM data/output/4_7_typology_closure/table_cluster_risk_dimensions.csv
Saved: QM data/output/4_7_typology_closure/fig_cluster_risk_profiles_3dims.png
Joined rows (LSOA with cluster+quadrant): 4813 | unique clusters: 6
Saved: QM data/output/4_7_typology_closure/table_cluster_x_quadrant_counts.csv
Saved: QM data/output/4_7_typology_closure/table_cluster_x_quadrant_rowshare.csv
Saved: QM data/output/4_7_typology_closure/table_cluster_x_quadrant_colshare.csv
Saved: QM data/output/4_7_typology_closure/fig_cluster_x_quadrant_stacked_rowshare.png
Saved: QM data/output/4_7_typology_closure/fig_mosaic_cluster_x_quadrant.png
Saved: QM data/output/4_7_typology_closure/table_overlooked_rate_by_cluster.csv
Saved: QM data/output/4_7_typology_closur